In [1]:
# Cell 0 - Install dependencies
!pip install -q transformers sentence-transformers pandas simple-icd-10-cm huggingface_hub scikit-learn
!nvidia-smi


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.5 MB/s eta 0:00:00
Tue May 19 17:14:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0

In [2]:
# Cell 1 - Load all models: Whisper ASR, LLaMA 3.1 8B, SapBERT, Cross-Encoder, ICD-10 embeddings
import torch
import os
import json
import time
import re
import textwrap
import warnings
import shutil
import pandas as pd

import subprocess
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from sentence_transformers import SentenceTransformer, util, CrossEncoder
import simple_icd_10_cm as icd
from IPython.display import FileLink, display
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline, AutoTokenizer
import transformers
import huggingface_hub
from huggingface_hub import login
from tqdm import tqdm

warnings.filterwarnings("ignore")

print("=== STARTING BOOT SEQUENCE: DOWNLOADING & LOADING ALL MODELS ===")
t_boot_start = time.time()

login(token='HF_TOKEN_REDACTED')

print("\n[1/4] Loading Whisper ASR Model...")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
whisper_model_id = "openai/whisper-large-v3-turbo"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    whisper_model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
processor = AutoProcessor.from_pretrained(whisper_model_id)
pipe = hf_pipeline(
    "automatic-speech-recognition",
    model=model,
    return_timestamps=True,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

print("\n[2/4] Loading LLaMA 3.1 8B Model...")
llama_model_id = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
pipeline_8b_instruct = transformers.pipeline(
    "text-generation",
    model=llama_model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(llama_model_id)
terminators = [
    pipeline_8b_instruct.tokenizer.eos_token_id,
    pipeline_8b_instruct.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]
pipeline_llm = pipeline_8b_instruct

print("\n[3/4] Loading SapBERT & Cross-Encoder...")
SAPBERT_MODEL_ID = "pritamdeka/S-PubMedBert-MS-MARCO"
CROSS_ENCODER_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
embedder = SentenceTransformer(SAPBERT_MODEL_ID)
cross_encoder = CrossEncoder(CROSS_ENCODER_ID)

print("\n[4/4] Building ICD-10 Embeddings...")
records = [{"code": c, "description": icd.get_description(c)}
           for c in icd.get_all_codes() if icd.is_leaf(c)]
icd10_db = pd.DataFrame(records)
icd10_embeddings = embedder.encode(
    icd10_db["description"].tolist(),
    convert_to_tensor=True, show_progress_bar=True, batch_size=256,
)

t_boot_end = time.time()
print(f"\n=== BOOT COMPLETE IN {round(t_boot_end - t_boot_start, 2)}s ===")


=== STARTING BOOT SEQUENCE: DOWNLOADING & LOADING ALL MODELS ===

[1/4] Loading Whisper ASR Model...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


[4/4] Building ICD-10 Embeddings...


Batches:   0%|          | 0/292 [00:00<?, ?it/s]


=== BOOT COMPLETE IN 221.16s ===


In [3]:
# NEW CELL BETRAC-SWITCH - BeTraC-2026 audio + reference-SOAP loader
# Activates when USE_BETRAC_AUDIO=True. Downloads dev-00000.tar once, picks the
# sample at BETRAC_SAMPLE_INDEX, writes its audio to disk so Cell 2 (ASR) can
# transcribe it, and stores the gold SOAP note as `reference_soap` for ROUGE/BLEU.

USE_BETRAC_AUDIO     = True            # flip to False to use original audio path
BETRAC_SAMPLE_INDEX  = 0               # 0..4 (within saved samples)
BETRAC_TOKEN         = 'HF_TOKEN_REDACTED'
BETRAC_LOCAL_DIR     = '/kaggle/working/betrac_samples'
BETRAC_TAR_URL       = 'https://huggingface.co/datasets/BeTraC/betrac-2026/resolve/main/data/dev-00000.tar'

if USE_BETRAC_AUDIO:
    import os, re, subprocess
    try:
        import webdataset as wds
    except ImportError:
        subprocess.run(['pip', 'install', '-q', 'webdataset'], check=True)
        import webdataset as wds

    os.makedirs(BETRAC_LOCAL_DIR, exist_ok=True)
    existing = sorted([f for f in os.listdir(BETRAC_LOCAL_DIR) if f.endswith('.opus')])
    if len(existing) < 5:
        print(f"[BeTraC] Downloading 5 samples from dev-00000.tar...")
        url = f"pipe:curl -s -L {BETRAC_TAR_URL} -H 'Authorization: Bearer {BETRAC_TOKEN}'"
        ds = wds.WebDataset(url, shardshuffle=False)
        for i, sample in enumerate(ds):
            if i >= 5:
                break
            base = os.path.join(BETRAC_LOCAL_DIR, f"sample_{i:02d}")
            with open(base + ".opus", "wb") as f:
                f.write(sample["opus"])
            with open(base + ".transcript.txt", "w", encoding="utf-8") as f:
                f.write(sample["transcript.txt"].decode())
            with open(base + ".soap.txt", "w", encoding="utf-8") as f:
                f.write(sample["soap.txt"].decode())
            print(f"  saved sample_{i:02d}")
    else:
        print(f"[BeTraC] Found cached samples in {BETRAC_LOCAL_DIR}")

    idx = max(0, min(4, BETRAC_SAMPLE_INDEX))
    base = os.path.join(BETRAC_LOCAL_DIR, f"sample_{idx:02d}")
    audio_file_path = base + ".opus"
    with open(base + ".soap.txt", encoding="utf-8") as f:
        reference_soap = f.read()
    with open(base + ".transcript.txt", encoding="utf-8") as f:
        _betrac_raw_transcript = f.read()
    # Normalize speaker labels to Doctor:/Patient: (whatever case the file uses)
    _t = re.sub(r"(?im)^\s*DOCTOR\s*:", "Doctor:", _betrac_raw_transcript)
    _t = re.sub(r"(?im)^\s*PATIENT\s*:", "Patient:", _t)
    _t = re.sub(r"(?im)^\s*(SPEAKER\s*A|DR)\s*:", "Doctor:", _t)
    _t = re.sub(r"(?im)^\s*(SPEAKER\s*B|PT)\s*:", "Patient:", _t)
    betrac_reference_transcript = _t.strip()

    transcript = None

    print(f"[BeTraC] Active sample: {idx}")
    print(f"[BeTraC] audio = {audio_file_path}")
    print(f"[BeTraC] reference_soap = {len(reference_soap)} chars")
    print(f"[BeTraC] reference transcript preview: {betrac_reference_transcript[:200]}...")
else:
    audio_file_path = None
    reference_soap  = ""   # OFF mode: no ground-truth SOAP; ROUGE/BLEU vs reference will be 0/skipped
    print("[BeTraC] USE_BETRAC_AUDIO=False  ->  using original CAR0001.mp3; reference_soap=\"\"")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.5 MB/s eta 0:00:00
[BeTraC] Downloading 5 samples from dev-00000.tar...
  saved sample_00
  saved sample_01
  saved sample_02
  saved sample_03
  saved sample_04
[BeTraC] Active sample: 0
[BeTraC] audio = /kaggle/working/betrac_samples/sample_00.opus
[BeTraC] reference_soap = 2184 chars
[BeTraC] reference transcript preview: Doctor: Good morning, Carmaleta. I'm Dr. Nooruddin. So, what brings you in to see me today?
Patient: Ugh, morning. Look, I just… I want to have a baby, okay? And I don’t want any trouble doing it. My ...


In [4]:
# Cell 2 - ASR: transcribe audio files to text using Whisper
if globals().get("USE_BETRAC_AUDIO", False) and globals().get("audio_file_path"):
    t_asr_start = time.time()
    asr_transcrip_data = []
    print(f"[BeTraC ASR] transcribing {audio_file_path}")
    asr_transcrip_data.append(pipe(audio_file_path)['text'])
    t_asr_end = time.time()
    t_asr_elapsed = round(t_asr_end - t_asr_start, 2)
    print(f"ASR done in {t_asr_elapsed}s | {len(asr_transcrip_data)} transcript(s)")
    print(f"  Preview: {asr_transcrip_data[0][:300]}...")
else:
    t_asr_start = time.time()
    asr_transcrip_data = []
    audio_data_folder_path = '/kaggle/input/datasets/amitlakhi/sample-audio-data'
    for audio_file in os.listdir(audio_data_folder_path):
        if 'CAR0001.mp3' not in audio_file:
            continue
        audio_file_path_local = os.path.join(audio_data_folder_path, audio_file)
        asr_transcrip_data.append(pipe(audio_file_path_local)['text'])
    t_asr_end = time.time()
    t_asr_elapsed = round(t_asr_end - t_asr_start, 2)
    print(f"ASR done in {t_asr_elapsed}s | {len(asr_transcrip_data)} transcript(s)")


[BeTraC ASR] transcribing /kaggle/working/betrac_samples/sample_00.opus


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

ASR done in 21.89s | 1 transcript(s)
  Preview:  Good morning, Carmeletta. I'm Dr. Noruddin. So, what brings you in to see me today? Ugh, morning. Look, I just, I want to have a baby, okay? And I don't want any trouble doing it. My sister, she had such a hard time, and I just, I don't want that. I'm not getting any younger, you know. I want to ge...


In [5]:
### Pre Process the ASR Text, Add correct delimeter and then add the specaker
system_msg_1 = """You are an advanced AI model trained to analyze and structure the medical conversations between a doctor and a patient."""
user_message_1 = """You are given a two party conversation transcript between a Doctor and a Patient that was generated by an ASR engine. The transcript does not contain speaker labels. Your task is to correctly assign speaker roles as Doctor or Patient based on the context.
Below is the guidelines for identyfying the speaker in for a given sentence: 
1. Doctor's Role:
    * Asks diagnostic questions, gathers medical history, and provides medical insights.
    * Uses structured questioning (e.g., "What brought you in today?" "Have you experienced this before?
2. Patient's Role:
    * Describes symptoms, answers doctor's questions, and shares personal health information.
    * Responses are often personal experiences (e.g., "I have chest pain," "I smoke a pack a day.")


Output Instructions:
1. Provide the output in the following JSON format:
    [
        {
            "text": "<generated_text_with_speaker_labels>"
        }
    ]
2. Only add the Speaker Labels in the provided medical conversations which is given below. Don't add or remove original context. 

Please strictly follow the above mentioned instructions and output should be strictly in above mentioned JSON format without any deviation. Don't include any instructions or steps in the output

Medical Conversations:
%s

Output:
"""
sample_input_example_text = """what brought you in today? Sure. I'm just having a lot of chest pain, and so I thought I should get it checked out. Okay. And before we start, could you remind me of your gender and age? Sure. I'm 39. I'm a male. Okay. And so when did this chest pain start?"""

sample_output_example_text = """[
    {
        "text": "Doctor: What brought you in today?\nPatient: Sure. I'm just having a lot of chest pain, and so I thought I should get it checked out.\nDoctor: Okay. And before we start, could you remind me of your gender and age?\nPatient: Sure. I'm 39. I'm a male.\nDoctor: Okay. And so when did this chest pain start?"
    }
]"""

user_msg_1 = user_message_1 % (sample_input_example_text)
asssistant_msg_1 = sample_output_example_text


In [6]:
# Cell 4 - Run diarization: full-context, single-call per transcript
import torch
torch.cuda.empty_cache()
t_diar_start = time.time()

import tqdm
from tqdm import tqdm
processed_asr_transcript = []
for data_no, asr_transcrip_data_i in tqdm(enumerate(asr_transcrip_data)):
    if len(processed_asr_transcript) > 10:
        continue
    user_msg_2 = user_message_1 % (asr_transcrip_data_i)
    messages = [
        {"role": "system", "content": system_msg_1},
        {"role": "user", "content": user_msg_1},
        {"role": "assistant", "content": asssistant_msg_1},
        {"role": "user", "content": user_msg_2}
    ]
    outputs = pipeline_8b_instruct(messages,
                                max_new_tokens=4096,
                                eos_token_id=terminators,
                                do_sample=True,
                                temperature=0.2,
                                top_p=0.9)
    raw = outputs[0]['generated_text'][-1]['content']
    parsed_text = None
    try:
        parsed = json.loads(raw, strict=False)
        if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict) and parsed[0].get("text"):
            parsed_text = parsed[0]["text"]
    except Exception:
        m = re.search(r'"text"\s*:\s*"((?:[^"\\]|\\.)*)"', raw, re.DOTALL)
        if m:
            try:
                parsed_text = json.loads('"' + m.group(1) + '"')
            except Exception:
                parsed_text = m.group(1).replace('\\n', '\n').replace('\\"', '"')

    if parsed_text and parsed_text.strip():
        processed_asr_transcript.append(parsed_text)
    else:
        print(f"[Diarization] Could not parse LLM output as JSON. Falling back to raw ASR transcript (no speaker labels).")
        print(f"  Raw LLM output preview: {raw[:300]}...")
        processed_asr_transcript.append(asr_transcrip_data_i)
    torch.cuda.empty_cache()

processed_asr_transcript = processed_asr_transcript if len(processed_asr_transcript) < 10 else processed_asr_transcript[:10]

t_diar_end = time.time()
t_diar_elapsed = round(t_diar_end - t_diar_start, 2)
print(f"Diarization done in {t_diar_elapsed}s | {len(processed_asr_transcript)} transcript(s)")
if processed_asr_transcript:
    print(f"  Preview: {processed_asr_transcript[0][:300]}...")



0it [00:00, ?it/s]Passing `generation_config` together with generation-related arguments=({'eos_token_id', 'temperature', 'do_sample', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

1it [02:52, 172.48s/it]

Diarization done in 172.49s | 1 transcript(s)
  Preview: Doctor: Good morning, Carmeletta. I'm Dr. Noruddin. So, what brings you in to see me today?
Patient: Ugh, morning. Look, I just, I want to have a baby, okay? And I don't want any trouble doing it.
Doctor: My sister, she had such a hard time, and I just, I don't want that. I'm not getting any younger...


In [7]:
# Cell 5 - Pipeline config: thresholds for ICD-10, SNOMED, reranking, quality knobs
ICD10_COSINE_THRESHOLD = 0.65
RERANK_THRESHOLD       = 0.8
SNOMED_COSINE_THRESHOLD = 0.90
VALID_LABELS   = {"Drug", "Disease", "Symptom", "Procedure", "NULL"}
VALID_STATUSES = {"Confirmed", "Negated", "Historical", "Family_History"}

# --- Quality knobs (added in 2026-05 quality upgrade) ---
USE_NER_VERIFICATION   = True   # 2nd-pass LLM check on each NER entity (drops wrong-label/status)
USE_CODE_VERIFICATION  = False  # 2nd-pass LLM check on each SNOMED/ICD code (was dropping too many real codes; flip True only if you want stricter coding)
USE_SELF_CONSISTENCY   = False  # generate SOAP body 3x, pick best by hallucination/coverage. ON for thesis runs.
SELF_CONSISTENCY_N     = 3

print(f"Config | NER_VERIFY={USE_NER_VERIFICATION} CODE_VERIFY={USE_CODE_VERIFICATION} SELF_CONSISTENCY={USE_SELF_CONSISTENCY}")

transcript = processed_asr_transcript[0] if processed_asr_transcript else ""
print(f"Config done. Transcript length: {len(transcript)} chars")
if len(transcript) < 50:
    print("  WARNING: transcript is very short or empty. NER/SOAP will produce no output.")
    print(f"  Transcript content: {repr(transcript)[:200]}")
else:
    print(f"  Transcript preview: {transcript[:200]}...")


Config | NER_VERIFY=True CODE_VERIFY=False SELF_CONSISTENCY=False
Config done. Transcript length: 5958 chars
  Transcript preview: Doctor: Good morning, Carmeletta. I'm Dr. Noruddin. So, what brings you in to see me today?
Patient: Ugh, morning. Look, I just, I want to have a baby, okay? And I don't want any trouble doing it.
Doc...


In [8]:
# Cell 6 - Helpers: LLM wrapper, JSON parser, SPELL offset verification,
# hardened triple negation, schema enforcement, non-clinical filter,
# NER verification (VerifyNER-style), per-code verification (MedCodER-style)

def _llm(messages, max_tokens=2048, temperature=0.15, seed=None):
    kwargs = dict(
        max_new_tokens=max_tokens, eos_token_id=terminators,
        do_sample=True, temperature=temperature, top_p=0.9,
        repetition_penalty=1.15,
    )
    if seed is not None:
        torch.manual_seed(seed)
    out = pipeline_llm(messages, **kwargs)
    return out[0]["generated_text"][-1]["content"]


def _parse_json(raw, expect="list"):
    for fence in ("```json", "```"):
        if fence in raw:
            raw = raw.split(fence)[1].split("```")[0].strip()
            break
    if expect == "dict":
        search_order = [("{", "}"), ("[", "]")]
    else:
        search_order = [("[", "]"), ("{", "}")]
    for opener, closer in search_order:
        start = raw.find(opener)
        if start == -1:
            continue
        depth, end = 0, -1
        for i, ch in enumerate(raw[start:], start):
            if ch == opener:
                depth += 1
            elif ch == closer:
                depth -= 1
            if depth == 0:
                end = i + 1
                break
        if end != -1:
            raw = raw[start:end]
            break
        if opener == "[" and start != -1:
            last_brace = raw.rfind("}")
            if last_brace > start:
                truncated = raw[start:last_brace + 1] + "]"
                try:
                    test = json.loads(truncated, strict=False)
                    if isinstance(test, list):
                        raw = truncated
                        break
                except Exception:
                    last_comma = truncated.rfind(",")
                    if last_comma > start:
                        truncated2 = truncated[:last_comma] + "]"
                        try:
                            test2 = json.loads(truncated2, strict=False)
                            if isinstance(test2, list):
                                raw = truncated2
                                break
                        except Exception:
                            pass
    raw = re.sub(r",\s*([}\]])", r"\1", raw)
    _ESC = {"\n": "\\n", "\r": "\\r", "\t": "\\t"}
    result, in_str, esc_next = [], False, False
    for ch in raw:
        if esc_next:
            result.append(ch)
            esc_next = False
            continue
        if ch == "\\":
            result.append(ch)
            esc_next = in_str
            continue
        if ch == '"':
            in_str = not in_str
        if in_str and ch in _ESC:
            result.append(_ESC[ch])
            continue
        result.append(ch)
    cleaned = "".join(result)
    try:
        parsed = json.loads(cleaned, strict=False)
    except Exception:
        for m in re.finditer(r'[\[{]', cleaned):
            try:
                parsed = json.loads(cleaned[m.start():], strict=False)
                if expect == "list" and isinstance(parsed, list):
                    return parsed
                if expect == "dict" and isinstance(parsed, dict):
                    return parsed
            except Exception:
                continue
        return [] if expect == "list" else {}
    if expect == "list":
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            for v in parsed.values():
                if isinstance(v, list) and v and isinstance(v[0], dict):
                    return v
        return []
    else:
        if isinstance(parsed, dict):
            return parsed
        return {}


_MEDICAL_KEYWORDS = {
    "pain","ache","aches","aching","sore","soreness","tender","tenderness",
    "fever","fevers","febrile","chills","sweats","sweaty","sweating",
    "cough","coughing","wheeze","wheezing","breathe","breathing","breath",
    "dyspnea","tachypnea","apnea","gasping","winded","suffocating",
    "nausea","nauseated","nauseous","vomiting","vomit","emesis","retching",
    "diarrhea","constipation","bloating","bloated","flatulence",
    "bleeding","blood","hemorrhage","bruise","bruising","hematoma",
    "swelling","swollen","edema","oedema","pitting",
    "headache","migraine","dizziness","dizzy","lightheaded","vertigo","syncope",
    "numbness","numb","tingling","paresthesia","weakness","paralysis",
    "rash","itching","itchy","hives","urticaria","eczema","lesion","blister",
    "fatigue","tired","exhaustion","malaise","lethargy",
    "insomnia","sleep","drowsy","drowsiness",
    "chest","palpitations","tachycardia","bradycardia","arrhythmia","murmur",
    "cramp","cramps","crampy","cramping","spasm","spasms",
    "stiffness","stiff","swollen","lump","mass","nodule","tumor",
    "discharge","secretion","mucus","phlegm","sputum","congestion",
    "thirsty","thirst","polydipsia","polyuria","oliguria","anuria",
    "appetite","anorexia","weight","obesity","cachexia",
    "anxiety","anxious","depression","depressed","panic","agitation",
    "confusion","delirium","hallucination","tremor","seizure","convulsion",
    "fainted","fainting","unconscious","consciousness","coma",
    "urinary","urination","dysuria","hematuria","incontinence","frequency",
    "constipated","indigestion","dyspepsia","heartburn","reflux","regurgitation",
    "jaundice","icterus","ascites","melena","hematemesis",
    "orthopnea","stridor","cyanosis","clubbing","pallor","diaphoresis",
    "tinnitus","vertigo","photophobia","diplopia","blurry","scotoma",
    "dysphagia","odynophagia","hoarseness","aphonia",
    "edematous","erythema","erythematous","purpura","petechiae",
    "rigidity","guarding","rebound","distension","distended",
    "crackles","rales","rhonchi","friction","gallop",
    "diabetes","diabetic","hypertension","hypotension",
    "asthma","copd","bronchitis","pneumonia","tuberculosis",
    "heart","cardiac","coronary","angina","infarction","myocardial",
    "failure","fibrillation","flutter","stenosis","regurgitation",
    "stroke","ischemic","hemorrhagic","aneurysm","embolism","thrombosis","clot",
    "cancer","carcinoma","melanoma","lymphoma","leukemia","tumor","neoplasm",
    "infection","infectious","sepsis","abscess","cellulitis",
    "arthritis","osteoporosis","fracture","dislocation","sprain","strain",
    "sciatica","radiculopathy","neuropathy","myelopathy",
    "anemia","thrombocytopenia","coagulopathy",
    "hypothyroidism","hyperthyroidism","thyroid","goiter",
    "cirrhosis","hepatitis","pancreatitis","cholecystitis","appendicitis",
    "colitis","diverticulitis","hernia","obstruction","ileus",
    "gastritis","gastroenteritis","enteritis","esophagitis",
    "nephritis","pyelonephritis","nephrolithiasis","cystitis",
    "uti","urinary","prostatitis","bph",
    "eczema","psoriasis","dermatitis","cellulitis","folliculitis",
    "migraine","epilepsy","parkinson","alzheimer","dementia","multiple sclerosis",
    "lupus","fibromyalgia","gout","rheumatoid",
    "hiv","aids","hepatitis","herpes","shingles","influenza","flu",
    "copd","emphysema","fibrosis","sarcoidosis","pleurisy","pleural",
    "pericarditis","myocarditis","endocarditis","cardiomyopathy",
    "dvt","pe","pulmonary","aortic","mitral","tricuspid",
    "cholesterol","hyperlipidemia","dyslipidemia","triglyceride",
    "obesity","bmi","overweight","underweight","malnutrition",
    "allergic","allergy","allergies","anaphylaxis","angioedema",
    "pregnancy","pregnant","prenatal","postpartum","miscarriage","ectopic",
    "amenorrhea","dysmenorrhea","menorrhagia","metrorrhagia","menstrual",
    "period","periods","menstruation","ovulation",
    "preeclampsia","eclampsia","gestational","trimester",
    "contraception","contraceptive",
    "medication","medications","medicine","drug","prescription","prescribed",
    "antibiotic","antiviral","antifungal","antihistamine","analgesic",
    "opioid","nsaid","steroid","corticosteroid","immunosuppressant",
    "insulin","metformin","lisinopril","amlodipine","metoprolol",
    "furosemide","losartan","atorvastatin","rosuvastatin","simvastatin",
    "omeprazole","pantoprazole","gabapentin","pregabalin",
    "amoxicillin","azithromycin","ciprofloxacin","penicillin",
    "acetaminophen","tylenol","ibuprofen","advil","motrin","naproxen",
    "aspirin","warfarin","heparin","eliquis","xarelto",
    "prednisone","albuterol","inhaler","puffer","puffers","nebulizer",
    "levothyroxine","synthroid","multivitamin","supplement",
    "marijuana","cannabis","cocaine","methamphetamine","meth","crystal",
    "heroin","fentanyl","oxycodone","hydrocodone","morphine","codeine",
    "alcohol","nicotine","cigarette","cigarettes","smoking","tobacco","vaping",
    "ginger","vitamin","mineral","probiotic",
    "birth control","oral contraceptive","condom","condoms","iud",
    "x-ray","xray","mri","ct","scan","ultrasound","sonogram","echocardiogram",
    "endoscopy","colonoscopy","bronchoscopy","cystoscopy","arthroscopy",
    "biopsy","aspiration","catheterization","intubation","ventilation",
    "ecg","ekg","electrocardiogram","holter","stress test",
    "surgery","surgical","appendectomy","cholecystectomy","hysterectomy",
    "mastectomy","colectomy","laparoscopy","thoracotomy","craniotomy",
    "transplant","implant","stent","pacemaker","defibrillator",
    "dialysis","transfusion","infusion","injection","vaccination","immunization",
    "therapy","rehabilitation","physiotherapy","chemotherapy","radiation",
    "lab","laboratory","test","panel","count","level","titer","culture",
    "referral","consultation","cardiology","neurology","orthopedic",
    "dermatology","gastroenterology","pulmonology","oncology","urology",
    "emergency","hospitalization","admission","discharge","transfer",
    "cbc","bmp","cmp","bnp","troponin","hemoglobin","hematocrit",
    "urinalysis","lfts","tsh","hba1c","lipid","glucose","creatinine",
    "oxygen","supplemental","pulse oximetry","spo2","saturation",
    "nickel","latex","shellfish","peanut","sulfa","sulfate",
    "jewelry","penicillin allergy",
    "vitals","blood pressure","pulse","heart rate","respiratory rate",
    "temperature","oxygen saturation",
    "auscultation","palpation","percussion","inspection",
    "reflexes","range of motion","gait","strength",
}

_JUNK_PATTERNS = re.compile(
    r"^(?:"
    r"(?:one|two|three|four|five|six|seven|eight|nine|ten|\d+)\s+"
    r"(?:week|day|month|year|hour|minute)s?\s*(?:ago|later|before|after)?$"
    r"|"
    r"\d+\s*(?:milligram|miligram|mg|mcg|ml|gram)s?(?:\s+(?:daily|twice|once|three|a\s+day))?$"
    r"|"
    r"(?:once|twice|three\s+times?)\s+(?:a|per)\s+(?:day|week|month)$"
    r"|"
    r"(?:feels?\s+like|kind\s+of|sort\s+of|all\s+the\s+time|right\s+now|"
    r"not\s+really|i\s+(?:think|guess|don'?t)|oh\s+gosh|let\s+me\s+think|"
    r"nothing\s+really|like\s+nothing|pretty\s+(?:good|bad|regular)|"
    r"that'?s?\s+about\s+it|i\s+see|sure|sounds?\s+good|got\s+it|"
    r"couple\s+(?:of\s+)?hours|couple\s+(?:of\s+)?days)"
    r"|"
    r"(?:shoveling|moving\s+furniture|playing\s+(?:rugby|basketball|sports)|"
    r"tackled|walking|running|driving|eating|cooking|working|sitting|"
    r"standing|lying\s+down|laying\s+down|exercis\w+|gardening)"
    r"|"
    r"(?:felt\s+fine|feels?\s+fine|chest\s+felt\s+fine|no\s+(?:pain|issues)|"
    r"similar\s+episodes|nothing\s+(?:else|new|wrong)|haven'?t\s+(?:had|noticed)|"
    r"never\s+had|not\s+that\s+I|recently\s+injured|didn'?t\s+(?:start|notice))"
    r"|"
    r"(?:yesterday|today|tonight|last\s+night|this\s+morning|a\s+month\s+ago|"
    r"a\s+week\s+ago|last\s+week|recently|a\s+few\s+weeks?\s+ago|"
    r"five\s+days?\s+ago|six\s+days?\s+ago)"
    r"|"
    r"(?:yes|no|yeah|okay|sure|alright|hmm|uh|um|oh|mm-mm|pee|wind|"
    r"suddenly|immobilized|travelling|hospitalized|previous\s+surgeries|"
    r"up\s+to\s+date|excellent|left\s+side|right\s+side|dull|sharp|"
    r"constant|gradually|worse|better)$"
    r")$", re.I
)

def _is_non_clinical(entity_text):
    text = entity_text.strip().lower()
    has_medical_word = False
    for kw in _MEDICAL_KEYWORDS:
        if kw in text:
            has_medical_word = True
            break
    if not has_medical_word:
        return True
    if _JUNK_PATTERNS.match(text):
        return True
    return False

def _spell_verify_and_align(entity_text, source):
    text = entity_text.strip()
    if not text or len(text) < 3:
        return None
    m = re.search(re.escape(text), source, re.IGNORECASE)
    if m:
        return {"start": m.start(), "end": m.end(), "method": "exact",
                "matched": source[m.start():m.end()]}
    simplified = re.sub(
        r'\b(my|the|a|an|in|of|on|for|with|like|very|really|'
        r'kind\s+of|just|been|some|little|bit)\b', '', text, flags=re.I
    ).strip()
    simplified = re.sub(r'\s+', ' ', simplified)
    if len(simplified) >= 3:
        m = re.search(re.escape(simplified), source, re.IGNORECASE)
        if m:
            return {"start": m.start(), "end": m.end(), "method": "fuzzy_stopword",
                    "matched": source[m.start():m.end()]}
    words = [w for w in text.split() if len(w) >= 3]
    if len(words) >= 2:
        for i in range(len(words) - 1):
            bigram = re.escape(words[i]) + r'\s+(?:\w+\s+){0,3}' + re.escape(words[i+1])
            m = re.search(bigram, source, re.IGNORECASE)
            if m:
                return {"start": m.start(), "end": m.end(), "method": "fuzzy_bigram",
                        "matched": source[m.start():m.end()]}
    if len(text.split()) == 1 and len(text) >= 4:
        m = re.search(re.escape(text), source, re.IGNORECASE)
        if m:
            return {"start": m.start(), "end": m.end(), "method": "keyword",
                    "matched": source[m.start():m.end()]}
    return None


# --- HARDENED NEGATION REGEX (2026-05 upgrade) ---
_NEG_PRE = re.compile(
    r"\b(?:no|not|denies?|denied|without|absent|negative|never|"
    r"doesn'?t|don'?t|hasn'?t|wasn'?t|isn'?t|cannot|haven'?t|"
    r"rules?\s*out|excludes?|free\s+of|neither|nor|"
    r"low\s+suspicion\s+for|unlikely|improbable|rules?\s+out|"
    r"work(?:ed)?\s*up\s+(?:negative\s+for|but\s+negative))\b", re.I
)
_NEG_POST = re.compile(
    r"\b(?:was\s+ruled\s+out|is\s+absent|"
    r"not\s+(?:found|seen|present|detected)|negative\s+for|"
    r"(?:is|was|seems?|appears?)\s+unlikely|"
    r"(?:was|is)\s+excluded)\b", re.I
)
_PATIENT_DENIAL = re.compile(
    r"\b(?:no[,.]?\s*(?:I\s+don'?t|not\s+really|I\s+haven'?t|"
    r"nothing|none|I\s+don'?t\s+think\s+so|mm-mm|nope)|"
    r"not\s+(?:really|that\s+I))\b", re.I
)
_FAMILY = re.compile(
    r"\b(?:father|mother|brother|sister|dad|mom|parent|"
    r"grandfather|grandmother|uncle|aunt|family)\b", re.I
)
_HISTORICAL = re.compile(
    r"\b(?:years?\s+ago|previously|in\s+the\s+past|former|"
    r"when\s+(?:I|she|he)\s+was\s+younger|history\s+of|"
    r"used\s+to|i\s+used\s+to|quit\s+\w+\s+(?:years?|months?)\s+ago|"
    r"had\s+\w+\s+(?:removed|done)|went\s+away|"
    r"resolved|has\s+cleared|cleared\s+up|(?:no\s+longer|not\s+anymore)|"
    r"recovered\s+from|past\s+history\s+of)\b", re.I
)
_UNCERTAIN = re.compile(
    r"\b(?:rule\s+out|work\s*up|workup\s+for|"
    r"differential\s+includes?|suspected|possible|"
    r"concerning\s+for|suggestive\s+of|consistent\s+with|"
    r"may\s+(?:have|be)|might\s+(?:have|be)|could\s+(?:have|be))\b", re.I
)

def _triple_negation(entity_text, transcript_text, llm_status, label, spell_offset=None):
    t_lower = transcript_text.lower()
    e_lower = entity_text.lower().strip()
    if spell_offset and spell_offset.get("start") is not None:
        idx = spell_offset["start"]
    else:
        idx = t_lower.find(e_lower)
    if idx == -1:
        return llm_status
    pre_window = t_lower[max(0, idx - 30):idx]
    post_window = t_lower[idx + len(e_lower):idx + len(e_lower) + 80]
    sent_start = max(t_lower.rfind(".", 0, idx), t_lower.rfind("?", 0, idx)) + 1
    sent_end = t_lower.find(".", idx + len(e_lower))
    if sent_end == -1:
        sent_end = min(len(t_lower), idx + len(e_lower) + 100)
    sentence = t_lower[sent_start:sent_end]
    fam_window = t_lower[max(0, idx - 60):idx + len(e_lower) + 60]
    if _FAMILY.search(fam_window) and label in ("Disease", "Symptom"):
        after = t_lower[idx + len(e_lower):idx + len(e_lower) + 80]
        if re.search(r'\bno\b', after):
            return "Negated"
        return "Family_History"
    if _HISTORICAL.search(sentence) and label in ("Drug", "Disease", "Symptom"):
        return "Historical"
    if label == "Drug":
        if re.search(r"used\s+to\s+(?:be\s+on|take)", pre_window):
            return "Historical"
        if re.search(r"ran\s+out", sentence):
            return "Confirmed"
        if _NEG_PRE.search(pre_window):
            return "Negated"
        return llm_status
    # explicit "rule out / unlikely / low suspicion" cluster -> Negated (treated as not-confirmed)
    if _UNCERTAIN.search(pre_window) and label in ("Disease",):
        # 'rule out X' / 'concerning for X' -> treat as uncertain; keep Confirmed only if doctor commits later
        if re.search(r"rule\s+out|low\s+suspicion|unlikely", pre_window):
            return "Negated"
        # 'concerning for / suggestive of' is positive uncertainty -> stay Confirmed
    layer1 = bool(_NEG_PRE.search(pre_window))
    layer2 = bool(_NEG_POST.search(post_window)) or bool(_PATIENT_DENIAL.search(post_window))
    layer3 = (llm_status == "Negated")
    neg_count = sum([layer1, layer2, layer3])
    if neg_count >= 2:
        return "Negated"
    if layer1 and not layer3 and llm_status == "Confirmed":
        return "Negated"
    if llm_status in VALID_STATUSES:
        return llm_status
    return "Confirmed"


_LABEL_MAP = {
    "drugs": "Drug", "drug": "Drug", "medication": "Drug", "medications": "Drug",
    "disease": "Disease", "diseases": "Disease", "condition": "Disease",
    "diagnosis": "Disease", "allergy": "Disease",
    "symptom": "Symptom", "symptoms": "Symptom", "finding": "Symptom",
    "sign": "Symptom", "vital": "Symptom",
    "procedure": "Procedure", "procedures": "Procedure",
    "null": "NULL", "none": "NULL", "unknown": "NULL", "other": "NULL",
    "test": "Procedure", "exam": "Procedure", "imaging": "Procedure",
}
_STATUS_MAP = {
    "confirmed": "Confirmed", "negated": "Negated", "historical": "Historical",
    "family_history": "Family_History", "family history": "Family_History",
    "denied": "Negated", "absent": "Negated", "present": "Confirmed",
}

def _stem_simple(text):
    t = text.lower().strip()
    if " " in t:
        return " ".join(_stem_simple(w) for w in t.split())
    for suffix in ["ting", "ness", "ment", "ated", "ing", "ies",
                    "ous", "ive", "ion", "ity", "ful", "ble", "ed", "er",
                    "ly", "es", "al", "s"]:
        if len(t) > len(suffix) + 3 and t.endswith(suffix):
            return t[:-len(suffix)]
    return t

def _enforce_schema(entities):
    clean, rejected, seen_exact = [], [], set()
    for ent in entities:
        text = str(ent.get("text", "")).strip()
        raw_label = str(ent.get("label", "")).strip().lower()
        label = _LABEL_MAP.get(raw_label, raw_label.capitalize())
        raw_status = str(ent.get("status", "Confirmed")).strip().lower().replace(" ", "_")
        status = _STATUS_MAP.get(raw_status, raw_status.replace("_", " ").title().replace(" ", "_"))
        if label not in VALID_LABELS:
            rejected.append({**ent, "_reason": f"Invalid label: {label}"})
            continue
        if status not in VALID_STATUSES:
            rejected.append({**ent, "_reason": f"Invalid status: {status}"})
            continue
        if not text or len(text) < 3:
            rejected.append({**ent, "_reason": "Too short"})
            continue
        if text.lower() in seen_exact:
            rejected.append({**ent, "_reason": "Duplicate"})
            continue
        seen_exact.add(text.lower())
        clean.append({"text": text, "label": label, "status": status,
                       "reasoning": ent.get("reasoning", "")})
    final, sub_rejected = [], []
    for i, e in enumerate(clean):
        is_sub = False
        for j, other in enumerate(clean):
            if i == j:
                continue
            if (e["label"] == other["label"] and e["status"] == other["status"] and
                e["text"].lower() in other["text"].lower() and
                len(e["text"]) < len(other["text"])):
                is_sub = True
                break
        if is_sub:
            sub_rejected.append({**e, "_reason": f"Substring of '{other['text']}'"})
        else:
            final.append(e)
    return final, sub_rejected + rejected


# --- NEW: NER verification pass (VerifyNER-style, 2026-05 upgrade) ---
_NER_VERIFY_PROMPT = textwrap.dedent("""
You are a clinical NER auditor. For each entity below, judge if its label and status are correct
given the original transcript context window.

LABELS: Drug | Disease | Symptom | Procedure | NULL
STATUSES: Confirmed | Negated | Historical | Family_History

Rules:
- Drug: medications, substances (alcohol, cannabis, tobacco, cocaine, etc.).
- Disease: diagnoses or chronic conditions (e.g., hypertension, asthma).
- Symptom: complaints, sensations, findings (e.g., chest pain, fever, swelling).
- Procedure: medical tests, imaging, referrals, or therapeutic interventions ordered by the clinician.
- NULL: clinically relevant info that does not fit the above (occupation, social context, etc.).
- Negated = patient or doctor denies / rules out the finding.
- Historical = "used to have", "had X removed", "resolved", "years ago".
- Family_History = a relative had it, not the patient.
- Confirmed = patient currently has it.

For each entity output one JSON object:
- If correct as-is: {"id": <i>, "decision": "keep"}
- If wrong label/status: {"id": <i>, "decision": "fix", "label": "...", "status": "..."}
- If the span should not be an entity at all: {"id": <i>, "decision": "drop"}

EXAMPLE entities:
[
  {"id": 0, "text": "chest pain", "label": "Symptom", "status": "Confirmed", "context": "...having chest pain for the past three days..."},
  {"id": 1, "text": "heart attack", "label": "Disease", "status": "Confirmed", "context": "...my father had a heart attack at 55..."},
  {"id": 2, "text": "smoked", "label": "Procedure", "status": "Confirmed", "context": "...I smoked for 20 years..."}
]
EXAMPLE output:
[
  {"id": 0, "decision": "keep"},
  {"id": 1, "decision": "fix", "label": "Disease", "status": "Family_History"},
  {"id": 2, "decision": "fix", "label": "Drug", "status": "Historical"}
]

Return ONLY the JSON array. No prose.
""").strip()


def verify_entities_llm(entities, source, batch_size=20):
    """VerifyNER-style 2nd pass. Drops entities marked 'drop', applies 'fix' decisions."""
    if not entities:
        return entities, []
    print(f"  [NER-Verify] Auditing {len(entities)} entities...")
    t0 = time.time()
    final, removed = [], []
    for batch_start in range(0, len(entities), batch_size):
        batch = entities[batch_start:batch_start + batch_size]
        items_payload = []
        for i, ent in enumerate(batch):
            offset = ent.get("first_offset") or {}
            s = max(0, (offset.get("start") or 0) - 60)
            e = min(len(source), (offset.get("end") or len(ent.get("text", ""))) + 60)
            context = source[s:e].replace("\n", " ").strip()
            items_payload.append({
                "id": i, "text": ent.get("text"), "label": ent.get("label"),
                "status": ent.get("status"), "context": context[:200]
            })
        msg = [
            {"role": "system", "content": _NER_VERIFY_PROMPT},
            {"role": "user", "content": "Entities:\n" + json.dumps(items_payload, ensure_ascii=False) + "\n\nDecisions JSON array:"}
        ]
        raw = _llm(msg, max_tokens=900, temperature=0.05)
        decisions = _parse_json(raw, "list")
        by_id = {}
        for d in decisions:
            if isinstance(d, dict) and "id" in d:
                by_id[int(d["id"])] = d
        for i, ent in enumerate(batch):
            d = by_id.get(i, {"decision": "keep"})
            decision = (d.get("decision") or "keep").lower().strip()
            if decision == "drop":
                removed.append({**ent, "_reason": "NER-Verify drop"})
                continue
            if decision == "fix":
                new_label = _LABEL_MAP.get((d.get("label") or "").lower(), d.get("label"))
                new_status_raw = (d.get("status") or "").strip().lower().replace(" ", "_")
                new_status = _STATUS_MAP.get(new_status_raw, new_status_raw.replace("_", " ").title().replace(" ", "_"))
                if new_label in VALID_LABELS:
                    if ent["label"] != new_label:
                        ent["_verify_fix"] = f"{ent['label']}->{new_label}"
                    ent["label"] = new_label
                if new_status in VALID_STATUSES:
                    if ent["status"] != new_status:
                        ent["_verify_fix"] = (ent.get("_verify_fix", "") + f" {ent['status']}->{new_status}").strip()
                    ent["status"] = new_status
            final.append(ent)
    print(f"  [NER-Verify] Kept {len(final)}/{len(entities)} in {time.time()-t0:.1f}s")
    return final, removed


# --- NEW: per-code verification (MedCodER-style, 2026-05 upgrade) ---
_CODE_VERIFY_PROMPT = textwrap.dedent("""
You are a medical coding auditor. For each row below, decide if the assigned code accurately
describes the span in its transcript context.

Rules:
- "keep" if the code clearly matches the clinical meaning of the span in context.
- "drop" if the code is wrong, too generic, or unrelated to the span's actual meaning.
- Borderline cases where the code is plausible but the span is too vague: keep.

Return ONLY a JSON array of decisions: [{"id": <i>, "decision": "keep"|"drop"}, ...]
""").strip()

def verify_code_llm(coded_entities, code_field, source, batch_size=20):
    """
    Per-code 2nd pass. code_field is "snomed" or "icd10_cm".
    Drops the specific code field on entities marked 'drop'; entity itself remains.
    """
    candidates_idx = [i for i, e in enumerate(coded_entities)
                       if e.get(code_field) and (e[code_field].get("code") or e[code_field].get("concept_id"))]
    if not candidates_idx:
        return coded_entities
    print(f"  [{code_field}-Verify] Auditing {len(candidates_idx)} codes...")
    t0 = time.time()
    for batch_start in range(0, len(candidates_idx), batch_size):
        batch_ids = candidates_idx[batch_start:batch_start + batch_size]
        items = []
        for bi, ei in enumerate(batch_ids):
            ent = coded_entities[ei]
            offset = ent.get("first_offset") or {}
            s = max(0, (offset.get("start") or 0) - 60)
            e = min(len(source), (offset.get("end") or len(ent.get("text", ""))) + 60)
            context = source[s:e].replace("\n", " ").strip()
            code_obj = ent[code_field]
            code_val = code_obj.get("code") or code_obj.get("concept_id")
            code_desc = code_obj.get("description") or code_obj.get("term", "")
            items.append({
                "id": bi, "span": ent.get("text"),
                "code": str(code_val), "code_description": code_desc,
                "context": context[:200],
            })
        msg = [
            {"role": "system", "content": _CODE_VERIFY_PROMPT},
            {"role": "user", "content": f"Field: {code_field}\nRows:\n" + json.dumps(items, ensure_ascii=False) + "\n\nDecisions JSON array:"}
        ]
        raw = _llm(msg, max_tokens=600, temperature=0.05)
        decisions = _parse_json(raw, "list")
        for d in decisions:
            if not isinstance(d, dict): continue
            try:
                bi = int(d.get("id", -1))
                if bi < 0 or bi >= len(batch_ids): continue
                if (d.get("decision") or "").lower().strip() == "drop":
                    ei = batch_ids[bi]
                    coded_entities[ei][code_field] = None
                    coded_entities[ei][f"_{code_field}_dropped"] = True
            except Exception:
                continue
    print(f"  [{code_field}-Verify] done in {time.time()-t0:.1f}s")
    return coded_entities


print("Cell 6 done: helpers + hardened negation + verify_entities_llm + verify_code_llm")


Cell 6 done: helpers + hardened negation + verify_entities_llm + verify_code_llm


In [9]:
# Cell 7 - NER: SPELL-architecture entity extraction with retry, non-clinical filter,
# offset-aware negation, and optional VerifyNER-style 2nd pass (2026-05 upgrade)
_NER_PROMPT = textwrap.dedent("""
You are a clinical Named Entity Recognition engine. Extract ALL clinically relevant
medical entities from the transcript below.

LABELS: Drug | Disease | Symptom | Procedure | NULL
STATUSES: Confirmed | Negated | Historical | Family_History

THINK STEP-BY-STEP INTERNALLY before producing the JSON.
Do NOT include any thinking in the output - return ONLY a JSON array.

RULES:
1. "text" MUST be an EXACT SUBSTRING copied verbatim from the transcript.
2. Extract PERTINENT NEGATIVES: when patient denies a symptom
   (e.g., "Any fevers? No."), extract it with status "Negated".
3. QUALIFIED AFFIRMATIVES are CONFIRMED, not Negated. If the patient gives
   a reason or qualifier ("just from breathing issues", "only when lying down",
   "a little bit"), that is a CONFIRMED symptom. "Just from X" means YES.
4. Substances (tobacco, cigarettes, cannabis, marijuana, alcohol, cocaine,
   opioids) are label "Drug", NOT "Procedure". Smoking is a Drug/substance.
5. Pain MODIFIERS are not entities. Do NOT extract aggravating factors like
   "makes the pain worse", "laying down", "taking a deep breath" as separate
   entities. Only extract the symptom itself (e.g., "chest pain").
6. "Procedure" is ONLY for medical tests, imaging, or referrals the DOCTOR
   explicitly orders (e.g., "EKG", "chest X-ray", "blood work").
7. Extract medications WITH dosage as a single entity when they appear
   together (e.g., "Lisinopril 20 milligrams daily" as one entity).
8. Extract allergies as Disease with status Confirmed.
9. For family history conditions, use status Family_History.
10. Do NOT extract conversational filler, time phrases, or dosage fragments
    without a drug name. Bad: "six weeks ago", "twice a day", "20 milligrams".
11. Extract vital signs and physical exam findings as Symptom (Confirmed).
12. If an entity does not clearly fit Drug, Disease, Symptom, or Procedure,
    label it "NULL". Examples: occupation, living situation, diet, exercise,
    immunization status. Do NOT force a wrong label. NULL beats a wrong label.
13. BE THOROUGH. A typical full encounter has 18-30 entities. Before stopping, scan again for:
    - All confirmed AND denied symptoms (don't skip patient denials)
    - Every medication mentioned (current, historical, denied, OTC)
    - Every disease/condition (current, historical, family history, allergies)
    - Every procedure/test/imaging/referral the doctor mentions
    - Vital signs, exam findings
    - Social history (occupation, smoking, alcohol, drugs)
    - Family history specifics (mother, father, sibling diagnoses)
    If you extract fewer than 15 entities for a normal-length encounter, you have UNDER-EXTRACTED. Re-scan and add the missed ones.
14. "rule out X" / "low suspicion for X" -> status Negated (not Confirmed).
    "concerning for X" / "suggestive of X" -> status Confirmed (positive uncertainty).
15. "used to have X" / "X resolved" / "X cleared up" -> status Historical.
16. ASPIRATIONAL or FUTURE-INTENT mentions are NOT current clinical states. Watch for verbs of desire / planning / consideration ("want to", "hope to", "planning to", "considering", "trying to", "thinking about", "would like to"). When these verbs introduce a clinical outcome, the outcome is the patient's GOAL, not a current Disease / Symptom / Procedure. Either skip the mention entirely, or extract it as NULL Confirmed (a desire / plan), never as the achieved state. Rule of thumb: ask "is the patient describing something that HAS happened / IS happening, or something they WANT to happen?" Only the first qualifies for Disease / Symptom / Procedure.

BAD vs GOOD examples (do NOT do BAD):
BAD : {"text": "weight loss", "label": "Symptom", "status": "Confirmed"} when patient says "I would like to lose weight" - this is a goal, not an actual symptom
GOOD: skip it OR {"text": "lose weight", "label": "NULL", "status": "Confirmed"} (a goal, not a finding)
BAD : {"text": "surgery", "label": "Procedure", "status": "Confirmed"} when patient says "I am considering surgery" - the procedure has NOT been ordered or performed
GOOD: skip it (no procedure has actually been ordered)
BAD : {"text": "twice a day", "label": "Drug"} - dosage fragment, no drug
GOOD: skip it
BAD : {"text": "shoveling", "label": "Symptom"} - activity, not symptom
GOOD: skip it
BAD : {"text": "smoke", "label": "Procedure"} - substance use is Drug
GOOD: {"text": "smoke", "label": "Drug", "status": "Confirmed"}
BAD : {"text": "heart attack", "label": "Disease", "status": "Confirmed"} - when father had it
GOOD: {"text": "heart attack", "label": "Disease", "status": "Family_History"}

EXAMPLE 1 - basic negatives + affirmatives:
Transcript: "Do you have chest pain? Yes. Any fevers? No. Sweaty? Just from breathing."
Output:
[
  {"text": "chest pain", "label": "Symptom", "status": "Confirmed"},
  {"text": "fevers", "label": "Symptom", "status": "Negated"},
  {"text": "Sweaty", "label": "Symptom", "status": "Confirmed"}
]

EXAMPLE 2 - drugs, family, social:
Transcript: "I take Metformin 500mg. My father had a heart attack. I smoke a pack a day. I'm an accountant."
Output:
[
  {"text": "Metformin 500mg", "label": "Drug", "status": "Confirmed"},
  {"text": "heart attack", "label": "Disease", "status": "Family_History"},
  {"text": "smoke", "label": "Drug", "status": "Confirmed"},
  {"text": "accountant", "label": "NULL", "status": "Confirmed"}
]

EXAMPLE 3 - negation cluster + historical:
Transcript: "Patient denies fever. We'll work up for pulmonary embolism. She had pneumonia two years ago that resolved."
Output:
[
  {"text": "fever", "label": "Symptom", "status": "Negated"},
  {"text": "pulmonary embolism", "label": "Disease", "status": "Negated"},
  {"text": "pneumonia", "label": "Disease", "status": "Historical"}
]

OUTPUT: Return ONLY a JSON array. No other text.
""").strip()

_NER_RETRY_PROMPT = textwrap.dedent("""
Extract medical entities from the transcript as a JSON array.
Each entity: {"text": "exact words from transcript", "label": "Drug|Disease|Symptom|Procedure|NULL", "status": "Confirmed|Negated|Historical|Family_History"}
Include symptoms the patient confirms AND denies.
Return ONLY the JSON array.
""").strip()

def run_ner_spell(source):
    print("  [NER] Phase 1: LLM extraction (3-shot)...")
    t0 = time.time()

    msg = [
        {"role": "system", "content": _NER_PROMPT},
        {"role": "user", "content": (
            f"Transcript:\n{source}\n\n"
            "Extract ALL entities. EXACT SUBSTRINGS only. JSON array:"
        )}
    ]
    raw = _llm(msg, max_tokens=3500, temperature=0.1)
    candidates = _parse_json(raw, "list")
    print(f"         -> {len(candidates)} raw candidates in {time.time()-t0:.1f}s")

    if len(candidates) == 0:
        print("  [NER] RETRY: 0 candidates, using simpler prompt...")
        retry_msg = [
            {"role": "system", "content": _NER_RETRY_PROMPT},
            {"role": "user", "content": f"Transcript:\n{source}\n\nJSON array:"}
        ]
        raw = _llm(retry_msg, max_tokens=3000, temperature=0.1)
        candidates = _parse_json(raw, "list")
        print(f"         -> {len(candidates)} candidates after retry in {time.time()-t0:.1f}s")

    schema_clean, schema_rejected = _enforce_schema(candidates)

    print("  [NER] Phase 3: Non-clinical filter (regex only)...")
    clinical, nonclinical_rejected = [], []
    for ent in schema_clean:
        if _is_non_clinical(ent["text"]):
            nonclinical_rejected.append({**ent, "_reason": "Non-clinical (regex)"})
            print(f"         JUNK: \"{ent['text']}\"")
            continue
        clinical.append(ent)

    print("  [NER] Phase 4: SPELL verification...")
    verified, spell_rejected = [], []
    for ent in clinical:
        offset = _spell_verify_and_align(ent["text"], source)
        if offset is not None:
            ent["first_offset"] = offset
            verified.append(ent)
        else:
            spell_rejected.append({**ent, "_reason": "SPELL: not in transcript"})
            print(f"         REJECTED: \"{ent['text']}\"")

    print("  [NER] Phase 5: Offset-aware negation (hardened)...")
    for e in verified:
        old = e["status"]
        e["status"] = _triple_negation(
            e["text"], source, old, e["label"],
            spell_offset=e.get("first_offset")
        )
        if e["status"] != old:
            e["_neg_fix"] = f"{old}->{e['status']}"

    # Phase 6: VerifyNER-style 2nd pass (new in 2026-05 upgrade)
    verify_rejected = []
    if globals().get("USE_NER_VERIFICATION", True):
        print("  [NER] Phase 6: LLM verification (VerifyNER-style)...")
        verified, verify_rejected = verify_entities_llm(verified, source, batch_size=15)

    all_rejected = schema_rejected + nonclinical_rejected + spell_rejected + verify_rejected

    print(f"  [NER] Final: {len(verified)} verified, {len(all_rejected)} rejected")
    return verified, all_rejected

print("Cell 7 done: NER with 3-shot prompt + verification pass")


Cell 7 done: NER with 3-shot prompt + verification pass


In [10]:
# Cell 8 - SOAP generation from structured facts (extractive only) + separate AI Suggestion call.
# This implements the 2026-05 quality upgrade:
#   1. SOAP body is STRICTLY extractive - paraphrases ONLY transcript content.
#   2. Hard-default empty O / P sections in Python (skip LLM).
#   3. AI Suggestion is a SEPARATE LLM call so it is ALWAYS populated.
#   4. Optional self-consistency sampling (n=3) when USE_SELF_CONSISTENCY=True.

def build_fact_table(entities, transcript):
    """
    Build a structured clinical fact table from NER entities.
    The SOAP generator uses THIS, not the raw transcript -> bounds the LLM to facts only.
    """
    facts = {
        "confirmed_symptoms": [],
        "negated_symptoms": [],
        "diseases": [],
        "family_history": [],
        "historical": [],
        "medications_current": [],
        "medications_denied": [],
        "allergies": [],
        "social_history": [],
        "vitals_and_exam": [],
        "procedures_ordered": [],
        "procedures_denied": [],
        "unclassified": [],
    }

    for e in entities:
        if e.get("label") == "NULL":
            text = e.get("text", "").strip()
            status = e.get("status", "Confirmed")
            if status == "Confirmed":
                facts["unclassified"].append(text)

    social_keywords = {"smoke", "cigarette", "tobacco", "cannabis", "marijuana",
                       "alcohol", "drink", "cocaine", "meth", "opioid", "heroin",
                       "iv drug", "recreational"}

    for e in entities:
        if e.get("label") == "NULL":
            continue
        text = e["text"]
        label = e["label"]
        status = e["status"]
        norm = e.get("normalized")
        display = norm if (norm and norm.lower() != text.lower()) else text

        offset = e.get("first_offset")
        context = ""
        if offset and offset.get("start") is not None:
            s = max(0, offset["start"] - 60)
            end = min(len(transcript), offset["end"] + 60)
            context = transcript[s:end].strip()

        entry = display
        if context and status == "Confirmed" and label == "Symptom":
            entry = f"{display} [context: {context}]"

        if label == "Symptom":
            if status == "Confirmed":
                facts["confirmed_symptoms"].append(entry)
            elif status == "Negated":
                facts["negated_symptoms"].append(display)
            elif status == "Family_History":
                facts["family_history"].append(display)
            elif status == "Historical":
                facts["historical"].append(display)

        elif label == "Disease":
            if status == "Confirmed":
                facts["diseases"].append(display)
            elif status == "Family_History":
                facts["family_history"].append(display)
            elif status == "Historical":
                facts["historical"].append(display)

        elif label == "Drug":
            is_social = any(kw in text.lower() for kw in social_keywords)
            if is_social:
                if status == "Confirmed":
                    facts["social_history"].append(f"{text} [context: {context}]" if context else text)
                elif status == "Negated":
                    facts["social_history"].append(f"Denies {text}")
            else:
                if status == "Confirmed":
                    facts["medications_current"].append(text)
                elif status == "Negated":
                    facts["medications_denied"].append(text)

        elif label == "Procedure":
            if status == "Confirmed":
                facts["procedures_ordered"].append(text)
            elif status == "Negated":
                facts["procedures_denied"].append(text)

    lines = []
    for key, items in facts.items():
        header = key.replace("_", " ").upper()
        if items:
            lines.append(f"{header}:")
            for item in items:
                lines.append(f"  - {item}")
        else:
            lines.append(f"{header}: None documented")
    return "\n".join(lines), facts


# --- STRICTLY EXTRACTIVE SOAP body prompt ---
_SOAP_BODY_SYSTEM = """You are a strict EXTRACTIVE Clinical Scribe. You PARAPHRASE the structured fact table into SOAP format. You NEVER add medical reasoning, diagnoses, or speculation.

OUTPUT FORMAT (plain text, no markdown, no bold, no bullet symbols):
S: Subjective - paraphrase the confirmed symptoms, history, and social context from the fact table. End with pertinent negatives ("Denies X, Y, Z") from the negated_symptoms section.
O: Objective - paraphrase ONLY items from the vitals_and_exam section. If none, write exactly: No objective findings recorded in transcript.
A: Assessment - a NEUTRAL SUMMARY of the patient's confirmed clinical picture. Do NOT diagnose. Do NOT use words like "suspected", "likely", "rule out", "concerning for", "consistent with". Just summarize: "Patient presents with [confirmed symptoms]. Relevant history includes [historical/family]. Current medications: [meds]."
P: Plan - paraphrase ONLY items from the procedures_ordered section. If none, write exactly: No specific medical orders were recorded in the transcript.

THINK STEP-BY-STEP INTERNALLY before writing. Do NOT include the thinking - output only the four SOAP lines.

ABSOLUTE RULES:
1. EVERY claim in S, O, A, P must come from the fact table. If it is not in the table, it does NOT appear.
2. Denied symptoms go in S as pertinent negatives only ("Denies fever, vomiting, chest pain").
3. The Assessment is descriptive ONLY, never diagnostic.
4. No "[Disclaimer]" line, no "AI Suggestion" - those are generated separately.
5. Plain text. No **bold**. No #headings. No bullet points (-, *, .).

BAD vs GOOD examples (NEVER do BAD):
BAD : A: Concerning for acute coronary syndrome. Recommend rule out MI.
GOOD: A: Patient presents with exertional chest pressure, dyspnea, and family history of premature MI. Currently taking Lisinopril 10mg and Aspirin 81mg.
BAD : P: Standard workup should include serial troponins, lipid panel, stress testing.
GOOD: P: Doctor ordered EKG, serum troponin levels, and chest X-ray.
BAD : O: Vital signs appear stable, patient is in no acute distress.
GOOD: O: Blood pressure 148/92 mmHg. Heart rate 88 bpm."""

_SOAP_BODY_USER = """FACT TABLE:
{fact_table}

Write the SOAP note now. S, O, A, P only. No markdown, no disclaimers, no AI suggestions.
S:"""


# --- AI Suggestion prompt (separate, structured, always called) ---
_AI_SUGGESTION_SYSTEM = """You are a clinical AI providing brief decision support to a physician AFTER they have written their SOAP note. This is the ONLY place clinical reasoning belongs.

OUTPUT FORMAT (plain text, 4 short lines, no markdown, no bullets, total under 80 words):
Differential: <1-2 most likely diagnoses, comma-separated, no reasoning>
Workup: <1-3 specific tests or actions, comma-separated>
Red flags: <symptoms warranting urgent escalation, or "None">
Follow-up: <appropriate timeframe>

HARD RULES:
- Be BRIEF and SPECIFIC to the SOAP content. No generic boilerplate.
- If the SOAP note is sparse or empty, output exactly: "Insufficient SOAP detail for clinical suggestions."
- Plain text only. No bold, no markdown headers."""

_AI_SUGGESTION_USER = """Here is the SOAP note for the encounter:

{soap_body}

Now generate the AI Suggestion section."""


def _build_soap_body_messages(fact_table):
    return [
        {"role": "system", "content": _SOAP_BODY_SYSTEM},
        {"role": "user", "content": _SOAP_BODY_USER.format(fact_table=fact_table)},
    ]


def _parse_soap_sections(raw_text):
    cleaned = re.sub(r'\*\*', '', raw_text)
    cleaned = re.sub(r'#{1,4}\s*', '', cleaned)
    soap = {"S": "", "O": "", "A": "", "P": ""}
    pattern = re.compile(r'(?:^|\n)\s*([SOAP])\s*:\s*', re.MULTILINE)
    matches = list(pattern.finditer(cleaned))
    for i, m in enumerate(matches):
        key = m.group(1)
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(cleaned)
        content = cleaned[start:end].strip()
        if key in soap and not soap[key]:
            soap[key] = content
    return soap


def _score_soap_candidate(soap, facts):
    """Score a candidate SOAP by hallucination risk + fact coverage. Lower is better."""
    soap_text = " ".join(soap.get(k, "") for k in "SOAP").lower()
    hallucination_phrases = [
        "suspected", "likely", "concerning for", "rule out", "consistent with",
        "in keeping with", "acute coronary syndrome", "myocardial infarction",
        "suggestive of", "differential diagnosis", "workup includes",
    ]
    hallucination_count = sum(1 for p in hallucination_phrases if p in soap_text)

    all_facts = []
    for k in ("confirmed_symptoms", "diseases", "medications_current", "vitals_and_exam", "procedures_ordered"):
        all_facts.extend(facts.get(k, []))
    if not all_facts:
        coverage = 1.0
    else:
        found = 0
        for fact in all_facts:
            key = re.sub(r"\[context:.*?\]", "", str(fact)).strip().lower()
            words = [w for w in key.split() if len(w) >= 4]
            if not words:
                continue
            if any(w in soap_text for w in words[:3]):
                found += 1
        coverage = found / max(len(all_facts), 1)
    score = hallucination_count - coverage
    return score, hallucination_count, coverage


def _generate_soap_body_once(fact_table, facts, seed=None):
    raw = _llm(_build_soap_body_messages(fact_table), max_tokens=900, temperature=0.1, seed=seed)
    soap = _parse_soap_sections(raw)

    # Hard-default empty sections (skip LLM entirely)
    if not facts.get("vitals_and_exam"):
        soap["O"] = "No objective findings recorded in transcript."
    if not facts.get("procedures_ordered"):
        soap["P"] = "No specific medical orders were recorded in the transcript."
    if not soap.get("S"):
        # build a deterministic S from the fact table so the section never disappears
        s_parts = []
        if facts.get("confirmed_symptoms"):
            s_parts.append("Patient reports " + ", ".join(
                re.sub(r"\s*\[context:.*?\]", "", x).strip()
                for x in facts["confirmed_symptoms"]) + ".")
        if facts.get("historical"):
            s_parts.append("History of " + ", ".join(facts["historical"]) + ".")
        if facts.get("family_history"):
            s_parts.append("Family history of " + ", ".join(facts["family_history"]) + ".")
        if facts.get("social_history"):
            s_parts.append("Social history: " + "; ".join(facts["social_history"]) + ".")
        if facts.get("medications_current"):
            s_parts.append("Current medications: " + ", ".join(facts["medications_current"]) + ".")
        if facts.get("negated_symptoms"):
            s_parts.append("Denies " + ", ".join(facts["negated_symptoms"]) + ".")
        if not facts.get("unclassified") and not s_parts:
            s_parts.append("No subjective findings recorded in transcript.")
        soap["S"] = " ".join(s_parts) if s_parts else "No subjective findings recorded in transcript."

    # Hard-default A section if LLM omitted it
    if not soap.get("A"):
        a_parts = []
        if facts.get("confirmed_symptoms"):
            a_parts.append("Patient presents with " + ", ".join(
                re.sub(r"\s*\[context:.*?\]", "", x).strip()
                for x in facts["confirmed_symptoms"]) + ".")
        if facts.get("diseases"):
            a_parts.append("Reported diagnoses: " + ", ".join(facts["diseases"]) + ".")
        if facts.get("family_history"):
            a_parts.append("Notable family history: " + ", ".join(facts["family_history"]) + ".")
        if facts.get("medications_current"):
            a_parts.append("Current medications: " + ", ".join(facts["medications_current"]) + ".")
        if not a_parts:
            a_parts.append("No assessment-relevant findings recorded in transcript.")
        soap["A"] = " ".join(a_parts)

    # Hard-default P section if LLM omitted it (only when procedures_ordered is non-empty)
    if not soap.get("P") and facts.get("procedures_ordered"):
        soap["P"] = "Doctor ordered " + ", ".join(facts["procedures_ordered"]) + "."

    score, halluc, cov = _score_soap_candidate(soap, facts)
    return soap, raw, score, halluc, cov


def _generate_ai_suggestion(soap_body_text):
    # Skip if SOAP body has no real content (only default placeholders)
    placeholders = (
        "No subjective findings recorded in transcript.",
        "No objective findings recorded in transcript.",
        "No specific medical orders were recorded in the transcript.",
        "No reported symptoms.",
        "No clinical information available.",
        "No clinical keywords extracted.",
    )
    stripped = soap_body_text
    for p in placeholders:
        stripped = stripped.replace(p, "")
    stripped = re.sub(r'[KSOAP]:\s*', '', stripped).strip()
    if len(stripped) < 40:
        return "Insufficient SOAP detail for clinical suggestions."

    msg = [
        {"role": "system", "content": _AI_SUGGESTION_SYSTEM},
        {"role": "user", "content": _AI_SUGGESTION_USER.format(soap_body=soap_body_text)},
    ]
    raw = _llm(msg, max_tokens=220, temperature=0.2)
    cleaned = re.sub(r'\*\*', '', raw).strip()
    cleaned = re.sub(r'^#{1,4}\s*', '', cleaned, flags=re.MULTILINE)
    return cleaned


def run_soap(source, entities=None):
    print("  [SOAP] Building fact table from NER entities...")
    t0 = time.time()

    if entities:
        fact_table, facts = build_fact_table(entities, source)
        print(f"  [SOAP] Fact table: {len(fact_table)} chars")
    else:
        facts = {"confirmed_symptoms": [], "negated_symptoms": [], "diseases": [],
                 "family_history": [], "historical": [], "medications_current": [],
                 "medications_denied": [], "allergies": [], "social_history": [],
                 "vitals_and_exam": [], "procedures_ordered": [], "procedures_denied": [],
                 "unclassified": []}
        fact_table = f"TRANSCRIPT (no fact table available):\n{source[:2000]}"

    if globals().get("USE_SELF_CONSISTENCY", False) and entities:
        n = globals().get("SELF_CONSISTENCY_N", 3)
        print(f"  [SOAP] Self-consistency: generating {n} candidates...")
        cands = []
        for k in range(n):
            soap_k, _, score, halluc, cov = _generate_soap_body_once(fact_table, facts, seed=42 + k * 7)
            cands.append((score, halluc, cov, soap_k))
            print(f"    cand {k}: score={score:.2f} halluc={halluc} coverage={cov:.2f}")
        cands.sort(key=lambda x: x[0])
        score, halluc, cov, soap = cands[0]
        print(f"  [SOAP] Best candidate: score={score:.2f}")
    else:
        soap, _, score, halluc, cov = _generate_soap_body_once(fact_table, facts)
        print(f"  [SOAP] Body: score={score:.2f} halluc={halluc} coverage={cov:.2f}")

    # Strip speculation words from every section
    for _k in "SOAP":
        if soap.get(_k):
            soap[_k] = sanitize_assessment(soap[_k])

    soap_body_text = "\n\n".join(f"{k}: {soap[k]}" for k in "SOAP" if soap[k])

    print("  [AI Suggestion] Generating dedicated suggestion section...")
    ai_suggestion_body = _generate_ai_suggestion(soap_body_text)
    ai_suggestion = f"[Disclaimer]\nAI Suggestion (clinical decision support; NOT part of the SOAP note above):\n{ai_suggestion_body}"

    soap["AI Suggestion"] = ai_suggestion_body
    if entities:
        soap["fact_table"] = fact_table

    soap_raw_parts = [f"{k}: {soap[k]}" for k in "SOAP" if soap[k]]
    soap_raw_parts.append("")
    soap_raw_parts.append(ai_suggestion)
    soap_raw = "\n\n".join(soap_raw_parts).strip()

    elapsed = time.time() - t0
    print(f"  [SOAP] Done in {elapsed:.1f}s (body + ai suggestion)")
    return soap, soap_raw




# --- Post-hoc speculation sanitizer for SOAP A section (extractive enforcement) ---
_SPECULATION_PATTERNS = [
    (re.compile(r'\b(?:suspected|likely|probable|possible|presumed)\s+', re.I), ''),
    (re.compile(r'\bconcerning\s+for\s+', re.I), 'presents with '),
    (re.compile(r'\bsuggestive\s+of\s+', re.I), 'presents with '),
    (re.compile(r'\bconsistent\s+with\s+', re.I), 'presents with '),
    (re.compile(r'\brule\s+out\s+', re.I), 'evaluate for '),
    (re.compile(r'\bdifferential\s+diagnos[ie]s?\s+(?:includes?|:)?\s*', re.I), ''),
    (re.compile(r'\bworkup\s+(?:should\s+)?includes?\s+', re.I), ''),
    (re.compile(r'\bin\s+keeping\s+with\s+', re.I), 'presents with '),
    (re.compile(r'\bmay\s+(?:have|be)\s+', re.I), 'reports '),
    (re.compile(r'\bmight\s+(?:have|be)\s+', re.I), 'reports '),
    (re.compile(r'\bcould\s+(?:have|be)\s+', re.I), 'reports '),
]

def sanitize_assessment(text):
    """Strip speculation words from any S/O/A/P section to enforce extractive output."""
    if not text:
        return text
    out = text
    for pat, repl in _SPECULATION_PATTERNS:
        out = pat.sub(repl, out)
    # Collapse double spaces
    out = re.sub(r'\s+', ' ', out).strip()
    return out


run_soap_v1_baseline = run_soap
print("Cell 8 done: extractive SOAP + separate AI Suggestion (always populated)")


Cell 8 done: extractive SOAP + separate AI Suggestion (always populated)


In [11]:
# Cell 9 - SNOMED CT: 300+ keyword entries covering 20 clinical domains, context-aware disambiguation, SapBERT semantic fallback with cross-encoder reranking
SNOMED_TABLE = [
    {"id":"422587007","t":"Nausea","c":"R11.0","kw":["nausea","nauseated","nauseous","feeling sick"]},
    {"id":"422400008","t":"Vomiting","c":"R11.10","kw":["vomiting","vomit","throw up","throwing up","emesis"]},
    {"id":"73879007","t":"Nausea and vomiting","c":"R11.2","kw":["nausea and vomiting","nausea with vomiting"]},
    {"id":"62315008","t":"Diarrhea","c":"R19.7","kw":["diarrhea","diarrhoea","loose stools","watery stools","diadea"]},
    {"id":"14760008","t":"Constipation","c":"K59.00","kw":["constipation","constipated"]},
    {"id":"21522001","t":"Abdominal pain","c":"R10.9","kw":["abdominal pain","belly pain","stomach pain","stomachache","abdominal tenderness"]},
    {"id":"73063007","t":"Abdominal cramps","c":"R10.84","kw":["cramps","crampy","cramping","abdominal cramps","stomach cramps"]},
    {"id":"698065002","t":"Heartburn","c":"R12","kw":["heartburn","acid reflux","pyrosis"]},
    {"id":"235365009","t":"GERD","c":"K21.0","kw":["gerd","gastroesophageal reflux","reflux disease"]},
    {"id":"235856003","t":"Gastroenteritis","c":"K52.9","kw":["gastroenteritis","stomach flu","stomach bug","food poisoning"]},
    {"id":"4556007","t":"Gastritis","c":"K29.70","kw":["gastritis"]},
    {"id":"64226004","t":"Dysphagia","c":"R13.10","kw":["dysphagia","difficulty swallowing","trouble swallowing"]},
    {"id":"271681002","t":"Loss of appetite","c":"R63.0","kw":["loss of appetite","not eating","not hungry","decreased appetite","anorexia"]},
    {"id":"249497008","t":"Bloating","c":"R14.0","kw":["bloating","bloated","distended","abdominal distension"]},
    {"id":"162607003","t":"Blood in stool","c":"K92.1","kw":["blood in stool","blood in stools","blood in your stools","rectal bleeding","bloody stool","melena","hematochezia"]},
    {"id":"422823003","t":"Hematemesis","c":"K92.0","kw":["blood in vomit","hematemesis","vomiting blood"]},
    {"id":"197456007","t":"IBS","c":"K58.9","kw":["irritable bowel","ibs"]},
    {"id":"396332003","t":"Appendicitis","c":"K35.80","kw":["appendicitis"]},
    {"id":"34000006","t":"Crohn disease","c":"K50.90","kw":["crohn","crohn's"]},
    {"id":"64766004","t":"Ulcerative colitis","c":"K51.90","kw":["ulcerative colitis","colitis"]},
    {"id":"44054006","t":"Bowel obstruction","c":"K56.60","kw":["bowel obstruction","intestinal obstruction"]},
    {"id":"60728008","t":"Jaundice","c":"R17","kw":["jaundice","yellow skin","icterus","yellow eyes"]},

    {"id":"29857009","t":"Chest pain","c":"R07.9","kw":["chest pain","chest tightness","tight feeling in my chest","pressure on my chest","chest pressure"]},
    {"id":"80313002","t":"Palpitations","c":"R00.2","kw":["palpitations","heart racing","racing of the heart","heart pounding","beating faster","heart fluttering","thumping"]},
    {"id":"3424008","t":"Tachycardia","c":"R00.0","kw":["tachycardia","fast heart rate","rapid heart rate"]},
    {"id":"48867003","t":"Bradycardia","c":"R00.1","kw":["bradycardia","slow heart rate"]},
    {"id":"42343007","t":"Heart failure","c":"I50.9","kw":["heart failure","chf","congestive heart failure","fluid overload"]},
    {"id":"22298006","t":"Myocardial infarction","c":"I21.9","kw":["heart attack","myocardial infarction","mi","stemi","nstemi"]},
    {"id":"194828000","t":"Angina","c":"I20.9","kw":["angina","angina pectoris"]},
    {"id":"38341003","t":"Hypertension","c":"I10","kw":["high blood pressure","hypertension","elevated bp","elevated blood pressure","blood pressure high"]},
    {"id":"45007003","t":"Hypotension","c":"I95.9","kw":["low blood pressure","hypotension"]},
    {"id":"49436004","t":"Atrial fibrillation","c":"I48.91","kw":["atrial fibrillation","afib","a-fib","irregular heartbeat"]},
    {"id":"233604007","t":"Pericarditis","c":"I30.9","kw":["pericarditis","pericardial"]},
    {"id":"253273006","t":"Heart murmur","c":"R01.1","kw":["heart murmur","murmur"]},
    {"id":"59282003","t":"Pulmonary edema","c":"J81.0","kw":["fluid in lungs","pulmonary edema","fluid in the lungs"]},

    {"id":"49727002","t":"Cough","c":"R05.9","kw":["cough","coughing","persistent cough","dry cough","productive cough"]},
    {"id":"267036007","t":"Dyspnea","c":"R06.00","kw":["shortness of breath","difficulty breathing","trouble breathing","short of breath","breathless","dyspnea","winded","gasping for air"]},
    {"id":"56018004","t":"Wheezing","c":"R06.2","kw":["wheeze","wheezing"]},
    {"id":"48409008","t":"Orthopnea","c":"R06.01","kw":["orthopnea","can't breathe lying flat","breathe lying flat","prop up with pillows","pillows at night"]},
    {"id":"70572006","t":"Stridor","c":"R06.1","kw":["stridor","noisy breathing"]},
    {"id":"195967001","t":"Asthma","c":"J45.909","kw":["asthma","asthmatic","reactive airway"]},
    {"id":"13645005","t":"COPD","c":"J44.1","kw":["copd","chronic obstructive","emphysema"]},
    {"id":"233604007","t":"Pneumonia","c":"J18.9","kw":["pneumonia"]},
    {"id":"233703007","t":"Pulmonary embolism","c":"I26.99","kw":["pulmonary embolism","pe","blood clot in lung"]},
    {"id":"18165001","t":"Pleural effusion","c":"J90","kw":["pleural effusion","fluid around lungs"]},
    {"id":"65124004","t":"Crackles","c":"R09.89","kw":["crackles","rales","crepitations","bibasilar crackles"]},
    {"id":"267038008","t":"Edema","c":"R60.0","kw":["swelling","edema","oedema","swollen","pitting edema","ankle swelling","swelling in my ankles","swelling in ankles","leg swelling"]},
    {"id":"87433001","t":"Hemoptysis","c":"R04.2","kw":["coughing blood","hemoptysis","blood in sputum","bringing up blood"]},
    {"id":"68235000","t":"Nasal congestion","c":"R09.81","kw":["congestion","stuffy nose","blocked nose","runny nose","nasal congestion","rhinorrhea"]},
    {"id":"162397003","t":"Sore throat","c":"R07.0","kw":["sore throat","throat pain","pharyngitis","scratchy throat"]},
    {"id":"301354004","t":"Mucus production","c":"R09.3","kw":["mucus","phlegm","sputum","clear mucus"]},

    {"id":"25064002","t":"Headache","c":"R51.9","kw":["headache","headaches","head pain","cephalgia"]},
    {"id":"37796009","t":"Migraine","c":"G43.909","kw":["migraine","migraines"]},
    {"id":"404640003","t":"Dizziness","c":"R42","kw":["dizziness","dizzy","lightheaded","lightheadedness","vertigo","room spinning"]},
    {"id":"271594007","t":"Syncope","c":"R55","kw":["syncope","fainted","fainting","faint","passed out","loss of consciousness","lost consciousness","blacked out"]},
    {"id":"62106007","t":"Numbness","c":"R20.0","kw":["numbness","numb","tingling","paresthesia","pins and needles"]},
    {"id":"26079004","t":"Tremor","c":"R25.1","kw":["tremor","tremors","shaking","shaky"]},
    {"id":"91175000","t":"Seizure","c":"R56.9","kw":["seizure","seizures","convulsion","convulsions","fit","fits"]},
    {"id":"230690007","t":"Stroke","c":"I63.9","kw":["stroke","cva","cerebrovascular"]},
    {"id":"52448006","t":"Dementia","c":"F03.90","kw":["dementia","alzheimer","memory loss","cognitive decline"]},
    {"id":"23056005","t":"Sciatica","c":"M54.30","kw":["sciatica","sciatic","shooting pain down leg"]},
    {"id":"128196005","t":"Confusion","c":"R41.0","kw":["confusion","confused","disoriented","altered mental status"]},
    {"id":"40917007","t":"Visual disturbance","c":"H53.9","kw":["blurry vision","blurred vision","double vision","diplopia","visual changes"]},

    {"id":"161891005","t":"Back pain","c":"M54.9","kw":["back pain","backache","lower back pain","back is killing","lumbago"]},
    {"id":"57676002","t":"Joint pain","c":"M25.50","kw":["joint pain","arthralgia","knee pain","hip pain","shoulder pain","elbow pain","wrist pain"]},
    {"id":"68962001","t":"Muscle pain","c":"M79.10","kw":["muscle aches","muscle pain","myalgia","body aches","sore muscles"]},
    {"id":"55300003","t":"Muscle cramp","c":"R25.2","kw":["muscle cramp","muscle spasm","charley horse"]},
    {"id":"263171000","t":"Stiffness","c":"M25.60","kw":["stiffness","stiff","joint stiffness"]},
    {"id":"125605004","t":"Fracture","c":"T14.8XXA","kw":["fracture","broken bone","broken"]},
    {"id":"44465007","t":"Sprain","c":"S93.409A","kw":["sprain","sprained","rolled ankle","twisted","ankle injury","ankle sprain"]},
    {"id":"299308007","t":"Neck pain","c":"M54.2","kw":["neck pain","stiff neck","neck stiffness"]},
    {"id":"239872002","t":"Limited ROM","c":"M25.60","kw":["limited range of motion","can't move","difficulty moving","limited mobility"]},

    {"id":"73211009","t":"Diabetes mellitus","c":"E11.9","kw":["diabetes","diabetic","type 2 diabetes","type 1 diabetes","sugar","blood sugar"]},
    {"id":"40930008","t":"Hypothyroidism","c":"E03.9","kw":["hypothyroidism","underactive thyroid","low thyroid"]},
    {"id":"34095006","t":"Dehydration","c":"E86.0","kw":["dehydration","dehydrated"]},
    {"id":"190372001","t":"Polydipsia","c":"R63.1","kw":["polydipsia","excessive thirst","very thirsty","thirsty","increased thirst"]},
    {"id":"55822004","t":"Hyperlipidemia","c":"E78.5","kw":["high cholesterol","cholesterol","cholesterol problems","hyperlipidemia","dyslipidemia","triglycerides"]},
    {"id":"5291005","t":"Hypoglycemia","c":"E16.2","kw":["hypoglycemia","low blood sugar","low sugar"]},
    {"id":"14304000","t":"Thyroid disorder","c":"E07.9","kw":["thyroid","thyroid disorder","thyroid problem","goiter"]},
    {"id":"190268003","t":"Obesity","c":"E66.9","kw":["obesity","obese","overweight","bmi"]},
    {"id":"238131007","t":"Weight loss","c":"R63.4","kw":["weight loss","losing weight","lost weight","unintentional weight loss"]},
    {"id":"8943002","t":"Weight gain","c":"R63.5","kw":["weight gain","gaining weight","gained weight"]},

    {"id":"162116003","t":"Urinary frequency","c":"R35.0","kw":["urinary frequency","frequent urination","peeing a lot","pee a lot","going to bathroom often"]},
    {"id":"49650001","t":"Dysuria","c":"R30.0","kw":["dysuria","pain when peeing","painful urination","burning urination","pain when urinating"]},
    {"id":"75088002","t":"Urinary urgency","c":"R39.15","kw":["urinary urgency","urgent urination","need to go urgently"]},
    {"id":"165232002","t":"Urinary incontinence","c":"R32","kw":["incontinence","urine leakage","leaking urine","can't hold it"]},
    {"id":"68566005","t":"UTI","c":"N39.0","kw":["uti","urinary tract infection","bladder infection"]},
    {"id":"34436003","t":"Hematuria","c":"R31.9","kw":["hematuria","blood in urine","bloody urine"]},
    {"id":"236578006","t":"Kidney stones","c":"N20.0","kw":["kidney stone","kidney stones","renal calculus","nephrolithiasis"]},
    {"id":"197927001","t":"Flank pain","c":"R10.9","kw":["flank pain","kidney pain","side pain"]},

    {"id":"77386006","t":"Pregnancy","c":"Z33.1","kw":["pregnancy","pregnant","expecting"]},
    {"id":"14094001","t":"Amenorrhea","c":"N91.2","kw":["amenorrhea","missed period","late period","no period","absent menstruation","period missed","period late","missed periods"]},
    {"id":"289903006","t":"Menstrual finding","c":"N94.89","kw":["period","periods","menstrual","menstruation","menses"]},
    {"id":"266897007","t":"Menstrual irregularity","c":"N92.6","kw":["irregular period","irregular periods","irregular menstruation"]},
    {"id":"418290006","t":"Dysmenorrhea","c":"N94.6","kw":["painful period","period pain","menstrual cramps","menstrual pain","dysmenorrhea"]},
    {"id":"237079002","t":"Morning sickness","c":"O21.0","kw":["morning sickness"]},
    {"id":"289908002","t":"Pregnancy test positive","c":"Z32.01","kw":["pregnancy test positive","positive pregnancy test"]},
    {"id":"169553002","t":"Contraception","c":"Z30.9","kw":["birth control","oral contraceptive","contraception","contraceptive","condom","condoms","iud"]},

    {"id":"386661006","t":"Fever","c":"R50.9","kw":["fever","fevers","febrile","temperature elevated","hot today","feel hot","feeling hot","warm at night"]},
    {"id":"43724002","t":"Chills","c":"R68.83","kw":["chills","shivering","rigors"]},
    {"id":"84229001","t":"Fatigue","c":"R53.83","kw":["fatigue","tired","exhausted","exhaustion","malaise","lethargy","low energy","feeling tired"]},
    {"id":"42984000","t":"Night sweats","c":"R61","kw":["night sweats","sweating at night","sweats"]},
    {"id":"271807003","t":"Rash","c":"R21","kw":["rash","rashes","skin rash","eruption","hives","urticaria"]},
    {"id":"64779008","t":"Blood","c":"R58","kw":["blood","bleeding","hemorrhage"]},

    {"id":"197480006","t":"Anxiety","c":"F41.9","kw":["anxiety","anxious","worried","worry","panic","panic attack","nervousness"]},
    {"id":"35489007","t":"Depression","c":"F32.9","kw":["depression","depressed","feeling down","low mood","sadness"]},
    {"id":"193462001","t":"Insomnia","c":"G47.00","kw":["insomnia","can't sleep","difficulty sleeping","trouble sleeping","sleep problems"]},
    {"id":"3415004","t":"Substance use","c":"F19.10","kw":["substance use","drug use","recreational drugs","illicit drugs"]},

    {"id":"6142004","t":"Influenza","c":"J11.1","kw":["flu","influenza","flu-like symptoms","flu-like"]},
    {"id":"186747009","t":"COVID-19","c":"U07.1","kw":["covid","covid-19","coronavirus","sars-cov-2"]},
    {"id":"36971009","t":"Sinusitis","c":"J32.9","kw":["sinusitis","sinus infection","sinus"]},
    {"id":"55735004","t":"Bronchitis","c":"J20.9","kw":["bronchitis"]},
    {"id":"302911003","t":"Pharyngitis","c":"J02.9","kw":["pharyngitis","strep throat"]},

    {"id":"294431004","t":"Nickel allergy","c":"L23.0","kw":["nickel","nickel allergy","jewelry allergy","jewelry","contact dermatitis"]},
    {"id":"43116000","t":"Eczema","c":"L30.9","kw":["eczema","atopic dermatitis"]},
    {"id":"9014002","t":"Psoriasis","c":"L40.9","kw":["psoriasis"]},
    {"id":"126485001","t":"Cellulitis","c":"L03.90","kw":["cellulitis"]},
    {"id":"271756007","t":"Pruritus","c":"L29.9","kw":["itching","itchy","pruritus"]},
    {"id":"95324001","t":"Bruising","c":"R23.3","kw":["bruise","bruising","ecchymosis"]},
    {"id":"95328003","t":"Skin lesion","c":"L98.9","kw":["lesion","sore","wound","ulcer","skin lesion"]},

    {"id":"419199007","t":"Allergy","c":"T78.40XA","kw":["allergy","allergic","allergies"]},
    {"id":"91936005","t":"Penicillin allergy","c":"Z88.0","kw":["penicillin allergy","allergic to penicillin"]},
    {"id":"300916003","t":"Latex allergy","c":"L23.4","kw":["latex allergy","allergic to latex"]},
    {"id":"414285001","t":"Food allergy","c":"T78.1XXA","kw":["food allergy","allergic to food"]},
    {"id":"213020009","t":"Drug allergy","c":"T88.7XXA","kw":["drug allergy","medication allergy"]},

    {"id":"271737000","t":"Anemia","c":"D64.9","kw":["anemia","anaemia","low hemoglobin","low iron"]},
    {"id":"363346000","t":"Cancer","c":"C80.1","kw":["cancer","malignancy","carcinoma","tumor","tumour","neoplasm"]},
    {"id":"93143009","t":"Leukemia","c":"C95.90","kw":["leukemia","leukaemia"]},
    {"id":"118600007","t":"Lymphoma","c":"C85.90","kw":["lymphoma"]},

    {"id":"60862001","t":"Tinnitus","c":"H93.19","kw":["tinnitus","ringing in ears","ringing ears"]},
    {"id":"1023001","t":"Hearing loss","c":"H91.90","kw":["hearing loss","deaf","hard of hearing","can't hear"]},
    {"id":"162298006","t":"Ear pain","c":"H92.09","kw":["ear pain","earache","otalgia"]},
    {"id":"65363002","t":"Nosebleed","c":"R04.0","kw":["nosebleed","epistaxis","nose bleed"]},

    {"id":"246636008","t":"Blurred vision","c":"H53.8","kw":["blurry vision","blurred vision","blurry","vision problems","visual changes"]},
    {"id":"225560005","t":"Eye pain","c":"H57.10","kw":["eye pain","painful eye"]},

    {"id":"412244003","t":"Ginger preparation","c":None,"kw":["ginger","ginger things","ginger supplement","ginger chews","ginger candy"]},
    {"id":"372614000","t":"Lisinopril","c":None,"kw":["lisinopril"]},
    {"id":"386864001","t":"Amlodipine","c":None,"kw":["amlodipine","norvasc"]},
    {"id":"109081006","t":"Metformin","c":None,"kw":["metformin","glucophage"]},
    {"id":"372826007","t":"Metoprolol","c":None,"kw":["metoprolol","lopressor","toprol"]},
    {"id":"387475002","t":"Furosemide","c":None,"kw":["furosemide","lasix"]},
    {"id":"387207008","t":"Naproxen","c":None,"kw":["naproxen","aleve","naprosyn"]},
    {"id":"386845007","t":"Gabapentin","c":None,"kw":["gabapentin","neurontin"]},
    {"id":"412588001","t":"Rosuvastatin","c":None,"kw":["rosuvastatin","rosvastetin","crestor"]},
    {"id":"373462001","t":"Atorvastatin","c":None,"kw":["atorvastatin","lipitor"]},
    {"id":"387207008","t":"Ibuprofen","c":None,"kw":["ibuprofen","advil","motrin"]},
    {"id":"387517004","t":"Paracetamol","c":None,"kw":["acetaminophen","tylenol","paracetamol"]},
    {"id":"387458008","t":"Aspirin","c":None,"kw":["aspirin","asa"]},
    {"id":"764146007","t":"Penicillin","c":None,"kw":["penicillin"]},
    {"id":"387501005","t":"Amoxicillin","c":None,"kw":["amoxicillin","amoxil"]},
    {"id":"387322009","t":"Multivitamin","c":None,"kw":["multivitamin","vitamin","vitamins"]},
    {"id":"108774000","t":"Losartan","c":None,"kw":["losartan","cozaar"]},
    {"id":"387207008","t":"Albuterol","c":None,"kw":["albuterol","ventolin","salbutamol","inhaler","puffer","puffers","nebulizer"]},
    {"id":"126218002","t":"Omeprazole","c":None,"kw":["omeprazole","prilosec"]},
    {"id":"387168006","t":"Levothyroxine","c":None,"kw":["levothyroxine","synthroid"]},
    {"id":"387286002","t":"Prednisone","c":None,"kw":["prednisone","prednisolone","steroid","steroids"]},
    {"id":"372756006","t":"Warfarin","c":None,"kw":["warfarin","coumadin","blood thinner"]},
    {"id":"442031002","t":"Recreational methamphetamine","c":None,"kw":["crystal meth","methamphetamine","meth"]},
    {"id":"229880002","t":"Cannabis use","c":None,"kw":["marijuana","cannabis","weed","pot"]},
    {"id":"228388006","t":"Cocaine use","c":None,"kw":["cocaine","crack"]},

    {"id":"77176002","t":"Smoking","c":"F17.210","kw":["smoking","smoker","cigarette","cigarettes","tobacco","pack a day","half a pack"]},
    {"id":"228273003","t":"Alcohol use","c":"F10.10","kw":["alcohol","drinking","drinks","wine","beer","liquor"]},

    {"id":"399208008","t":"Chest X-ray","c":None,"kw":["chest x-ray","cxr","chest xray","x-ray"]},
    {"id":"241441007","t":"Ankle X-ray","c":None,"kw":["ankle x-ray","ankle xray","left ankle x-ray","right ankle x-ray"]},
    {"id":"113091000","t":"MRI","c":None,"kw":["mri","magnetic resonance","lumbar mri","cervical mri","brain mri"]},
    {"id":"77477000","t":"CT scan","c":None,"kw":["ct scan","ct","cat scan","computed tomography"]},
    {"id":"16310003","t":"Ultrasound","c":None,"kw":["ultrasound","ultrasonography","sonogram","echo","echocardiogram"]},
    {"id":"29303009","t":"ECG","c":None,"kw":["ecg","ekg","electrocardiogram"]},
    {"id":"26604007","t":"CBC","c":None,"kw":["complete blood count","cbc","blood count","blood work"]},
    {"id":"20109005","t":"BMP","c":None,"kw":["basic metabolic panel","bmp","metabolic panel"]},
    {"id":"390840006","t":"BNP measurement","c":None,"kw":["bnp","bnp level","natriuretic peptide"]},
    {"id":"167252002","t":"Pregnancy test","c":None,"kw":["pregnancy test","urine pregnancy","hcg","beta hcg"]},
    {"id":"27171005","t":"Urinalysis","c":None,"kw":["urinalysis","urine test","urine analysis","ua"]},
    {"id":"371907003","t":"Oxygen therapy","c":None,"kw":["supplemental oxygen","oxygen therapy","oxygen","nasal cannula"]},
    {"id":"183524004","t":"Referral","c":None,"kw":["referral","referring","consultation","cardiology referral","orthopedic referral","specialist","cardiology","neurology","orthopedic"]},
    {"id":"183519002","t":"ED referral","c":None,"kw":["emergency department","ed referral","emergency room","er"]},
    {"id":"80146002","t":"Appendectomy","c":None,"kw":["appendectomy"]},
    {"id":"236886002","t":"Hysterectomy","c":None,"kw":["hysterectomy"]},
    {"id":"118949005","t":"Wisdom teeth extraction","c":None,"kw":["wisdom teeth","wisdom tooth","wisdom teeth removal","wisdom teeth removed"]},
    {"id":"71388002","t":"Endoscopy","c":None,"kw":["endoscopy","upper endoscopy","egd"]},
    {"id":"73761001","t":"Colonoscopy","c":None,"kw":["colonoscopy"]},
    {"id":"387713003","t":"Stool test","c":None,"kw":["stool test","stool culture","stool sample"]},

    {"id":"75367002","t":"Blood pressure","c":"R03.0","kw":["blood pressure","bp","systolic","diastolic"]},
    {"id":"364075005","t":"Heart rate","c":None,"kw":["heart rate","pulse","bpm"]},
    {"id":"86290005","t":"Respiratory rate","c":None,"kw":["respiratory rate","breaths per minute"]},
    {"id":"386725007","t":"Body temperature","c":"R50.9","kw":["temperature","temp","38 degrees","39 degrees","100.4","101","102","103","104"]},
    {"id":"431314004","t":"Oxygen saturation","c":None,"kw":["oxygen saturation","spo2","o2 sat","91 percent","92 percent","88 percent"]},
    {"id":"12063002","t":"Pitting edema","c":"R60.0","kw":["pitting edema","2+ edema","3+ edema","pitting"]},
    {"id":"442551007","t":"JVD","c":None,"kw":["jvd","jugular venous distension","neck swelling","neck swollen","neck seems to be a little swollen"]},

    {"id":"160303001","t":"Family history of MI","c":"Z82.49","kw":["father had a heart attack","mother had a heart attack","family heart attack","family history of heart attack"]},
    {"id":"160357008","t":"Family history of cancer","c":"Z80.9","kw":["family cancer","cancer in family","family history of cancer"]},
    {"id":"160377001","t":"Family history of diabetes","c":"Z83.3","kw":["family diabetes","family history of diabetes"]},
    {"id":"160303001","t":"Family history of hypertension","c":"Z82.49","kw":["family high blood pressure","father has high blood pressure","mother has high blood pressure","dad has high blood pressure"]},
    {"id":"160364003","t":"Family history of stroke","c":"Z82.3","kw":["family stroke","family history of stroke"]},
]

_AMENORRHEA_CONTEXT = re.compile(
    r"\b(?:missed|late|six\s+weeks?\s+ago|last\s+.*?period|"
    r"weeks?\s+without|not\s+on\s+my\s+period|haven'?t\s+had)\b", re.I
)

def _contextual_period_snomed(entity_text, ctx):
    if not ctx:
        return None
    if _AMENORRHEA_CONTEXT.search(ctx.lower()):
        return {"concept_id":"14094001","term":"Amenorrhea","entity_text":entity_text,
                "method":"context_disambiguated","crosswalk_icd10":"N91.2",
                "crosswalk_icd10_desc":"Amenorrhea, unspecified"}
    return None

def get_snomed(entity_text, transcript_context=""):
    text_lower = entity_text.lower().strip()

    for entry in SNOMED_TABLE:
        matched = False
        for kw in entry["kw"]:
            if len(kw) < 4:
                if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
                    matched = True
                    break
            else:
                if kw in text_lower:
                    matched = True
                    break
        if matched:
            return {
                "concept_id": entry["id"], "term": entry["t"],
                "entity_text": entity_text, "method": "keyword",
                "crosswalk_icd10": entry.get("c"),
                "crosswalk_icd10_desc": entry["t"] if entry.get("c") else None,
            }

    if text_lower in ("period", "my period", "her period", "periods"):
        ctx = _contextual_period_snomed(entity_text, transcript_context)
        if ctx:
            return ctx

    if len(text_lower) < 4:
        return None

    q = embedder.encode(entity_text, convert_to_tensor=True)
    hits = util.semantic_search(q, icd10_embeddings, top_k=15)[0]

    candidates = []
    for hit in hits:
        if hit["score"] >= 0.85:
            idx = hit["corpus_id"]
            code = icd10_db.iloc[idx]["code"]
            desc = icd10_db.iloc[idx]["description"]
            candidates.append({
                "code": code, "desc": desc,
                "cosine": round(hit["score"], 4),
            })

    if not candidates:
        return None

    pairs = [(entity_text, c["desc"]) for c in candidates]
    rerank_scores = cross_encoder.predict(pairs)
    for i, s in enumerate(rerank_scores):
        candidates[i]["rerank"] = round(float(s), 4)

    candidates.sort(key=lambda x: x["rerank"], reverse=True)
    best = candidates[0]

    if best["rerank"] >= 0.5:
        return {
            "concept_id": best["code"], "term": best["desc"],
            "entity_text": entity_text,
            "method": f"icd10_reranked(cos={best['cosine']},rr={best['rerank']})",
            "crosswalk_icd10": best["code"],
            "crosswalk_icd10_desc": best["desc"],
        }

    return None

print(f"Cell 7 done  SNOMED: {len(SNOMED_TABLE)} keyword entries")


Cell 7 done  SNOMED: 194 keyword entries


In [12]:
# Cell 10 - ICD-10: MedCodER-inspired normalize-retrieve-rerank with SNOMED crosswalk priority, chapter-aware routing
_NORMALIZE_PROMPT = textwrap.dedent("""
You are a medical terminology normalization engine.

Given a list of medical entities extracted from a patient transcript,
convert each one to its STANDARD medical term.

RULES:
1. Convert colloquial patient language to formal medical terminology.
2. Keep the meaning identical — only change the wording.
3. If the entity is already a standard medical term, keep it as-is.
4. Return ONLY a JSON object mapping original → normalized.

EXAMPLES:
Input: ["racing of the heart", "belly pain", "threw up", "can't breathe"]
Output: {"racing of the heart": "palpitations", "belly pain": "abdominal pain", "threw up": "vomiting", "can't breathe": "dyspnea"}

Input: ["chest pain", "headaches", "nausea"]
Output: {"chest pain": "chest pain", "headaches": "headache", "nausea": "nausea"}

Input: ["ginger things", "birth control", "high blood pressure"]
Output: {"ginger things": "ginger supplement", "birth control": "oral contraceptive", "high blood pressure": "hypertension"}

Return ONLY the JSON object. No other text.
""").strip()

_FAST_NORMALIZE = {
    # GI
    "nauseated": "nausea", "nauseous": "nausea", "feeling sick": "nausea",
    "throw up": "vomiting", "throwing up": "vomiting", "threw up": "vomiting",
    "throwing up blood": "hematemesis", "vomiting blood": "hematemesis",
    "crampy": "abdominal cramps", "belly pain": "abdominal pain",
    "stomach pain": "abdominal pain", "tummy pain": "abdominal pain",
    "loose stools": "diarrhea", "watery stools": "diarrhea", "the runs": "diarrhea",
    "blood in stool": "rectal bleeding", "blood in stools": "rectal bleeding",
    "blood in your stools": "rectal bleeding", "bloody stool": "rectal bleeding",
    "bloating": "abdominal distension", "bloated": "abdominal distension",
    "heartburn": "acid reflux", "acid reflux": "gastroesophageal reflux disease",
    "indigestion": "dyspepsia",
    "trouble swallowing": "dysphagia", "difficulty swallowing": "dysphagia",
    "painful swallowing": "odynophagia",
    "no appetite": "anorexia", "not eating": "decreased appetite",
    "yellow skin": "jaundice", "yellow eyes": "jaundice",

    # Cardiac / respiratory
    "high blood pressure": "hypertension", "high bp": "hypertension",
    "blood pressure high": "hypertension",
    "low blood pressure": "hypotension",
    "shortness of breath": "dyspnea", "difficulty breathing": "dyspnea",
    "trouble breathing": "dyspnea", "short of breath": "dyspnea",
    "can't breathe": "dyspnea", "winded": "dyspnea", "gasping for air": "dyspnea",
    "breathless": "dyspnea",
    "chest tightness": "chest pressure", "tight feeling in my chest": "chest pressure",
    "pressure on my chest": "chest pressure", "tight chest": "chest pressure",
    "heart racing": "palpitations", "racing of the heart": "palpitations",
    "beating faster": "tachycardia", "heart pounding": "palpitations",
    "heart fluttering": "palpitations", "irregular heartbeat": "atrial fibrillation",
    "high cholesterol": "hyperlipidemia", "cholesterol problems": "hyperlipidemia",
    "coughing blood": "hemoptysis", "bringing up blood": "hemoptysis",
    "noisy breathing": "stridor", "wheeze": "wheezing",
    "can't breathe lying flat": "orthopnea", "pillows at night": "orthopnea",
    "prop up with pillows": "orthopnea",

    # Neuro / general
    "lightheaded": "dizziness", "dizzy": "dizziness", "room spinning": "vertigo",
    "passing out": "syncope", "fainted": "syncope", "fainting": "syncope",
    "numb": "numbness", "tingling": "paresthesia", "pins and needles": "paresthesia",
    "shooting pain": "neuralgia",
    "feeling down": "depressed mood", "feeling blue": "depressed mood",
    "can't sleep": "insomnia", "trouble sleeping": "insomnia",
    "warm at night": "night sweats", "sweating at night": "night sweats",

    # GU / GYN
    "burning urination": "dysuria", "pain when peeing": "dysuria",
    "blood in urine": "hematuria", "peeing a lot": "polyuria",
    "missed period": "amenorrhea", "late period": "amenorrhea",
    "heavy period": "menorrhagia", "irregular period": "metrorrhagia",

    # Endocrine / general
    "sugar": "diabetes mellitus", "high sugar": "diabetes mellitus",
    "thirsty": "polydipsia", "excessive thirst": "polydipsia",
    "tired": "fatigue", "exhausted": "fatigue", "no energy": "fatigue",

    # Derm / MSK
    "swelling": "edema", "swollen": "edema", "pitting edema": "edema",
    "muscle aches": "myalgia", "body aches": "myalgia",
    "stiff joints": "joint stiffness",
    "rash": "skin rash", "hives": "urticaria", "itchy skin": "pruritus",

    # ENT
    "runny nose": "rhinorrhea", "stuffy nose": "nasal congestion",
    "sore throat": "pharyngitis", "scratchy throat": "pharyngitis",
    "ringing in ears": "tinnitus",

    # Misc
    "feel sick": "malaise", "off-color": "malaise",
    "ginger things": "ginger supplement", "ginger chews": "ginger supplement",
    "birth control": "oral contraceptive",
    "flu-like symptoms": "influenza-like illness", "flu-like": "influenza-like illness",
    "crystal meth": "methamphetamine", "crystal": "methamphetamine",
    "pack a day": "tobacco use", "half a pack": "tobacco use",
    "sweaty": "diaphoresis",
    "neck swollen": "cervical lymphadenopathy",
    "neck seems to be a little swollen": "cervical swelling",
    "water pills": "diuretics", "fluid pills": "diuretics",
    "blood thinner": "anticoagulant", "blood thinners": "anticoagulant",
    "pain killer": "analgesic", "pain killers": "analgesic",
    "sleeping pills": "sedative", "sleep medicine": "sedative",
    "steroid shot": "corticosteroid injection",
    "inhaler": "bronchodilator inhaler",
}

def _normalize_entities_batch(entities):
    """
    MedCodER Step 1: Normalize all entities in ONE LLM call.
    Uses hardcoded dict first, LLM for everything else.
    Returns dict mapping original_text -> normalized_text.
    """
    normalized = {}
    needs_llm = []

    for ent in entities:
        if ent["status"] == "Negated":
            continue
        text = ent["text"].lower().strip()
        if text in _FAST_NORMALIZE:
            normalized[ent["text"]] = _FAST_NORMALIZE[text]
        else:
            needs_llm.append(ent["text"])

    if needs_llm:
        msg = [
            {"role": "system", "content": _NORMALIZE_PROMPT},
            {"role": "user", "content": f"Normalize these: {json.dumps(needs_llm)}\n\nJSON object:"}
        ]
        raw = _llm(msg, max_tokens=500)
        llm_result = _parse_json(raw, "dict")
        if isinstance(llm_result, dict):
            for orig, norm in llm_result.items():
                if isinstance(norm, str) and len(norm) >= 3:
                    normalized[orig] = norm.strip()

    return normalized

def _determine_likely_chapters(entity_text, entity_label):
    text_lower = entity_text.lower()
    chapters = set()
    rules = {
        "K,R": ["nausea", "vomit", "abdomin", "cramp", "stomach", "belly",
                "diarrhea", "constipat", "heartburn", "bowel", "stool", "gastri"],
        "N,O,R,Z": ["pregnan", "amenorrh", "period", "menstr", "morning sickness",
                     "gravid", "obstet", "contracept"],
        "N,R": ["urin", "bladder", "kidney", "micturi", "dysuria", "polydipsia"],
        "I,R": ["hypertens", "blood pressure", "heart", "chest pain", "palpitat",
                "cardiac", "coronary", "angina", "tachycard", "infarct"],
        "J,R": ["cough", "breath", "dyspnea", "wheez", "pulmon", "lung",
                "asthma", "copd", "pneumon", "bronch", "stridor"],
        "M,R": ["muscle", "joint", "back pain", "arthral", "myalgia",
                "sciatica", "sprain", "fracture", "stiff"],
        "E,R": ["diabet", "thyroid", "dehydrat", "cholesterol", "lipid",
                "polydipsia", "obesity", "hypoglyc"],
        "L,T": ["allerg", "nickel", "rash", "dermatit", "eczema", "hives",
                "urticaria", "pruritus"],
        "G,R": ["headache", "migraine", "dizz", "seizure", "numb", "tingling",
                "stroke", "tremor", "vertigo", "syncope"],
        "F": ["anxiety", "depress", "insomnia", "panic", "substance"],
    }
    for chapter_str, terms in rules.items():
        if any(t in text_lower for t in terms):
            chapters.update(chapter_str.split(","))
    if entity_label == "Symptom":
        chapters.add("R")
    if not chapters:
        chapters = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
    return list(chapters)

def get_icd10(entity_text, entity_label="Symptom", snomed_crosswalk=None,
              normalized_text=None):
    """
    MedCodER Steps 2-4: Retrieve and Re-rank.
    PATH A: SNOMED crosswalk (deterministic)
    PATH B: Universal Router retrieve-rerank (semantic)
    """
    if snomed_crosswalk and snomed_crosswalk.get("crosswalk_icd10"):
        code = snomed_crosswalk["crosswalk_icd10"]
        match = icd10_db[icd10_db["code"] == code]
        if not match.empty:
            return {
                "code": code, "description": match.iloc[0]["description"],
                "cosine_score": 1.0, "rerank_score": 10.0,
                "method": "SNOMED_crosswalk",
            }
        return {
            "code": code,
            "description": snomed_crosswalk.get("crosswalk_icd10_desc", ""),
            "cosine_score": 1.0, "rerank_score": 10.0,
            "method": "SNOMED_crosswalk",
        }

    search_term = normalized_text if normalized_text else entity_text
    return _retrieve_rerank(search_term, entity_label)

def _retrieve_rerank(search_text, entity_label):
    likely_chapters = _determine_likely_chapters(search_text, entity_label)

    q_emb = embedder.encode(search_text, convert_to_tensor=True)
    hits = util.semantic_search(q_emb, icd10_embeddings, top_k=25)[0]

    all_cands = []
    for h in hits:
        if h["score"] >= ICD10_COSINE_THRESHOLD:
            idx = h["corpus_id"]
            code = icd10_db.iloc[idx]["code"]
            desc = icd10_db.iloc[idx]["description"]
            chapter_bonus = 0.05 if code[0] in likely_chapters else 0.0
            all_cands.append({
                "code": code, "description": desc,
                "cosine_score": round(h["score"] + chapter_bonus, 4),
                "chapter": code[0],
            })

    if not all_cands:
        return {"code": None, "description": None, "cosine_score": 0.0,
                "rerank_score": 0.0, "method": "no_match"}

    seen = {}
    for c in all_cands:
        if c["code"] not in seen or c["cosine_score"] > seen[c["code"]]["cosine_score"]:
            seen[c["code"]] = c
    cands = list(seen.values())

    pairs = [(search_text, c["description"]) for c in cands]
    scores = cross_encoder.predict(pairs)
    for i, s in enumerate(scores):
        cands[i]["rerank_score"] = round(float(s), 4)

    def sort_key(x):
        return x["rerank_score"] + (0.5 if x["chapter"] in likely_chapters else 0.0)

    cands.sort(key=sort_key, reverse=True)
    best = cands[0]
    if best["rerank_score"] >= RERANK_THRESHOLD:
        best["method"] = "universal_router"
        return best
    return {"code": None, "description": None, "cosine_score": 0.0,
            "rerank_score": 0.0, "method": "below_threshold"}

def enrich_codes(entities, transcript_text=""):
    """
    MedCodER-inspired pipeline:
    1. Normalize all entities (LLM batch call)
    2. For each entity: SNOMED keyword lookup (on normalized text)
    3. If no SNOMED match: ICD-10 retrieve-rerank (on normalized text)
    """
    print("  [CODES] Normalizing entities (MedCodER Step 1)...")
    norm_map = _normalize_entities_batch(entities)
    for orig, norm in norm_map.items():
        if orig.lower() != norm.lower():
            print(f"         '{orig}' -> '{norm}'")

    result = []
    for ent in entities:
        e = ent.copy()

        if ent["status"] == "Negated" or ent.get("label") == "NULL":
            e["snomed"] = None
            e["icd10_cm"] = None
            e["normalized"] = None
            result.append(e)
            continue

        normalized = norm_map.get(ent["text"], ent["text"])
        e["normalized"] = normalized if normalized != ent["text"] else None

        context = ""
        offset = ent.get("first_offset")
        if offset and transcript_text:
            start = max(0, offset["start"] - 100)
            end = min(len(transcript_text), offset["end"] + 100)
            context = transcript_text[start:end]

        sn = get_snomed(normalized, transcript_context=context)
        if not sn:
            sn = get_snomed(ent["text"], transcript_context=context)
        e["snomed"] = sn

        if (ent["label"] in ("Disease", "Symptom") and
                ent["status"] in ("Confirmed", "Historical", "Family_History")):
            e["icd10_cm"] = get_icd10(
                ent["text"], entity_label=ent["label"],
                snomed_crosswalk=sn, normalized_text=normalized
            )
        else:
            e["icd10_cm"] = None

        result.append(e)
    return result

print("Cell 8 done  MedCodER-inspired Normalize->Retrieve->Rerank")


Cell 8 done  MedCodER-inspired Normalize->Retrieve->Rerank


In [13]:
# Cell 11 - CPT codes: deterministic keyword + SapBERT + cross-encoder reranked matching, no LLM
CPT_TABLE = [
    {"code": "99213", "description": "Office visit, established patient, low complexity",
     "keywords": ["office visit", "follow-up", "routine visit"]},
    {"code": "99214", "description": "Office visit, established patient, moderate complexity",
     "keywords": ["office visit moderate", "established patient moderate"]},
    {"code": "99215", "description": "Office visit, established patient, high complexity",
     "keywords": ["complex visit", "multiple problems"]},
    {"code": "99202", "description": "Office visit, new patient, low complexity",
     "keywords": ["new patient visit", "initial visit"]},
    {"code": "99281", "description": "Emergency department visit, low severity",
     "keywords": ["emergency department", "ED visit"]},

    {"code": "81025", "description": "Urine pregnancy test",
     "keywords": ["pregnancy test", "urine pregnancy", "HCG", "hcg test"]},
    {"code": "59400", "description": "Routine obstetric care",
     "keywords": ["obstetric care", "prenatal care", "pregnancy care"]},
    {"code": "76801", "description": "Ultrasound, pregnant uterus, first trimester",
     "keywords": ["obstetric ultrasound", "pregnancy ultrasound", "dating ultrasound"]},

    {"code": "81001", "description": "Urinalysis, automated with microscopy",
     "keywords": ["urinalysis", "urine test", "urine analysis", "UA"]},
    {"code": "80048", "description": "Basic metabolic panel (BMP)",
     "keywords": ["basic metabolic panel", "BMP", "metabolic panel"]},
    {"code": "85025", "description": "Complete blood count with differential (CBC)",
     "keywords": ["complete blood count", "CBC", "blood count"]},
    {"code": "80061", "description": "Lipid panel",
     "keywords": ["lipid panel", "cholesterol test"]},
    {"code": "84443", "description": "Thyroid stimulating hormone (TSH)",
     "keywords": ["TSH", "thyroid test", "thyroid function"]},
    {"code": "82947", "description": "Glucose, quantitative, blood",
     "keywords": ["blood glucose", "glucose test", "blood sugar"]},

    {"code": "71046", "description": "Chest X-ray, 2 views",
     "keywords": ["chest x-ray", "chest xray", "CXR"]},
    {"code": "74018", "description": "Abdominal X-ray, 1 view",
     "keywords": ["abdominal x-ray", "KUB", "abdominal xray"]},
    {"code": "73610", "description": "X-ray of ankle, 3 views",
     "keywords": ["ankle x-ray", "ankle xray", "left ankle x-ray", "right ankle x-ray"]},
    {"code": "72148", "description": "MRI of lumbar spine without contrast",
     "keywords": ["MRI lumbar", "MRI spine", "lumbar MRI", "lumbar spine"]},
    {"code": "72141", "description": "MRI of cervical spine without contrast",
     "keywords": ["MRI cervical", "cervical MRI", "cervical spine"]},

    {"code": "83880", "description": "B-type natriuretic peptide (BNP)",
     "keywords": ["BNP", "natriuretic peptide", "BNP level"]},
    {"code": "80053", "description": "Comprehensive metabolic panel (CMP)",
     "keywords": ["comprehensive metabolic panel", "CMP"]},

    {"code": "93000", "description": "Electrocardiogram (ECG), 12-lead",
     "keywords": ["ECG", "EKG", "electrocardiogram"]},

    {"code": "94760", "description": "Pulse oximetry, single reading",
     "keywords": ["oxygen saturation", "pulse oximetry", "SpO2"]},
    {"code": "94002", "description": "Ventilation assist and management",
     "keywords": ["supplemental oxygen", "oxygen therapy", "oxygen"]},

    {"code": "99242", "description": "Consultation, moderate complexity",
     "keywords": ["referral", "referring", "consultation", "follow-up",
                   "cardiology", "orthopedic", "spine specialist"]},
]

_cpt_descs = [c["description"] for c in CPT_TABLE]
_cpt_embs = embedder.encode(_cpt_descs, convert_to_tensor=True, batch_size=64)

def get_cpt_deterministic(procedure_text):
    """Match procedure entity to CPT code. Keyword-first, reranked semantic fallback."""
    text_lower = procedure_text.lower()

    for entry in CPT_TABLE:
        matched = False
        for kw in entry["keywords"]:
            if len(kw) < 4:
                if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
                    matched = True
                    break
            else:
                if kw in text_lower:
                    matched = True
                    break
        if matched:
            return {"code": entry["code"], "description": entry["description"],
                    "justification": f"Keyword match: '{procedure_text}'", "method": "keyword"}

    q = embedder.encode(procedure_text, convert_to_tensor=True)
    sims = util.cos_sim(q, _cpt_embs)[0]

    top_indices = torch.argsort(sims, descending=True)[:5]
    candidates = []
    for idx in top_indices:
        score = float(sims[idx])
        if score >= 0.50:
            candidates.append({
                "idx": int(idx), "cosine": round(score, 4),
                "code": CPT_TABLE[int(idx)]["code"],
                "desc": CPT_TABLE[int(idx)]["description"],
            })

    if not candidates:
        return None

    pairs = [(procedure_text, c["desc"]) for c in candidates]
    rerank_scores = cross_encoder.predict(pairs)
    for i, s in enumerate(rerank_scores):
        candidates[i]["rerank"] = round(float(s), 4)

    candidates.sort(key=lambda x: x["rerank"], reverse=True)
    best = candidates[0]

    if best["rerank"] >= 0.5:
        return {"code": best["code"], "description": best["desc"],
                "justification": f"Reranked semantic ({best['cosine']:.2f}/{best['rerank']:.2f}): '{procedure_text}'",
                "method": "semantic_reranked"}
    return None

print(f"Cell 9 done  CPT table loaded ({len(CPT_TABLE)} codes)")


Cell 9 done  CPT table loaded (25 codes)


In [14]:
# Cell 12 - Report builder: single comprehensive text report with all sections

def build_report(r):
    lines = [
        "=" * 70,
        "CLINICAL NLP PIPELINE — PRODUCTION REPORT",
        f"Generated: {datetime.now():%Y-%m-%d %H:%M:%S}",
        f"Config: {r.get('config', {}).get('k_shot', '?')}-shot {r.get('config', {}).get('strategy', '?')}",
        f"Pipeline Time: {r.get('time_s', 0)}s",
        "DISCLAIMER: AI-generated. Requires clinician review.",
        "=" * 70,
    ]

    lines += ["", "RAW TRANSCRIPT", "-" * 50, r.get("transcript", "")]

    soap = r.get("soap", {})
    fact_table = soap.get("fact_table", "")
    if fact_table:
        lines += ["", "STRUCTURED FACT TABLE (intermediate representation)", "-" * 50, fact_table]

    lines += ["", "SOAP NOTE", "-" * 50]
    for key in "KSOAP":
        val = soap.get(key, "")
        if val:
            lines.append(f"{key}: {val}")
            lines.append("")

    ai_sugg = soap.get("ai_suggestion", "")
    if ai_sugg:
        lines += ["AI SUGGESTION (clinical reasoning — NOT part of SOAP)", "-" * 50, ai_sugg]

    lines += ["", "NER ENTITIES", "-" * 50]
    for i, e in enumerate(r.get("entities", []), 1):
        fo = e.get("first_offset")
        off = f"[{fo['start']}:{fo['end']}]" if fo else "N/A"
        neg = " <- NEGATED" if e["status"] == "Negated" else ""
        fix = f" [{e['_neg_fix']}]" if e.get("_neg_fix") else ""
        inf = " [INFERRED]" if e.get("_inferred") else ""
        sn = e.get("snomed")
        sn_str = f" | SN:{sn['concept_id']}" if sn else ""
        ic = e.get("icd10_cm")
        ic_str = f" | ICD:{ic['code']}" if ic and ic.get("code") else ""
        norm = e.get("normalized")
        norm_str = f" [norm: {norm}]" if norm else ""
        lines.append(f"  {i:>2}. [{e['label']:<10}|{e['status']:<16}] \"{e['text']}\" {off}{neg}{fix}{inf}{norm_str}{sn_str}{ic_str}")

    lines += ["", "SNOMED CT MAPPINGS", "-" * 50]
    seen_sn = set()
    for e in r.get("entities", []):
        sn = e.get("snomed")
        if sn and sn["concept_id"] not in seen_sn:
            seen_sn.add(sn["concept_id"])
            xw = f" -> {sn['crosswalk_icd10']}" if sn.get("crosswalk_icd10") else ""
            lines.append(f"  {sn['concept_id']:>12}  {sn['term']:<40} <- \"{sn['entity_text']}\" ({sn['method']}){xw}")

    lines += ["", "ICD-10-CM CODES", "-" * 50]
    patient_codes, fh_codes = [], []
    for e in r.get("entities", []):
        ic = e.get("icd10_cm")
        if ic and ic.get("code"):
            entry = f"  {e['text']:<40} -> {ic['code']:>10}  ({ic.get('description','')})"
            if e.get("status") == "Family_History":
                fh_codes.append(entry)
            elif e.get("status") != "Negated":
                patient_codes.append(entry)
    if patient_codes:
        lines.append("  Patient Diagnoses/Symptoms:")
        lines.extend(patient_codes)
    if fh_codes:
        lines.append("  Family History (NOT patient diagnoses):")
        lines.extend(fh_codes)
    neg_ents = [e for e in r.get("entities", []) if e["status"] == "Negated"]
    if neg_ents:
        lines.append(f"  Negated ({len(neg_ents)} excluded from coding)")

    null_ents = [e for e in r.get("entities", []) if e.get("label") == "NULL"]
    if null_ents:
        lines += ["", "UNCLASSIFIED ENTITIES (NULL label)", "-" * 50]
        for e in null_ents:
            fo = e.get("first_offset")
            off = f"[{fo['start']}:{fo['end']}]" if fo else "N/A"
            lines.append(f"  {e['text']:<40} {e['status']:<16} {off}")

    lines += ["", "CPT CODES", "-" * 50]
    if r.get("cpt_codes"):
        for c in r["cpt_codes"]:
            lines.append(f"  {c['code']:>7}  {c['description']:<50}  {c['justification']}")
    else:
        lines.append("  (No procedures explicitly ordered in transcript)")

    em = r.get("eval_metrics", {})
    if em:
        lines += ["", "EVALUATION METRICS", "-" * 50]
        kc = em.get("keyword_coverage", {})
        lines.append(f"  Keyword Coverage: {kc.get('coverage',0):.1%} ({kc.get('found',0)}/{kc.get('total',0)})")
        if kc.get("missing"):
            lines.append(f"    Missing: {', '.join(kc['missing'][:8])}")
        h = em.get("hallucination", {})
        lines.append(f"  Hallucinations: {h.get('count', 0)}")
        for item in h.get("items", []):
            lines.append(f"    ! {item}")
        cons = em.get("consistency", {})
        if cons:
            lines.append(f"  Entity-SOAP Consistency: {cons.get('score', 0):.1%}")
            for issue in cons.get("issues", [])[:5]:
                lines.append(f"    ! {issue}")
        j = em.get("llm_judge", {})
        lines.append(f"  LLM Judge: {j.get('composite','?')}/5.0")
        lines.append(f"    Completeness={j.get('completeness','?')} Correctness={j.get('correctness','?')} Coherence={j.get('coherence','?')}")
        lines.append(f"    Assessment={j.get('assessment_quality','?')} Plan Safety={j.get('plan_safety','?')}")
        if em.get("rouge1"):
            lines.append(f"  ROUGE-1 F1: {em['rouge1']['f1']:.3f}")
        if em.get("bleu"):
            lines.append(f"  BLEU: {em['bleu']['bleu']:.3f}")

    lines += ["", "TIMING", "-" * 50, f"  Pipeline: {r.get('time_s', 0)}s"]
    lines.append("=" * 70)
    lines.append("END OF REPORT")
    return "\n".join(lines)


In [15]:
# Cell 13 - Few-shot example bank: 8 examples across 7 specialties, KMeans clustering with smallest-doc and query-aware selection
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
 
FEWSHOT_BANK = [
    {
        "id": "cardiac_01", "specialty": "Cardiology",
        "conversation": (
            "Doctor: What brings you in today? Patient: I've been having chest pain "
            "for the past three days. It feels like a pressure on my chest, especially "
            "when I walk or climb stairs. Doctor: Does it go away when you rest? "
            "Patient: Yes, usually within a few minutes. Doctor: Any shortness of "
            "breath? Patient: Yes, I get winded easily now. Doctor: Any history of "
            "heart problems? Patient: My father had a heart attack at 55. Doctor: Do "
            "you smoke? Patient: I quit two years ago but I smoked for 20 years. "
            "Doctor: Any medications? Patient: I take Lisinopril 10 milligrams daily "
            "and Aspirin 81 milligrams. Doctor: Let me check your vitals. Blood "
            "pressure is 148 over 92, heart rate 88. I'm going to order an EKG and "
            "troponin levels. I'd also like to get a chest X-ray."
        ),
        "ksoap": (
            'K: ("chest pain", Chest pain) [Symptom, Confirmed], '
            '("shortness of breath", Dyspnea) [Symptom, Confirmed], '
            '("heart attack", Myocardial infarction) [Disease, Family_History], '
            '("smoked for 20 years", Tobacco use) [Social, Historical], '
            '("Lisinopril 10 milligrams daily", Lisinopril) [Drug, Confirmed], '
            '("Aspirin 81 milligrams", Aspirin) [Drug, Confirmed], '
            '("148 over 92", Elevated blood pressure) [Vital, Confirmed], '
            '("heart rate 88", Heart rate) [Vital, Confirmed], '
            '("EKG", Electrocardiogram) [Procedure, Confirmed], '
            '("troponin levels", Troponin measurement) [Procedure, Confirmed], '
            '("chest X-ray", Chest radiograph) [Procedure, Confirmed]\n\n'
            "S: Patient presents with a three-day history of exertional chest "
            "pressure that resolves with rest. Reports associated dyspnea on "
            "exertion. Significant family history of premature coronary artery "
            "disease (father with MI at age 55). Former smoker with 20-pack-year "
            "history, quit two years ago. Currently taking Lisinopril 10mg daily "
            "and Aspirin 81mg daily.\n\n"
            "O: Blood pressure 148/92 mmHg (elevated). Heart rate 88 bpm.\n\n"
            "A: Exertional chest pain consistent with stable angina pectoris. "
            "Significant cardiac risk factors include hypertension, extensive "
            "smoking history, and family history of premature MI. Differential "
            "diagnoses include unstable angina and musculoskeletal chest wall pain.\n\n"
            "P: Doctor ordered EKG, serum troponin levels, and chest X-ray. "
            "[Disclaimer] AI Suggestion: Standard workup includes serial troponins, "
            "lipid panel, stress testing, and cardiology referral if troponins positive."
        ),
    },
    {
        "id": "gi_01", "specialty": "Gastroenterology",
        "conversation": (
            "Doctor: What's going on today? Patient: I've had really bad diarrhea "
            "for about five days now. Doctor: How many times a day? Patient: Like "
            "six or seven times. It's watery. Doctor: Any blood in the stool? "
            "Patient: No, no blood. Doctor: Any fever? Patient: Yeah, I had a "
            "fever of 101 two days ago. Doctor: Nausea or vomiting? Patient: Some "
            "nausea but no vomiting. Doctor: Any belly pain? Patient: Yeah, crampy "
            "pain around my belly button. Doctor: Any medications? Patient: No. "
            "Doctor: Allergies? Patient: None. Doctor: Abdomen is soft, diffusely "
            "tender, hyperactive bowel sounds. I'm going to send a stool culture "
            "and order a basic metabolic panel."
        ),
        "ksoap": (
            'K: ("diarrhea", Diarrhea) [Symptom, Confirmed], '
            '("blood in the stool", Rectal bleeding) [Symptom, Negated], '
            '("fever of 101", Fever) [Symptom, Confirmed], '
            '("nausea", Nausea) [Symptom, Confirmed], '
            '("no vomiting", Vomiting) [Symptom, Negated], '
            '("crampy pain around my belly button", Abdominal cramps) [Symptom, Confirmed], '
            '("diffusely tender", Abdominal tenderness) [Symptom, Confirmed], '
            '("hyperactive bowel sounds", Hyperactive bowel sounds) [Symptom, Confirmed], '
            '("stool culture", Stool culture) [Procedure, Confirmed], '
            '("basic metabolic panel", BMP) [Procedure, Confirmed]\n\n'
            "S: Patient reports five days of watery diarrhea six to seven times "
            "daily without hematochezia. Associated fever (101F two days ago), "
            "nausea without emesis, and periumbilical crampy abdominal pain. "
            "Denies medications and allergies.\n\n"
            "O: Abdomen soft, diffusely tender. Hyperactive bowel sounds.\n\n"
            "A: Acute infectious gastroenteritis with dehydration risk given "
            "frequency of diarrhea and fever. Differential includes viral "
            "gastroenteritis and bacterial food poisoning.\n\n"
            "P: Doctor ordered stool culture and basic metabolic panel. "
            "[Disclaimer] AI Suggestion: Oral rehydration therapy, BRAT diet, "
            "empiric antibiotics if culture positive."
        ),
    },
    {
        "id": "resp_01", "specialty": "Pulmonology",
        "conversation": (
            "Doctor: Tell me what's been happening. Patient: I've had this terrible "
            "cough for two weeks. It started dry but now I'm coughing up green mucus. "
            "Doctor: Any fever? Patient: Yes, about 100.8. Doctor: Shortness of "
            "breath? Patient: Yes, especially lying down at night. Doctor: Any "
            "wheezing? Patient: Yes. Doctor: Do you have asthma? Patient: I had "
            "mild asthma as a kid but haven't needed an inhaler in years. Doctor: "
            "Do you smoke? Patient: No, never. Doctor: I'm hearing crackles in your "
            "right lower lobe. Oxygen saturation is 94 percent. I'm going to order "
            "a chest X-ray and start you on antibiotics."
        ),
        "ksoap": (
            'K: ("cough", Cough) [Symptom, Confirmed], '
            '("green mucus", Productive cough) [Symptom, Confirmed], '
            '("100.8", Fever) [Symptom, Confirmed], '
            '("shortness of breath", Dyspnea) [Symptom, Confirmed], '
            '("wheezing", Wheezing) [Symptom, Confirmed], '
            '("asthma as a kid", Asthma) [Disease, Historical], '
            '("No, never", Tobacco use) [Social, Negated], '
            '("crackles in your right lower lobe", Pulmonary crackles) [Symptom, Confirmed], '
            '("94 percent", Oxygen saturation) [Vital, Confirmed], '
            '("chest X-ray", Chest radiograph) [Procedure, Confirmed], '
            '("antibiotics", Antibiotic therapy) [Drug, Confirmed]\n\n'
            "S: Two-week cough progressing from dry to productive with green sputum. "
            "Fever 100.8F, dyspnea worse supine, audible wheezing. Childhood asthma "
            "history, currently not on inhalers. Non-smoker.\n\n"
            "O: Crackles in right lower lobe on auscultation. Oxygen saturation "
            "94% on room air.\n\n"
            "A: Right lower lobe community-acquired pneumonia as evidenced by "
            "productive cough with purulent sputum, focal crackles, fever, and "
            "mild hypoxemia. Differential includes acute bronchitis and asthma "
            "exacerbation with superimposed infection.\n\n"
            "P: Doctor ordered chest X-ray and initiated antibiotic therapy. "
            "[Disclaimer] AI Suggestion: Amoxicillin-clavulanate or azithromycin. "
            "Consider bronchodilator given asthma history."
        ),
    },
    {
        "id": "obgyn_01", "specialty": "OB/GYN",
        "conversation": (
            "Doctor: What brings you in? Patient: I've been feeling really nauseous "
            "and I missed my period. Doctor: When was your last period? Patient: "
            "About seven weeks ago. Doctor: Are your periods usually regular? "
            "Patient: Yes, every 28 days usually. Doctor: Any vomiting? Patient: "
            "Yes, mostly in the mornings. Doctor: Breast tenderness? Patient: "
            "Actually yes. Doctor: Are you sexually active? Patient: Yes. Doctor: "
            "What birth control do you use? Patient: We use condoms but not every "
            "time. Doctor: Let me order a urine pregnancy test."
        ),
        "ksoap": (
            'K: ("nauseous", Nausea) [Symptom, Confirmed], '
            '("missed my period", Amenorrhea) [Symptom, Confirmed], '
            '("vomiting", Vomiting) [Symptom, Confirmed], '
            '("Breast tenderness", Breast tenderness) [Symptom, Confirmed], '
            '("condoms", Condom use) [Drug, Confirmed], '
            '("urine pregnancy test", Pregnancy test) [Procedure, Confirmed]\n\n'
            "S: Patient reports nausea with morning vomiting and missed menstrual "
            "period. LMP approximately seven weeks ago with normally regular 28-day "
            "cycles. Reports breast tenderness. Sexually active with inconsistent "
            "condom use.\n\n"
            "O: No physical exam findings or vitals recorded.\n\n"
            "A: Suspected early pregnancy based on amenorrhea, morning emesis, and "
            "breast tenderness in reproductive-age female with inconsistent "
            "contraception. Differential includes ectopic pregnancy.\n\n"
            "P: Doctor ordered urine pregnancy test. [Disclaimer] AI Suggestion: "
            "If positive, quantitative beta-hCG, prenatal labs, dating ultrasound, "
            "and prenatal vitamins with folic acid."
        ),
    },
    {
        "id": "msk_01", "specialty": "Orthopedics",
        "conversation": (
            "Doctor: What happened? Patient: I twisted my ankle playing basketball "
            "yesterday. It swelled up right away. Doctor: Can you put weight on it? "
            "Patient: I can walk but it hurts a lot. Doctor: Any numbness or "
            "tingling? Patient: No. Doctor: Have you injured this ankle before? "
            "Patient: No. Doctor: There's swelling and tenderness over the lateral "
            "malleolus. No deformity. Pedal pulses intact. I'm ordering an X-ray "
            "of your left ankle."
        ),
        "ksoap": (
            'K: ("twisted my ankle", Ankle sprain) [Symptom, Confirmed], '
            '("swelled up", Edema) [Symptom, Confirmed], '
            '("numbness", Numbness) [Symptom, Negated], '
            '("tingling", Paresthesia) [Symptom, Negated], '
            '("tenderness over the lateral malleolus", Lateral malleolus tenderness) [Symptom, Confirmed], '
            '("X-ray of your left ankle", Ankle radiograph) [Procedure, Confirmed]\n\n'
            "S: Acute left ankle injury from basketball the prior day with immediate "
            "swelling. Weight-bearing with significant pain. Denies numbness and "
            "tingling. No prior ankle injuries.\n\n"
            "O: Left ankle swelling and tenderness over lateral malleolus. No "
            "deformity. Pedal pulses intact.\n\n"
            "A: Acute left lateral ankle sprain, likely ATFL injury. Ottawa Ankle "
            "Rules positive given lateral malleolus tenderness. Differential "
            "includes lateral malleolus fracture.\n\n"
            "P: Doctor ordered left ankle X-ray. [Disclaimer] AI Suggestion: "
            "RICE protocol, NSAIDs, air-stirrup brace. Orthopedic referral if fracture."
        ),
    },
    {
        "id": "neuro_01", "specialty": "Neurology",
        "conversation": (
            "Doctor: What's troubling you? Patient: I've been getting terrible "
            "headaches for about a month. Doctor: Where is the pain? Patient: "
            "Right side, behind my eye. Doctor: How often? Patient: Three times a "
            "week. Doctor: Any nausea? Patient: Yes, and sometimes flashing lights "
            "before they start. Doctor: How long do they last? Patient: Four to six "
            "hours. Doctor: Family history of migraines? Patient: My mother gets "
            "them. Doctor: Medications? Patient: Ibuprofen but it doesn't help "
            "anymore. Doctor: I'm starting you on Sumatriptan for acute episodes."
        ),
        "ksoap": (
            'K: ("headaches", Headache) [Symptom, Confirmed], '
            '("nausea", Nausea) [Symptom, Confirmed], '
            '("flashing lights", Visual aura) [Symptom, Confirmed], '
            '("My mother gets them", Migraine) [Disease, Family_History], '
            '("Ibuprofen", Ibuprofen) [Drug, Confirmed], '
            '("Sumatriptan", Sumatriptan) [Drug, Confirmed]\n\n'
            "S: One-month history of recurrent right-sided retro-orbital headaches "
            "three times weekly, lasting four to six hours. Associated nausea and "
            "visual aura. Family history of migraines in mother. Ibuprofen no "
            "longer effective.\n\n"
            "O: No physical exam findings or vitals recorded.\n\n"
            "A: Migraine with aura meeting ICHD-3 criteria given recurrent "
            "unilateral headaches with visual aura, nausea, and positive family "
            "history. Differential includes tension-type and cluster headache.\n\n"
            "P: Doctor prescribed Sumatriptan for acute migraine episodes. "
            "[Disclaimer] AI Suggestion: Consider preventive therapy (topiramate, "
            "propranolol). Headache diary for trigger identification."
        ),
    },
    {
        "id": "infectious_01", "specialty": "Internal Medicine",
        "conversation": (
            "Doctor: What's going on? Patient: Sore throat and fever for three "
            "days. Doctor: How high? Patient: About 102. Doctor: Any cough? "
            "Patient: No. Doctor: Body aches? Patient: Yes. Doctor: Rash? Patient: "
            "No. Doctor: Your pharynx is erythematous with tonsillar exudates. "
            "Tender anterior cervical lymphadenopathy. I'm doing a rapid strep test."
        ),
        "ksoap": (
            'K: ("Sore throat", Pharyngitis) [Symptom, Confirmed], '
            '("fever", Fever) [Symptom, Confirmed], '
            '("cough", Cough) [Symptom, Negated], '
            '("Body aches", Myalgia) [Symptom, Confirmed], '
            '("Rash", Rash) [Symptom, Negated], '
            '("erythematous", Pharyngeal erythema) [Symptom, Confirmed], '
            '("tonsillar exudates", Tonsillar exudate) [Symptom, Confirmed], '
            '("cervical lymphadenopathy", Cervical lymphadenopathy) [Symptom, Confirmed], '
            '("rapid strep test", Rapid streptococcal antigen test) [Procedure, Confirmed]\n\n'
            "S: Three-day sore throat and fever to 102F. Diffuse body aches. "
            "Denies cough and rash.\n\n"
            "O: Erythematous pharynx with bilateral tonsillar exudates. Tender "
            "anterior cervical lymphadenopathy.\n\n"
            "A: Acute pharyngitis with high suspicion for Group A Streptococcal "
            "infection. Centor score 4/4 (fever, exudates, lymphadenopathy, no "
            "cough). Differential includes mononucleosis.\n\n"
            "P: Doctor ordered rapid strep test. [Disclaimer] AI Suggestion: "
            "If positive, penicillin V or amoxicillin 10 days. Monospot if negative."
        ),
    },
    {
        "id": "endo_01", "specialty": "Endocrinology",
        "conversation": (
            "Doctor: How are you feeling? Patient: Very thirsty lately and going "
            "to the bathroom all the time. Doctor: How often? Patient: Probably "
            "every hour. Doctor: Weight loss? Patient: About ten pounds in a month "
            "without trying. Doctor: Blurry vision? Patient: A little. Doctor: "
            "Family history of diabetes? Patient: My father has type 2. Doctor: "
            "Your fasting glucose is 285. I need to check your hemoglobin A1c. "
            "I'm starting you on Metformin 500 milligrams twice daily."
        ),
        "ksoap": (
            'K: ("Very thirsty", Polydipsia) [Symptom, Confirmed], '
            '("bathroom all the time", Polyuria) [Symptom, Confirmed], '
            '("ten pounds in a month", Unintentional weight loss) [Symptom, Confirmed], '
            '("Blurry vision", Blurred vision) [Symptom, Confirmed], '
            '("father has type 2", Diabetes mellitus type 2) [Disease, Family_History], '
            '("fasting glucose is 285", Hyperglycemia) [Vital, Confirmed], '
            '("hemoglobin A1c", HbA1c measurement) [Procedure, Confirmed], '
            '("Metformin 500 milligrams twice daily", Metformin) [Drug, Confirmed]\n\n'
            "S: Polydipsia, polyuria (hourly), unintentional 10-pound weight loss "
            "over one month, blurry vision. Family history of type 2 diabetes in "
            "father.\n\n"
            "O: Fasting blood glucose 285 mg/dL (severely elevated, normal <100).\n\n"
            "A: New-onset type 2 diabetes mellitus with classic presentation of "
            "polyuria, polydipsia, weight loss, and fasting glucose 285. Visual "
            "changes may indicate hyperglycemia-related lens changes.\n\n"
            "P: Doctor ordered hemoglobin A1c and started Metformin 500mg twice "
            "daily. [Disclaimer] AI Suggestion: CMP, lipid panel, urine "
            "microalbumin, ophthalmology referral, diabetes education."
        ),
    },
]
 
def build_fewshot_embeddings(bank, embed_model):
    texts = [ex["conversation"] for ex in bank]
    embs = embed_model.encode(texts, convert_to_tensor=False, show_progress_bar=False)
    return np.array(embs)
 
def select_fewshot_smallest(bank, embeddings, k=3):
    if k >= len(bank):
        return list(range(len(bank)))
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)
    indices = []
    for i in range(k):
        cluster_idx = np.where(labels == i)[0]
        if len(cluster_idx) == 0:
            continue
        lengths = np.array([len(bank[j]["conversation"]) for j in cluster_idx])
        best_local = int(np.argmin(lengths))
        indices.append(int(cluster_idx[best_local]))
    return indices
 
def select_fewshot_query_aware(bank, embeddings, query_text, embed_model, k=3):
    if k >= len(bank):
        return list(range(len(bank)))
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)
    query_emb = embed_model.encode([query_text], convert_to_tensor=False)
    indices = []
    for i in range(k):
        cluster_idx = np.where(labels == i)[0]
        if len(cluster_idx) == 0:
            continue
        cluster_embs = embeddings[cluster_idx]
        sims = sklearn_cosine(cluster_embs, query_emb).flatten()
        best_local = int(np.argmax(sims))
        indices.append(int(cluster_idx[best_local]))
    return indices
 
_fewshot_embs = build_fewshot_embeddings(FEWSHOT_BANK, embedder)
 
print(f"Cell A done  {len(FEWSHOT_BANK)} examples, embeddings shape {_fewshot_embs.shape}")
for i, ex in enumerate(FEWSHOT_BANK):
    print(f"  {i}: [{ex['specialty']:<18}] {len(ex['conversation']):>4} chars  {ex['id']}")


Cell A done  8 examples, embeddings shape (8, 768)
  0: [Cardiology        ]  753 chars  cardiac_01
  1: [Gastroenterology  ]  645 chars  gi_01
  2: [Pulmonology       ]  597 chars  resp_01
  3: [OB/GYN            ]  525 chars  obgyn_01
  4: [Orthopedics       ]  429 chars  msk_01
  5: [Neurology         ]  543 chars  neuro_01
  6: [Internal Medicine ]  336 chars  infectious_01
  7: [Endocrinology     ]  461 chars  endo_01


In [16]:
# Cell 14 - K-SOAP few-shot generation using fact table intermediate representation.
# 2026-05 upgrade: strict extractive body + separate AI Suggestion call (always populated).

def build_keyword_section(entities):
    if not entities:
        return "No clinical keywords extracted."
    parts = []
    for e in entities:
        span = e.get("text", "").strip()
        if not span:
            continue
        normalized = e.get("normalized")
        snomed = e.get("snomed")
        fast_norm = None
        try:
            fast_norm = _FAST_NORMALIZE.get(span.lower())
        except NameError:
            pass
        if normalized and normalized.lower() != span.lower():
            entity_name = normalized.title()
        elif snomed and snomed.get("term"):
            entity_name = snomed["term"]
        elif fast_norm:
            entity_name = fast_norm.title()
        else:
            entity_name = span.title() if len(span) < 40 else span[:40].title()
        label = e.get("label", "Symptom")
        status = e.get("status", "Confirmed")
        parts.append(f'("{span}", {entity_name}) [{label}, {status}]')
    return ", ".join(parts) if parts else "No clinical keywords extracted."


def _build_ksoap_body_prompt(examples, fact_table, keyword_section):
    system = (
        "You are a strict EXTRACTIVE Clinical Scribe writing K-SOAP notes.\n"
        "You PARAPHRASE the structured fact table into the K-SOAP format.\n"
        "You NEVER add diagnoses, speculation, or reasoning beyond what is in the facts.\n\n"
        "K-SOAP FORMAT (plain text, no markdown):\n"
        'K: Keywords in ("span", Entity) [Label, Assertion] format - PROVIDED below, copy VERBATIM.\n'
        "S: Subjective - paraphrase confirmed symptoms, history, social context. End with pertinent negatives.\n"
        "O: Objective - paraphrase ONLY vitals/exam from the fact table. If none, write exactly: No objective findings recorded in transcript.\n"
        "A: Assessment - NEUTRAL SUMMARY of confirmed clinical picture. Do NOT diagnose. Do NOT use 'suspected', 'likely', 'rule out', 'concerning for'.\n"
        "P: Plan - paraphrase ONLY procedures_ordered from the fact table. If none, write exactly: No specific medical orders were recorded in the transcript.\n\n"
        "ABSOLUTE RULES:\n"
        "1. Copy the K section verbatim from the keyword list below.\n"
        "2. No markdown. No **bold**. Plain text K:, S:, O:, A:, P: prefixes only.\n"
        "3. EVERY claim in S/O/A/P must come from the fact table.\n"
        "4. Do NOT include any [Disclaimer] or AI Suggestion - those are generated separately.\n"
        "5. STOP after the P: line.\n\n"
        "BAD : A: Likely acute coronary syndrome; consider stress test.\n"
        "GOOD: A: Patient reports exertional chest pressure, dyspnea on exertion, and family history of MI.\n"
    )

    user_parts = []
    for i, ex in enumerate(examples):
        user_parts.append(f"### Example {i+1}")
        user_parts.append(f"Conversation:\n{ex['conversation'].strip()}")
        # strip any [Disclaimer]/AI Suggestion from the example ksoap when shown
        ex_ksoap = ex['ksoap'].strip()
        cut = ex_ksoap.find("[Disclaimer]")
        if cut != -1:
            ex_ksoap = ex_ksoap[:cut].strip()
        user_parts.append(f"K-SOAP Note (body only):\n{ex_ksoap}")
        user_parts.append("")

    user_parts.append("### Now generate the K-SOAP note BODY ONLY for:")
    user_parts.append(f"Clinical Keywords (copy verbatim into K):\n{keyword_section}")
    user_parts.append(f"\nFact Table:\n{fact_table}")
    user_parts.append("\nK-SOAP Note (STOP after P: line. NO AI Suggestion here):\nK:")

    return system, "\n".join(user_parts)


def _parse_ksoap_sections(raw_output):
    cleaned = raw_output
    cleaned = re.sub(r'\*\*([A-Z]):\*\*', r'\1:', cleaned)
    cleaned = re.sub(r'\*\*([A-Z])\*\*\s*:', r'\1:', cleaned)
    cleaned = re.sub(r'\*\*', '', cleaned)
    cleaned = re.sub(r'#{1,4}\s*', '', cleaned)
    sections = {"K": "", "S": "", "O": "", "A": "", "P": ""}
    pattern = re.compile(r'(?:^|\n)\s*([KSOAP])\s*:\s*', re.MULTILINE)
    matches = list(pattern.finditer(cleaned))
    for i, m in enumerate(matches):
        key = m.group(1)
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(cleaned)
        content = cleaned[start:end].strip()
        if key in sections and not sections[key]:
            sections[key] = content
    return sections


def _score_ksoap_candidate(sections, facts):
    """Reuse the SOAP scorer on S/O/A/P portion."""
    body_text = " ".join(sections.get(k, "") for k in "SOAP").lower()
    hallucination_phrases = [
        "suspected", "likely", "concerning for", "rule out", "consistent with",
        "acute coronary syndrome", "myocardial infarction", "suggestive of",
        "differential diagnosis", "workup includes", "in keeping with",
    ]
    hallucination_count = sum(1 for p in hallucination_phrases if p in body_text)
    all_facts = []
    for k in ("confirmed_symptoms", "diseases", "medications_current", "vitals_and_exam", "procedures_ordered"):
        all_facts.extend(facts.get(k, []))
    if not all_facts:
        coverage = 1.0
    else:
        found = 0
        for fact in all_facts:
            key = re.sub(r"\[context:.*?\]", "", str(fact)).strip().lower()
            words = [w for w in key.split() if len(w) >= 4]
            if not words:
                continue
            if any(w in body_text for w in words[:3]):
                found += 1
        coverage = found / max(len(all_facts), 1)
    return hallucination_count - coverage, hallucination_count, coverage


run_soap_v1_baseline = run_soap


def run_ksoap_fewshot(source, entities, k_shot=3, strategy="query_aware"):
    print(f"  [K-SOAP] Generating {k_shot}-shot K-SOAP note (strategy={strategy})...")
    t0 = time.time()

    fact_table, facts = build_fact_table(entities, source)
    keyword_section = build_keyword_section(entities)
    print(f"  [K-SOAP] Fact table: {len(fact_table)} chars, K section: {len(entities)} entities")

    if strategy == "smallest":
        indices = select_fewshot_smallest(FEWSHOT_BANK, _fewshot_embs, k=k_shot)
    else:
        indices = select_fewshot_query_aware(
            FEWSHOT_BANK, _fewshot_embs, source[:500], embedder, k=k_shot
        )
    examples = [FEWSHOT_BANK[i] for i in indices]
    specs = [ex["specialty"] for ex in examples]
    print(f"  [K-SOAP] Selected {len(examples)} examples: {specs}")

    system_msg, user_msg = _build_ksoap_body_prompt(examples, fact_table, keyword_section)
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]

    if globals().get("USE_SELF_CONSISTENCY", False):
        n = globals().get("SELF_CONSISTENCY_N", 3)
        print(f"  [K-SOAP] Self-consistency: generating {n} candidates...")
        cands = []
        for k in range(n):
            raw = _llm(messages, max_tokens=1300, temperature=0.1, seed=100 + k * 13)
            secs = _parse_ksoap_sections(raw)
            if not secs["K"]:
                secs["K"] = keyword_section
            if not facts.get("vitals_and_exam"):
                secs["O"] = "No objective findings recorded in transcript."
            if not facts.get("procedures_ordered"):
                secs["P"] = "No specific medical orders were recorded in the transcript."
            if not secs.get("S"):
                s_parts = []
                if facts.get("confirmed_symptoms"):
                    s_parts.append("Patient reports " + ", ".join(re.sub(r"\s*\[context:.*?\]", "", x).strip() for x in facts["confirmed_symptoms"]) + ".")
                if facts.get("historical"):
                    s_parts.append("History of " + ", ".join(facts["historical"]) + ".")
                if facts.get("family_history"):
                    s_parts.append("Family history of " + ", ".join(facts["family_history"]) + ".")
                if facts.get("social_history"):
                    s_parts.append("Social history: " + "; ".join(facts["social_history"]) + ".")
                if facts.get("medications_current"):
                    s_parts.append("Current medications: " + ", ".join(facts["medications_current"]) + ".")
                if facts.get("negated_symptoms"):
                    s_parts.append("Denies " + ", ".join(facts["negated_symptoms"]) + ".")
                secs["S"] = " ".join(s_parts) if s_parts else "No subjective findings recorded in transcript."
            if not secs.get("A"):
                a_parts = []
                if facts.get("confirmed_symptoms"):
                    a_parts.append("Patient presents with " + ", ".join(re.sub(r"\s*\[context:.*?\]", "", x).strip() for x in facts["confirmed_symptoms"]) + ".")
                if facts.get("diseases"):
                    a_parts.append("Reported diagnoses: " + ", ".join(facts["diseases"]) + ".")
                if facts.get("family_history"):
                    a_parts.append("Notable family history: " + ", ".join(facts["family_history"]) + ".")
                if facts.get("medications_current"):
                    a_parts.append("Current medications: " + ", ".join(facts["medications_current"]) + ".")
                if not a_parts:
                    a_parts.append("No assessment-relevant findings recorded in transcript.")
                secs["A"] = " ".join(a_parts)
            if not secs.get("P") and facts.get("procedures_ordered"):
                secs["P"] = "Doctor ordered " + ", ".join(facts["procedures_ordered"]) + "."
            score, halluc, cov = _score_ksoap_candidate(secs, facts)
            cands.append((score, halluc, cov, secs, raw))
            print(f"    cand {k}: score={score:.2f} halluc={halluc} coverage={cov:.2f}")
        cands.sort(key=lambda x: x[0])
        _, halluc, cov, sections, raw = cands[0]
        print(f"  [K-SOAP] Best candidate selected (halluc={halluc} cov={cov:.2f})")
    else:
        raw = _llm(messages, max_tokens=1300, temperature=0.1)
        sections = _parse_ksoap_sections(raw)
        if not sections["K"]:
            sections["K"] = keyword_section
        if not facts.get("vitals_and_exam"):
            sections["O"] = "No objective findings recorded in transcript."
        if not facts.get("procedures_ordered"):
            sections["P"] = "No specific medical orders were recorded in the transcript."
        if not sections.get("S"):
            s_parts = []
            if facts.get("confirmed_symptoms"):
                s_parts.append("Patient reports " + ", ".join(re.sub(r"\s*\[context:.*?\]", "", x).strip() for x in facts["confirmed_symptoms"]) + ".")
            if facts.get("historical"):
                s_parts.append("History of " + ", ".join(facts["historical"]) + ".")
            if facts.get("family_history"):
                s_parts.append("Family history of " + ", ".join(facts["family_history"]) + ".")
            if facts.get("social_history"):
                s_parts.append("Social history: " + "; ".join(facts["social_history"]) + ".")
            if facts.get("medications_current"):
                s_parts.append("Current medications: " + ", ".join(facts["medications_current"]) + ".")
            if facts.get("negated_symptoms"):
                s_parts.append("Denies " + ", ".join(facts["negated_symptoms"]) + ".")
            sections["S"] = " ".join(s_parts) if s_parts else "No subjective findings recorded in transcript."
            if not sections.get("A"):
                a_parts = []
                if facts.get("confirmed_symptoms"):
                    a_parts.append("Patient presents with " + ", ".join(re.sub(r"\s*\[context:.*?\]", "", x).strip() for x in facts["confirmed_symptoms"]) + ".")
                if facts.get("diseases"):
                    a_parts.append("Reported diagnoses: " + ", ".join(facts["diseases"]) + ".")
                if facts.get("family_history"):
                    a_parts.append("Notable family history: " + ", ".join(facts["family_history"]) + ".")
                if facts.get("medications_current"):
                    a_parts.append("Current medications: " + ", ".join(facts["medications_current"]) + ".")
                if not a_parts:
                    a_parts.append("No assessment-relevant findings recorded in transcript.")
                sections["A"] = " ".join(a_parts)
            if not sections.get("P") and facts.get("procedures_ordered"):
                sections["P"] = "Doctor ordered " + ", ".join(facts["procedures_ordered"]) + "."

    # Strip speculation words from every section (extractive enforcement)
    for _k in "SOAP":
        if sections.get(_k):
            sections[_k] = sanitize_assessment(sections[_k])

    body_text = "\n\n".join(f"{k}: {sections[k]}" for k in "KSOAP" if sections[k])

    print("  [AI Suggestion] Generating dedicated suggestion section...")
    ai_suggestion_body = _generate_ai_suggestion(body_text)
    ai_suggestion = f"[Disclaimer]\nAI Suggestion (clinical decision support; NOT part of the K-SOAP note above):\n{ai_suggestion_body}"

    sections["AI Suggestion"] = ai_suggestion_body
    sections["fact_table"] = fact_table

    ksoap_raw_parts = [f"{k}: {sections[k]}" for k in "KSOAP" if sections[k]]
    ksoap_raw_parts.append("")
    ksoap_raw_parts.append(ai_suggestion)
    ksoap_raw = "\n\n".join(ksoap_raw_parts).strip()

    elapsed = time.time() - t0
    print(f"  [K-SOAP] Done in {elapsed:.1f}s")
    return sections, ksoap_raw


print("Cell 14 done: K-SOAP with strict extractive body + separate AI Suggestion call")


Cell 14 done: K-SOAP with strict extractive body + separate AI Suggestion call


In [17]:
# Cell 15 - Evaluation metrics: ROUGE-1/2/L, BLEU, BERTScore via SapBERT, keyword coverage, multi-layer hallucination detection, LLM-as-Judge
from collections import Counter
 
def _tokenize(text):
    """Simple whitespace + punctuation tokenizer for ROUGE."""
    return re.findall(r'\b\w+\b', text.lower())
 
def rouge_n(candidate, reference, n=1):
    """ROUGE-N: precision, recall, F1 of n-gram overlap."""
    cand_tok = _tokenize(candidate)
    ref_tok = _tokenize(reference)
    if not cand_tok or not ref_tok:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
 
    cand_ng = Counter(tuple(cand_tok[i:i+n]) for i in range(len(cand_tok)-n+1))
    ref_ng = Counter(tuple(ref_tok[i:i+n]) for i in range(len(ref_tok)-n+1))
 
    overlap = sum((cand_ng & ref_ng).values())
    p = overlap / max(sum(cand_ng.values()), 1)
    r = overlap / max(sum(ref_ng.values()), 1)
    f1 = 2 * p * r / max(p + r, 1e-8)
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}
 
def rouge_l(candidate, reference):
    """ROUGE-L: longest common subsequence."""
    cand_tok = _tokenize(candidate)
    ref_tok = _tokenize(reference)
    if not cand_tok or not ref_tok:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
 
    m, n = len(cand_tok), len(ref_tok)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if cand_tok[i-1] == ref_tok[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    p = lcs / max(m, 1)
    r = lcs / max(n, 1)
    f1 = 2 * p * r / max(p + r, 1e-8)
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}
 
def bleu_score(candidate, reference, max_n=4):
    """
    BLEU score (Bilingual Evaluation Understudy).
    Computes modified n-gram precision with brevity penalty.
 
    BLEU = BP * exp(sum(w_n * log(p_n)))
    where BP = min(1, exp(1 - ref_len/cand_len))
 
    Reference: Papineni et al. 2002
    Used in: Biswas & Talukdar 2024 for clinical note evaluation
    """
    cand_tok = _tokenize(candidate)
    ref_tok = _tokenize(reference)
    if not cand_tok or not ref_tok:
        return {"bleu": 0.0, "bleu1": 0.0, "bleu2": 0.0, "bleu4": 0.0, "brevity_penalty": 0.0}
 
    import math
 
    c_len = len(cand_tok)
    r_len = len(ref_tok)
    if c_len == 0:
        bp = 0.0
    elif c_len >= r_len:
        bp = 1.0
    else:
        bp = math.exp(1 - r_len / c_len)
 
    precisions = []
    individual = {}
    for n in range(1, max_n + 1):
        cand_ng = Counter(tuple(cand_tok[i:i+n]) for i in range(max(len(cand_tok)-n+1, 0)))
        ref_ng = Counter(tuple(ref_tok[i:i+n]) for i in range(max(len(ref_tok)-n+1, 0)))
 
        clipped = sum(min(cand_ng[ng], ref_ng[ng]) for ng in cand_ng)
        total = max(sum(cand_ng.values()), 1)
        p_n = clipped / total
 
        precisions.append(p_n)
        individual[f"bleu{n}"] = round(p_n, 4)
 
    log_avg = 0.0
    weight = 1.0 / max_n
    for p_n in precisions:
        if p_n == 0:
            log_avg = float('-inf')
            break
        log_avg += weight * math.log(p_n)
 
    if log_avg == float('-inf'):
        bleu = 0.0
    else:
        bleu = bp * math.exp(log_avg)
 
    return {
        "bleu": round(bleu, 4),
        "bleu1": individual.get("bleu1", 0.0),
        "bleu2": individual.get("bleu2", 0.0),
        "bleu4": individual.get("bleu4", 0.0),
        "brevity_penalty": round(bp, 4),
    }
 
def bertscore_document(candidate, reference):
    """Document-level cosine similarity using SapBERT embeddings."""
    if not candidate.strip() or not reference.strip():
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    c_emb = embedder.encode(candidate, convert_to_tensor=True)
    r_emb = embedder.encode(reference, convert_to_tensor=True)
    sim = float(util.cos_sim(c_emb, r_emb)[0][0])
    return {"precision": round(sim, 4), "recall": round(sim, 4), "f1": round(sim, 4)}
 
def bertscore_sentence(candidate, reference):
    """
    Sentence-level BERTScore: encode each sentence, compute greedy matching.
    This catches paraphrases that ROUGE misses.
    """
    c_sents = [s.strip() for s in re.split(r'[.!?]+', candidate) if len(s.strip()) > 10]
    r_sents = [s.strip() for s in re.split(r'[.!?]+', reference) if len(s.strip()) > 10]
    if not c_sents or not r_sents:
        return bertscore_document(candidate, reference)
 
    c_embs = embedder.encode(c_sents, convert_to_tensor=True)
    r_embs = embedder.encode(r_sents, convert_to_tensor=True)
    sim_matrix = util.cos_sim(c_embs, r_embs)
 
    p = float(sim_matrix.max(dim=1).values.mean())
    r = float(sim_matrix.max(dim=0).values.mean())
    f1 = 2 * p * r / max(p + r, 1e-8)
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}
 
def keyword_coverage(soap_text, entities):
    """What fraction of confirmed/historical NER entities appear in the SOAP note."""
    if not entities:
        return {"coverage": 0.0, "found": 0, "total": 0, "missing": []}
 
    soap_lower = soap_text.lower()
    relevant = [e for e in entities
                if e.get("status") in ("Confirmed", "Historical", "Family_History")]
    found = 0
    missing = []
    for e in relevant:
        text = e["text"].lower()
        words = [w for w in text.split() if len(w) >= 4]
        if text in soap_lower or any(w in soap_lower for w in words):
            found += 1
        else:
            missing.append(e["text"])
 
    total = len(relevant)
    return {
        "coverage": round(found / max(total, 1), 4),
        "found": found, "total": total, "missing": missing
    }
 
_KNOWN_DRUGS = re.compile(
    r'\b(?:metformin|lisinopril|amlodipine|aspirin|ibuprofen|acetaminophen|'
    r'amoxicillin|azithromycin|ciprofloxacin|prednisone|omeprazole|sumatriptan|'
    r'gabapentin|metoprolol|furosemide|atorvastatin|rosuvastatin|warfarin|'
    r'levothyroxine|losartan|albuterol|naproxen|hydrochlorothiazide|'
    r'clopidogrel|pantoprazole|sertraline|fluoxetine|diazepam|lorazepam)\b',
    re.I
)
 
def detect_hallucinations(soap_text, transcript, entities):
    """
    Multi-layer hallucination detection:
    Layer 1: Drug names in SOAP not in transcript
    Layer 2: Fabricated clinical claims (radiating pain, invented orders)
    Layer 3: Negated symptoms presented as confirmed in SOAP
    """
    hallucinated = []
    soap_lower = soap_text.lower()
    trans_lower = transcript.lower()

    soap_drugs = set(m.group().lower() for m in _KNOWN_DRUGS.finditer(soap_text))
    trans_drugs = set(m.group().lower() for m in _KNOWN_DRUGS.finditer(transcript))
    ent_drugs = set()
    for e in entities:
        if e.get("label") == "Drug":
            ent_drugs.update(
                m.group().lower() for m in _KNOWN_DRUGS.finditer(e["text"])
            )
    for drug in soap_drugs:
        if drug not in trans_drugs and drug not in ent_drugs:
            hallucinated.append(f"Drug not in transcript: {drug}")

    fabrication_checks = [
        (r'radiat\w+\s+to\s+(?:left\s+)?(?:arm|shoulder|jaw|back|neck)',
         "Radiating pain"),
        (r'doctor\s+ordered\s+(?:further\s+)?(?:evaluation|ecg|ekg|troponin|imaging|blood|labs)',
         "Doctor ordered tests"),
        (r'doctor\s+prescribed',
         "Doctor prescribed medication"),
    ]
    for pattern, label in fabrication_checks:
        if re.search(pattern, soap_lower):
            if not re.search(pattern, trans_lower):
                if "radiat" in pattern:
                    if re.search(r'radiat\w+\s+anywhere.*?\bno\b', trans_lower, re.DOTALL):
                        hallucinated.append(f"Fabricated: {label} (patient explicitly denied)")
                    elif "radiating anywhere" in trans_lower:
                        hallucinated.append(f"Fabricated: {label} (patient denied radiation)")
                elif "order" in pattern or "prescri" in pattern:
                    if not re.search(r"i'?m\s+(?:going\s+to\s+)?order", trans_lower):
                        if not re.search(r"i'?m\s+(?:going\s+to\s+)?start", trans_lower):
                            if not re.search(r"i'?m\s+prescribing", trans_lower):
                                hallucinated.append(f"Fabricated: {label} (not spoken by doctor)")

    return {
        "count": len(hallucinated),
        "items": hallucinated,
    }


_JUDGE_PROMPT = textwrap.dedent("""
You are a clinical documentation quality evaluator.
Score the SOAP note against the transcript on these 5 dimensions (1-5 scale).

TRANSCRIPT:
{transcript}

SOAP NOTE:
{soap}

SCORING (1=poor, 5=excellent):
1. COMPLETENESS: All clinically relevant info captured?
2. CORRECTNESS: Everything grounded in transcript? No invented findings?
3. COHERENCE: Well-organized with standard medical terminology?
4. ASSESSMENT_QUALITY: Specific primary diagnosis with reasonable differentials?
5. PLAN_SAFETY: Plan reflects ONLY explicit doctor orders?

Return ONLY a JSON object:
{{"completeness": N, "correctness": N, "coherence": N, "assessment_quality": N, "plan_safety": N}}
""").strip()

def llm_judge(transcript, soap_text):
    """Reference-free evaluation using the LLM itself as judge."""
    prompt = _JUDGE_PROMPT.format(
        transcript=transcript[:3000],
        soap=soap_text[:2000]
    )
    msg = [
        {"role": "system", "content": "Respond ONLY with a JSON object. No other text."},
        {"role": "user", "content": prompt}
    ]
    raw = _llm(msg, max_tokens=200)
    scores = _parse_json(raw, expect="dict")
 
    dims = ["completeness", "correctness", "coherence", "assessment_quality", "plan_safety"]
    for k in dims:
        v = scores.get(k)
        if not isinstance(v, (int, float)) or v < 1 or v > 5:
            scores[k] = 3
        else:
            scores[k] = max(1, min(5, int(v)))
 
    scores["composite"] = round(sum(scores[k] for k in dims) / len(dims), 2)
    return scores
 


def check_entity_soap_consistency(soap_text, entities):
    """
    Check if SOAP note is consistent with extracted entities.
    Detects: confirmed entities missing from SOAP, negated entities appearing as confirmed.
    """
    soap_lower = soap_text.lower()
    issues = []
    confirmed_found = 0
    confirmed_total = 0

    s_section = ""
    s_match = re.search(r'S:\s*(.+?)(?=\n[OAP]:|$)', soap_text, re.DOTALL)
    if s_match:
        s_section = s_match.group(1).lower()

    a_section = ""
    a_match = re.search(r'A:\s*(.+?)(?=\n[P]:|$)', soap_text, re.DOTALL)
    if a_match:
        a_section = a_match.group(1).lower()

    for e in entities:
        text_lower = e["text"].lower()
        words = [w for w in text_lower.split() if len(w) >= 4]
        in_soap = text_lower in soap_lower or any(w in soap_lower for w in words)
        norm = e.get("normalized", "")
        if norm:
            in_soap = in_soap or norm.lower() in soap_lower

        if e["status"] == "Confirmed" and e["label"] in ("Symptom", "Disease"):
            confirmed_total += 1
            if in_soap:
                confirmed_found += 1
            else:
                issues.append(f"Confirmed {e['label']} '{e['text']}' not found in SOAP")

        if e["status"] == "Negated" and e["label"] == "Symptom":
            if words:
                for w in words:
                    if w in s_section and f"deni" not in s_section[max(0,s_section.find(w)-30):s_section.find(w)]:
                        if w not in {"pain", "chest", "breath", "blood"}:
                            issues.append(f"Negated symptom '{e['text']}' may appear as confirmed in S section")
                            break

    score = confirmed_found / max(confirmed_total, 1)
    return {"score": round(score, 4), "found": confirmed_found, "total": confirmed_total, "issues": issues}

def evaluate_soap(soap_text, transcript, entities, reference_soap=None):
    """
    Run all evaluation metrics on a SOAP note.
 
    Args:
        soap_text: generated SOAP note text
        transcript: source transcript
        entities: NER entities list
        reference_soap: gold-standard SOAP note (optional)
 
    Returns:
        dict with all metric results
    """
    print("  [EVAL] Running evaluation metrics...")
    t0 = time.time()
    results = {}
 
    results["keyword_coverage"] = keyword_coverage(soap_text, entities)
    results["hallucination"] = detect_hallucinations(soap_text, transcript, entities)
    print(f"         Keyword coverage: {results['keyword_coverage']['coverage']:.1%}")
    print(f"         Hallucinations: {results['hallucination']['count']}")
 
    if reference_soap:
        results["rouge1"] = rouge_n(soap_text, reference_soap, n=1)
        results["rouge2"] = rouge_n(soap_text, reference_soap, n=2)
        results["rougeL"] = rouge_l(soap_text, reference_soap)
        results["bertscore_doc"] = bertscore_document(soap_text, reference_soap)
        results["bertscore_sent"] = bertscore_sentence(soap_text, reference_soap)
        results["bleu"] = bleu_score(soap_text, reference_soap)
 
        r1r = results["rouge1"]["recall"]
        bsr = results["bertscore_sent"]["recall"]
        results["mist_composite"] = round((r1r + bsr) / 2, 4)
 
        print(f"         ROUGE-1 F1: {results['rouge1']['f1']:.3f}")
        print(f"         BLEU: {results['bleu']['bleu']:.3f} (B1={results['bleu']['bleu1']:.3f} B4={results['bleu']['bleu4']:.3f})")
        print(f"         BERTScore(sent) F1: {results['bertscore_sent']['f1']:.3f}")
        print(f"         MIST Composite: {results['mist_composite']:.3f}")
 
    print("         Running LLM-as-Judge...")
    results["llm_judge"] = llm_judge(transcript, soap_text)
    print(f"         LLM Judge composite: {results['llm_judge']['composite']}/5.0")
 
    results["consistency"] = check_entity_soap_consistency(soap_text, entities)
    print(f"         Entity-SOAP consistency: {results['consistency']['score']:.1%}")

    results["eval_time_s"] = round(time.time() - t0, 2)
    print(f"  [EVAL] Done in {results['eval_time_s']}s")
    return results
 


# --- 2026-05 quality upgrade: per-section eval for BeTraC mode ---

def _split_soap_sections(text):
    """Split a SOAP-formatted text into {S, O, A, P} dicts.
    Tolerates uppercase/lowercase headers, K section prefixes, markdown bold."""
    if not text:
        return {"S": "", "O": "", "A": "", "P": ""}
    cleaned = re.sub(r'\*\*', '', text)
    cleaned = re.sub(r'#{1,4}\s*', '', cleaned)
    cleaned = re.sub(r'\b(\d\.\s+)?(Subjective)\b', 'S:', cleaned, flags=re.I)
    cleaned = re.sub(r'\b(\d\.\s+)?(Objective)\b', 'O:', cleaned, flags=re.I)
    cleaned = re.sub(r'\b(\d\.\s+)?(Assessment)\b', 'A:', cleaned, flags=re.I)
    cleaned = re.sub(r'\b(\d\.\s+)?(Plan)\b', 'P:', cleaned, flags=re.I)
    sections = {"S": "", "O": "", "A": "", "P": ""}
    pattern = re.compile(r'(?:^|\n)\s*([SOAP])\s*:\s*', re.MULTILINE)
    matches = list(pattern.finditer(cleaned))
    for i, m in enumerate(matches):
        key = m.group(1)
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(cleaned)
        content = cleaned[start:end]
        cut = content.find("[Disclaimer]")
        if cut != -1:
            content = content[:cut]
        if key in sections and not sections[key]:
            sections[key] = content.strip()
    return sections


def evaluate_soap_per_section(candidate_text, reference_text):
    """
    Compute ROUGE-1/2/L, BLEU, BERTScore per S/O/A/P section + macro-average.
    Use this when reference_soap is in plain SOAP format (e.g., BeTraC).
    """
    cand_secs = _split_soap_sections(candidate_text)
    ref_secs  = _split_soap_sections(reference_text)

    per_section = {}
    rouge_l_values, rouge_1_values, bleu_values, bert_values = [], [], [], []
    for key in "SOAP":
        c = cand_secs[key]
        r = ref_secs[key]
        if not c.strip() or not r.strip():
            per_section[key] = {
                "rouge1_f1": 0.0, "rouge2_f1": 0.0, "rougeL_f1": 0.0,
                "bleu": 0.0, "bertscore_f1": 0.0,
            }
            continue
        r1 = rouge_n(c, r, n=1)
        r2 = rouge_n(c, r, n=2)
        rl = rouge_l(c, r)
        bl = bleu_score(c, r)
        bs = bertscore_sentence(c, r)
        per_section[key] = {
            "rouge1_f1": r1["f1"], "rouge2_f1": r2["f1"], "rougeL_f1": rl["f1"],
            "bleu": bl["bleu"], "bertscore_f1": bs["f1"],
        }
        rouge_1_values.append(r1["f1"])
        rouge_l_values.append(rl["f1"])
        bleu_values.append(bl["bleu"])
        bert_values.append(bs["f1"])

    def _avg(xs):
        return round(sum(xs) / len(xs), 4) if xs else 0.0

    macro = {
        "rouge1_f1": _avg(rouge_1_values),
        "rougeL_f1": _avg(rouge_l_values),
        "bleu":      _avg(bleu_values),
        "bertscore_f1": _avg(bert_values),
        "sections_compared": len(rouge_l_values),
    }
    return {"per_section": per_section, "macro": macro}


print("Per-section evaluator loaded (use evaluate_soap_per_section for BeTraC mode)")

print("Cell C done  Evaluation metrics loaded")
print("  Metrics: ROUGE-1/2/L, BLEU (1-4), BERTScore(doc+sent), Keyword Coverage,")
print("           Hallucination Detection, LLM-as-Judge (5-dim rubric)")
print("  Reference: Biswas & Talukdar 2024 used ROUGE-1 F1 with GT SOAP notes")


Per-section evaluator loaded (use evaluate_soap_per_section for BeTraC mode)
Cell C done  Evaluation metrics loaded
  Metrics: ROUGE-1/2/L, BLEU (1-4), BERTScore(doc+sent), Keyword Coverage,
           Hallucination Detection, LLM-as-Judge (5-dim rubric)
  Reference: Biswas & Talukdar 2024 used ROUGE-1 F1 with GT SOAP notes


In [18]:
# Cell 16 - Clinical inference patterns + full pipeline runner with K-SOAP and evaluation
# 2026-05 quality upgrade:
#   - calls verify_code_llm for SNOMED + ICD after enrich_codes
#   - adds per-section ROUGE/BLEU/BERTScore when BeTraC mode is active
def _get_confirmed_data(coded_entities):
    ids, texts = set(), set()
    for e in coded_entities:
        if e["status"] == "Confirmed":
            texts.add(e["text"].lower())
            sn = e.get("snomed")
            if sn:
                ids.add(sn["concept_id"])
    return ids, texts

def _has(sn_ids, texts, target_ids, target_kw):
    return (any(t in sn_ids for t in target_ids) or
            any(kw in txt for txt in texts for kw in target_kw))

def _make_inferred(text, sn_id, sn_term, icd_code, icd_desc):
    return {
        "text": text, "label": "Disease", "status": "Confirmed",
        "reasoning": "Deterministic clinical inference",
        "first_offset": None, "_inferred": True,
        "snomed": {"concept_id": sn_id, "term": sn_term,
                   "entity_text": text, "method": "clinical_inference",
                   "crosswalk_icd10": icd_code, "crosswalk_icd10_desc": icd_desc},
        "icd10_cm": {"code": icd_code, "description": icd_desc,
                     "cosine_score": 1.0, "rerank_score": 10.0,
                     "method": "clinical_inference"},
    }

def _run_clinical_inference(coded_entities):
    sn_ids, texts = _get_confirmed_data(coded_entities)
    inferred = []
    if (_has(sn_ids, texts, ["14094001"], ["missed period", "late period", "amenorrhea"]) and
        _has(sn_ids, texts, ["422587007", "422400008"], ["nausea", "vomiting", "nauseated"]) and
        _has(sn_ids, texts, ["162116003"], ["urinary frequency"])):
        inferred.append(_make_inferred("suspected pregnancy (inferred: amenorrhea + nausea + urinary frequency)",
            "77386006", "Pregnancy", "Z32.00", "Encounter for pregnancy test, result unknown"))
    if (_has(sn_ids, texts, ["267036007"], ["shortness of breath", "difficulty breathing"]) and
        _has(sn_ids, texts, ["267038008"], ["swelling", "edema", "swollen"]) and
        _has(sn_ids, texts, [], ["orthopnea", "lying flat", "pillows", "crackles", "heart failure"])):
        inferred.append(_make_inferred("CHF exacerbation (inferred: dyspnea + edema + orthopnea)",
            "42343007", "Congestive heart failure", "I50.9", "Heart failure, unspecified"))
    if (_has(sn_ids, texts, ["29857009"], ["chest pain", "chest pressure"]) and
        _has(sn_ids, texts, ["267036007"], ["shortness of breath", "difficulty breathing"]) and
        _has(sn_ids, texts, [], ["smoking", "smoker", "cocaine", "high blood pressure", "hypertension", "diabetes"])):
        inferred.append(_make_inferred("suspected ACS (inferred: chest pain + dyspnea + risk factors)",
            "394659003", "Acute coronary syndrome", "I24.9", "Acute ischemic heart disease, unspecified"))
    if (_has(sn_ids, texts, ["62315008"], ["diarrhea", "loose stools"]) and
        _has(sn_ids, texts, ["422587007", "422400008"], ["nausea", "vomiting", "nauseated"]) and
        _has(sn_ids, texts, ["21522001", "73063007"], ["abdominal pain", "belly pain", "cramp", "crampy"])):
        inferred.append(_make_inferred("suspected gastroenteritis (inferred: diarrhea + N/V + abdominal pain)",
            "235856003", "Gastroenteritis", "K52.9", "Noninfective gastroenteritis and colitis, unspecified"))
    return inferred

def run_full_pipeline(transcript_text, k_shot=3, strategy="query_aware", reference_soap=None):
    t_start = time.time()
    print("\n--- Phase 1: NER (SPELL Architecture + VerifyNER) ---")
    entities, rejected = run_ner_spell(transcript_text)
    print("\n--- Phase 2: SOAP + Code Enrichment (parallel) ---")
    global _pipeline_entities
    _pipeline_entities = entities
    with ThreadPoolExecutor(max_workers=2) as pool:
        if k_shot == 0:
            f_soap = pool.submit(run_soap_v1_baseline, transcript_text, entities)
        else:
            f_soap = pool.submit(run_ksoap_fewshot, transcript_text, entities, k_shot, strategy)
        f_codes = pool.submit(enrich_codes, entities, transcript_text)
        soap, soap_raw = f_soap.result()
        entities_coded = f_codes.result()

    if globals().get("USE_CODE_VERIFICATION", True):
        print("\n--- Phase 2.25: Per-code LLM verification (MedCodER-style) ---")
        entities_coded = verify_code_llm(entities_coded, "snomed", transcript_text)
        entities_coded = verify_code_llm(entities_coded, "icd10_cm", transcript_text)

    print("\n--- Phase 2.5: Clinical Inference (deterministic) ---")
    inferred_entities = _run_clinical_inference(entities_coded)
    for ie in inferred_entities:
        entities_coded.append(ie)
        print(f"  INFERRED: {ie['text']}")
    if not inferred_entities:
        print("  No inference patterns matched")
    print("\n--- Phase 3: CPT Assignment ---")
    cpt_codes = []
    for ent in entities_coded:
        if ent["label"] == "Procedure" and ent["status"] == "Confirmed":
            norm_text = ent.get("normalized") or ent["text"]
            cpt = get_cpt_deterministic(norm_text)
            if not cpt and norm_text != ent["text"]:
                cpt = get_cpt_deterministic(ent["text"])
            if cpt:
                cpt_codes.append(cpt)
    print(f"  CPT: {len(cpt_codes)} codes assigned")
    print("\n--- Phase 4: SOAP Quality Evaluation ---")
    eval_metrics = evaluate_soap(soap_raw, transcript_text, entities, reference_soap)
    if reference_soap and globals().get("USE_BETRAC_AUDIO", False):
        try:
            per_section = evaluate_soap_per_section(soap_raw, reference_soap)
            eval_metrics["per_section"] = per_section
            macro = per_section["macro"]
            print(f"         [Per-section vs BeTraC GT] macro ROUGE-L={macro['rougeL_f1']:.3f}  BLEU={macro['bleu']:.3f}  BERT={macro['bertscore_f1']:.3f}")
            for sec, m in per_section["per_section"].items():
                print(f"           {sec}: ROUGE-L={m['rougeL_f1']:.3f}  BLEU={m['bleu']:.3f}  BERT={m['bertscore_f1']:.3f}")
        except Exception as e:
            print(f"         per-section eval failed: {e}")

    total = time.time() - t_start
    snomed_count = sum(1 for e in entities_coded if e.get("snomed"))
    icd10_count = sum(1 for e in entities_coded if e.get("icd10_cm") and e["icd10_cm"].get("code"))
    crosswalk_count = sum(1 for e in entities_coded if e.get("icd10_cm") and e["icd10_cm"].get("method") == "SNOMED_crosswalk")
    j = eval_metrics["llm_judge"]
    kc = eval_metrics["keyword_coverage"]
    print(f"\n{'='*65}")
    print(f"  PIPELINE COMPLETE  {total:.1f}s  ({k_shot}-shot, {strategy})")
    print(f"  Entities: {len(entities)} | Rejected: {len(rejected)}")
    print(f"  SNOMED: {snomed_count} | ICD-10: {icd10_count} ({crosswalk_count} crosswalk)")
    print(f"  CPT: {len(cpt_codes)}")
    print(f"  Quality: Judge={j['composite']}/5.0  KW={kc['coverage']:.0%}  Halluc={eval_metrics['hallucination']['count']}")
    print(f"{'='*65}")
    return {
        "transcript": transcript_text, "entities": entities_coded, "rejected": rejected,
        "soap": soap, "soap_raw": soap_raw, "cpt_codes": cpt_codes,
        "eval_metrics": eval_metrics, "config": {"k_shot": k_shot, "strategy": strategy},
        "time_s": round(total, 2),
    }

print("Cell 16 done  Pipeline runner with K-SOAP + per-code verification + per-section eval")


Cell 16 done  Pipeline runner with K-SOAP + per-code verification + per-section eval


In [19]:
# Cell 17 - Run all configs and export formatted TXT + JSON + ZIP
# Includes ROUGE / BLEU in report + JSON + comparison summary

import copy
import json
import time
import zipfile
from datetime import datetime
from IPython.display import display, FileLink

t_run_start = time.time()

configs = [
    {"name": "0-shot_v1_baseline",  "k_shot": 0, "strategy": "smallest"},
    {"name": "3-shot_smallest",     "k_shot": 3, "strategy": "smallest"},
    {"name": "3-shot_query_aware",  "k_shot": 3, "strategy": "query_aware"},
    {"name": "5-shot_smallest",     "k_shot": 5, "strategy": "smallest"},
]

all_results = {}


def _get_ksoap_context_for_export():
    return None

def _attach_ksoap_context(result_dict):
    return result_dict

for cfg in configs:
    print(f"\n{'#'*70}")
    print(f"# RUNNING: {cfg['name']}")
    print(f"{'#'*70}")
    torch.cuda.empty_cache()
    try:
        result = run_full_pipeline(
            transcript,
            k_shot=cfg["k_shot"],
            strategy=cfg["strategy"],
            reference_soap=globals().get("reference_soap"),
        )
        all_results[cfg["name"]] = result
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"  OOM: {cfg['name']} skipped (prompt too long for GPU)")
        all_results[cfg["name"]] = {
            "transcript": transcript,
            "entities": [],
            "rejected": [],
            "soap": {},
            "soap_raw": f"SKIPPED: OOM on {cfg['name']}",
            "cpt_codes": [],
            "eval_metrics": {
                "llm_judge": {"composite": 0},
                "keyword_coverage": {"coverage": 0, "found": 0, "total": 0, "missing": []},
                "hallucination": {"count": 0, "items": []},
                "consistency": {"score": 0, "issues": []},
                "rouge": {},
                "bleu": {},
            },
            "config": {"k_shot": cfg["k_shot"], "strategy": cfg["strategy"]},
            "time_s": 0,
        }

for _result in all_results.values():
    _attach_ksoap_context(_result)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
t_total = round(t_asr_elapsed + t_diar_elapsed + sum(r.get("time_s", 0) for r in all_results.values()), 2)

def hr(char="=", n=78):
    return char * n

def safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def normalize_offset(offset_value):
    if offset_value is None:
        return None
    if isinstance(offset_value, (list, tuple)) and len(offset_value) >= 2:
        return (offset_value[0], offset_value[1])
    if isinstance(offset_value, dict):
        start_value = offset_value.get("start")
        end_value = offset_value.get("end")
        if start_value is not None and end_value is not None:
            return (start_value, end_value)
    return None

def format_offset(offset_value):
    normalized = normalize_offset(offset_value)
    if normalized is None:
        return ""
    return f" [{normalized[0]}:{normalized[1]}]"

def get_rouge_metrics(eval_metrics):
    # Prefer per-section macro values when present (BeTraC mode)
    per_section = eval_metrics.get("per_section", {}).get("macro") if isinstance(eval_metrics.get("per_section"), dict) else None
    if per_section:
        return {
            "rouge1": float(per_section.get("rouge1_f1", 0) or 0),
            "rouge2": 0.0,
            "rougeL": float(per_section.get("rougeL_f1", 0) or 0),
        }
    rouge_block = eval_metrics.get("rouge", {}) or eval_metrics.get("rouge_scores", {}) or {}
    rouge1 = (
        safe_get(rouge_block, "rouge1", "f1") or
        safe_get(rouge_block, "rouge-1", "f1") or
        rouge_block.get("rouge1") or
        rouge_block.get("rouge-1") or
        0
    )
    rouge2 = (
        safe_get(rouge_block, "rouge2", "f1") or
        safe_get(rouge_block, "rouge-2", "f1") or
        rouge_block.get("rouge2") or
        rouge_block.get("rouge-2") or
        0
    )
    rougeL = (
        safe_get(rouge_block, "rougeL", "f1") or
        safe_get(rouge_block, "rouge-l", "f1") or
        safe_get(rouge_block, "rouge_l", "f1") or
        rouge_block.get("rougeL") or
        rouge_block.get("rouge-l") or
        rouge_block.get("rouge_l") or
        0
    )
    return {
        "rouge1": float(rouge1 or 0),
        "rouge2": float(rouge2 or 0),
        "rougeL": float(rougeL or 0),
    }

def get_bleu_metric(eval_metrics):
    per_section = eval_metrics.get("per_section", {}).get("macro") if isinstance(eval_metrics.get("per_section"), dict) else None
    if per_section:
        return float(per_section.get("bleu", 0) or 0)
    bleu_block = eval_metrics.get("bleu", {}) or {}
    if isinstance(bleu_block, dict):
        bleu_value = (
            bleu_block.get("score") or
            bleu_block.get("bleu") or
            bleu_block.get("value") or
            0
        )
    else:
        bleu_value = bleu_block or 0
    return float(bleu_value or 0)

def format_keywords_for_ksoap(entities):
    lines = []
    for idx, e in enumerate(entities, 1):
        span_text = e.get("text", "")
        norm_text = e.get("normalized") or span_text
        label = e.get("label", "Unknown")
        status = e.get("status", "Unknown")
        lines.append(f"{idx:>3}. ({span_text}, {norm_text}) [{label}, {status}]")
    return "\n".join(lines) if lines else "None documented"

def format_ner_entities(entities):
    lines = []
    for idx, e in enumerate(entities, 1):
        text_value = e.get("text", "")
        label_value = e.get("label", "")
        status_value = e.get("status", "")
        offset_str = format_offset(e.get("first_offset"))
        normalized_value = e.get("normalized")
        snomed_value = e.get("snomed")
        icd_value = e.get("icd10_cm")

        left_part = f'{idx:>3}. [{label_value:<10} | {status_value:<14}] "{text_value}"{offset_str}'

        extra_parts = []
        if normalized_value and normalized_value != text_value:
            extra_parts.append(f'norm: "{normalized_value}"')
        if snomed_value:
            extra_parts.append(f'SNOMED: {snomed_value.get("concept_id")} | {snomed_value.get("term")}')
        if icd_value and icd_value.get("code"):
            extra_parts.append(f'ICD-10-CM: {icd_value.get("code")} | {icd_value.get("description")}')

        if extra_parts:
            left_part += " | " + " | ".join(extra_parts)

        lines.append(left_part)

    return "\n".join(lines) if lines else "No entities extracted"

def format_snomed(entities):
    lines = []
    count = 1
    for e in entities:
        sn = e.get("snomed")
        if not sn:
            continue
        src_text = e.get("text", "")
        src_norm = e.get("normalized") or src_text
        icd = e.get("icd10_cm")
        line = f'{count:>3}. {sn.get("concept_id")}  {sn.get("term")}  <- "{src_norm}"'
        if icd and icd.get("code"):
            line += f'  -> {icd.get("code")}'
        lines.append(line)
        count += 1
    return "\n".join(lines) if lines else "No SNOMED mappings"

def format_icd(entities):
    patient_lines = []
    family_lines = []

    for e in entities:
        icd = e.get("icd10_cm")
        if not icd or not icd.get("code"):
            continue

        text_value = e.get("text", "")
        status_value = e.get("status", "")
        offset_str = format_offset(e.get("first_offset"))

        line = f'- "{text_value}"{offset_str} -> {icd.get("code")} ({icd.get("description")})'
        if status_value == "Family_History":
            family_lines.append(line)
        elif status_value != "Negated":
            patient_lines.append(line)

    lines = []
    lines.append("Patient Diagnoses/Symptoms:")
    lines.append("\n".join(patient_lines) if patient_lines else "None")
    lines.append("")
    lines.append("Family History (NOT patient diagnoses):")
    lines.append("\n".join(family_lines) if family_lines else "None")
    return "\n".join(lines)

def format_cpt(cpt_codes):
    if not cpt_codes:
        return "(No procedures explicitly ordered in transcript)"
    lines = []
    for idx, c in enumerate(cpt_codes, 1):
        if isinstance(c, dict):
            lines.append(f"{idx:>3}. {c.get('code', '')} | {c.get('description', '')}")
        else:
            lines.append(f"{idx:>3}. {str(c)}")
    return "\n".join(lines)

def format_soap(soap_obj, soap_raw=""):
    if isinstance(soap_obj, dict) and soap_obj:
        sections = []
        for key in ["K", "S", "O", "A", "P"]:
            if key in soap_obj and soap_obj[key]:
                sections.append(f"{key}: {soap_obj[key]}")
        return "\n\n".join(sections) if sections else (soap_raw or "No SOAP generated")
    return soap_raw or "No SOAP generated"

def format_ai_suggestion(soap_obj, soap_raw=""):
    if isinstance(soap_obj, dict):
        ai_text = soap_obj.get("AI Suggestion") or soap_obj.get("AI_Suggestion") or soap_obj.get("AI")
        if ai_text:
            return f"[Disclaimer]\n{ai_text}"
    return "[Disclaimer]\nNo AI suggestion generated"

def format_eval(eval_metrics):
    kc = eval_metrics.get("keyword_coverage", {})
    hall = eval_metrics.get("hallucination", {})
    judge = eval_metrics.get("llm_judge", {})
    cons = eval_metrics.get("consistency", {})
    rouge_vals = get_rouge_metrics(eval_metrics)
    bleu_val = get_bleu_metric(eval_metrics)

    lines = []
    lines.append(f"Keyword Coverage: {kc.get('coverage', 0):.1%} ({kc.get('found', 0)}/{kc.get('total', 0)})")
    if kc.get("missing"):
        lines.append(f"Missing Keywords: {', '.join(map(str, kc.get('missing', [])))}")

    lines.append(f"Hallucinations: {hall.get('count', 0)}")
    if hall.get("items"):
        lines.append("Hallucination Items:")
        for item in hall.get("items", []):
            lines.append(f"  - {item}")

    lines.append(f"Entity-SOAP Consistency: {cons.get('score', 0):.1%}")
    if cons.get("issues"):
        lines.append("Consistency Issues:")
        for item in cons.get("issues", []):
            lines.append(f"  - {item}")

    lines.append(
        f"ROUGE: R-1={rouge_vals['rouge1']:.4f} | "
        f"R-2={rouge_vals['rouge2']:.4f} | "
        f"R-L={rouge_vals['rougeL']:.4f}"
    )
    lines.append(f"BLEU: {bleu_val:.4f}")

    lines.append(f"LLM Judge Composite: {judge.get('composite', 0):.2f}")
    if judge:
        lines.append(
            f"Judge Breakdown: Completeness={judge.get('completeness', judge.get('comp', 0))} | "
            f"Correctness={judge.get('correctness', judge.get('corr', 0))} | "
            f"Coherence={judge.get('coherence', judge.get('coh', 0))} | "
            f"Assessment={judge.get('assessment', judge.get('ax', 0))} | "
            f"Plan Safety={judge.get('plan_safety', judge.get('plan', 0))}"
        )
    return "\n".join(lines)

report_lines = []
report_lines.append(hr())
report_lines.append("COMPREHENSIVE CLINICAL NLP PIPELINE REPORT")
report_lines.append(f"Generated: {datetime.now():%Y-%m-%d %H:%M:%S}")
report_lines.append(f"Total Time: {t_total}s (ASR={t_asr_elapsed}s | Diarization={t_diar_elapsed}s)")
report_lines.append(hr())

report_lines.append("\nRAW ASR TRANSCRIPT")
report_lines.append("-" * 78)
report_lines.append(asr_transcrip_data[0] if asr_transcrip_data else "No ASR data")

report_lines.append("\nDIARIZED / PROCESSED TRANSCRIPT")
report_lines.append("-" * 78)
report_lines.append(transcript)

report_lines.append("\nNER OUTPUT BY CONFIG")
report_lines.append("-" * 78)
for name, res in all_results.items():
    report_lines.append(f"\n[{name}]")
    report_lines.append(format_ner_entities(res.get("entities", [])))

report_lines.append("\nK-SOAP KEYWORDS BY CONFIG")
report_lines.append("-" * 78)
for name, res in all_results.items():
    report_lines.append(f"\n[{name}]")
    report_lines.append(format_keywords_for_ksoap(res.get("entities", [])))

report_lines.append("\nSOAP NOTES BY CONFIG")
report_lines.append("-" * 78)
for name, res in all_results.items():
    report_lines.append(f"\n[{name}]")
    report_lines.append(format_soap(res.get("soap", {}), res.get("soap_raw", "")))

report_lines.append("\nAI SUGGESTIONS BY CONFIG")
report_lines.append("-" * 78)
for name, res in all_results.items():
    report_lines.append(f"\n[{name}]")
    report_lines.append(format_ai_suggestion(res.get("soap", {}), res.get("soap_raw", "")))

report_lines.append("\nSNOMED CT MAPPINGS BY CONFIG")
report_lines.append("-" * 78)
for name, res in all_results.items():
    report_lines.append(f"\n[{name}]")
    report_lines.append(format_snomed(res.get("entities", [])))

report_lines.append("\nICD-10-CM CODES BY CONFIG")
report_lines.append("-" * 78)
for name, res in all_results.items():
    report_lines.append(f"\n[{name}]")
    report_lines.append(format_icd(res.get("entities", [])))

report_lines.append("\nCPT CODES BY CONFIG")
report_lines.append("-" * 78)
for name, res in all_results.items():
    report_lines.append(f"\n[{name}]")
    report_lines.append(format_cpt(res.get("cpt_codes", [])))

report_lines.append("\nTIMING BY CONFIG")
report_lines.append("-" * 78)
for name, res in all_results.items():
    report_lines.append(f"{name}: {res.get('time_s', 0)}s")

report_lines.append("\nEVALUATION BY CONFIG")
report_lines.append("-" * 78)
for name, res in all_results.items():
    report_lines.append(f"\n[{name}]")
    report_lines.append(format_eval(res.get("eval_metrics", {})))

report_lines.append(f"\n{hr()}")
report_lines.append("COMPARISON SUMMARY")
report_lines.append(hr())

header = f"{'Config':<25} {'KW Cov':>8} {'Consist':>9} {'Halluc':>8} {'R-L':>8} {'BLEU':>8} {'Judge':>8} {'Time':>8}"
report_lines.append(header)
report_lines.append("-" * len(header))

for name, res in all_results.items():
    em = res.get("eval_metrics", {})
    kc = em.get("keyword_coverage", {})
    hall = em.get("hallucination", {})
    judge = em.get("llm_judge", {})
    cons = em.get("consistency", {})
    rouge_vals = get_rouge_metrics(em)
    bleu_val = get_bleu_metric(em)

    line = (
        f"{name:<25} "
        f"{kc.get('coverage', 0):>8.1%} "
        f"{cons.get('score', 0):>9.1%} "
        f"{hall.get('count', 0):>8} "
        f"{rouge_vals['rougeL']:>8.4f} "
        f"{bleu_val:>8.4f} "
        f"{judge.get('composite', 0):>8.2f} "
        f"{res.get('time_s', 0):>7.0f}s"
    )
    report_lines.append(line)

report_txt = "\n".join(report_lines)

report_path = f"Clinical_Report_{ts}.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_txt)

json_export = {
    "generated": datetime.now().isoformat(),
    "timing": {
        "total_s": t_total,
        "asr_s": t_asr_elapsed,
        "diarization_s": t_diar_elapsed,
    },
    "raw_transcript": asr_transcrip_data[0] if asr_transcrip_data else "",
    "diarized_transcript": transcript,
    "ksoap_context": _get_ksoap_context_for_export(),
    "results": {},
}

for name, res in all_results.items():
    em = res.get("eval_metrics", {})
    json_export["results"][name] = {
        "sample_id": res.get("sample_id"),
        "dialogue_file": res.get("dialogue_file"),
        "note_file": res.get("note_file"),
        "reference_soap": res.get("reference_soap"),
        "config": res.get("config", {}),
        "time_s": res.get("time_s", 0),
        "ksoap_keywords": [
            {
                "span": e.get("text"),
                "entity": e.get("normalized") or e.get("text"),
                "label": e.get("label"),
                "assertion": e.get("status"),
                "offset": normalize_offset(e.get("first_offset")),
            }
            for e in res.get("entities", [])
        ],
        "soap": res.get("soap", {}),
        "soap_raw": res.get("soap_raw", ""),
        "ai_suggestion": (
            safe_get(res, "soap", "AI Suggestion") or
            safe_get(res, "soap", "AI_Suggestion") or
            ""
        ),
        "entities": [
            {
                "text": e.get("text"),
                "label": e.get("label"),
                "status": e.get("status"),
                "offset": normalize_offset(e.get("first_offset")),
                "normalized": e.get("normalized"),
                "snomed": {
                    "id": e["snomed"]["concept_id"],
                    "term": e["snomed"]["term"],
                } if e.get("snomed") else None,
                "icd10_cm": {
                    "code": e["icd10_cm"]["code"],
                    "description": e["icd10_cm"]["description"],
                } if e.get("icd10_cm") and e["icd10_cm"].get("code") else None,
            }
            for e in res.get("entities", [])
        ],
        "cpt_codes": res.get("cpt_codes", []),
        "eval_metrics": em,
        "objective_metrics": {
            "rouge": get_rouge_metrics(em),
            "bleu": get_bleu_metric(em),
        },
    }

json_path = f"Clinical_Result_{ts}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_export, f, indent=2, default=str)

zip_path = f"Clinical_Output_{ts}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(report_path, arcname=report_path)
    zipf.write(json_path, arcname=json_path)

t_run_total = round(time.time() - t_run_start, 2)

print(f"\n{hr('#', 65)}")
print("EXPORT COMPLETE")
print(f"TXT : {report_path}")
print(f"JSON: {json_path}")
print(f"ZIP : {zip_path}")
print(f"Total run time: {t_run_total}s")
print(f"{hr('#', 65)}")

display(FileLink(report_path))
display(FileLink(json_path))
display(FileLink(zip_path))



######################################################################
# RUNNING: 0-shot_v1_baseline
######################################################################

--- Phase 1: NER (SPELL Architecture + VerifyNER) ---
  [NER] Phase 1: LLM extraction (3-shot)...


Passing `generation_config` together with generation-related arguments=({'eos_token_id', 'temperature', 'do_sample', 'repetition_penalty', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=3500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=900) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


         -> 33 raw candidates in 108.2s
  [NER] Phase 3: Non-clinical filter (regex only)...
         JUNK: "baby"
         JUNK: "trouble"
         JUNK: "complications"
         JUNK: "years"
         JUNK: "husband"
         JUNK: "checkup"
         JUNK: "general"
         JUNK: "insurance"
         JUNK: "physical"
         JUNK: "exam"
         JUNK: "baselines"
         JUNK: "sinus"
         JUNK: "congestant"
         JUNK: "bottles"
         JUNK: "dosage"
         JUNK: "immune system"
         JUNK: "fertility"
         JUNK: "workup"
         JUNK: "pap smear"
         JUNK: "family history"
         JUNK: "genealogy"
         JUNK: "research"
  [NER] Phase 4: SPELL verification...
  [NER] Phase 5: Offset-aware negation (hardened)...
  [NER] Phase 6: LLM verification (VerifyNER-style)...
  [NER-Verify] Auditing 9 entities...


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=900) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [NER-Verify] Kept 7/9 in 20.2s
  [NER] Final: 7 verified, 26 rejected

--- Phase 2: SOAP + Code Enrichment (parallel) ---
  [SOAP] Building fact table from NER entities...
  [SOAP] Fact table: 568 chars
  [CODES] Normalizing entities (MedCodER Step 1)...
         'pregnant' -> 'pregnancy'
         'period' -> 'menstruation'
         'tests' -> 'diagnostic tests'
         'medications' -> 'pharmacotherapy'
         'vitamin D' -> 'calciferol'
         'supplement' -> 'dietary supplement'
         'pelvic' -> 'pelvis'


...

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


         ROUGE-1 F1: 0.186
         BLEU: 0.000 (B1=0.336 B4=0.000)
         BERTScore(sent) F1: 0.929
         MIST Composite: 0.530
         Running LLM-as-Judge...
         LLM Judge composite: 4.0/5.0
         Entity-SOAP consistency: 100.0%
  [EVAL] Done in 12.3s
         [Per-section vs BeTraC GT] macro ROUGE-L=0.127  BLEU=0.000  BERT=0.899
           S: ROUGE-L=0.050  BLEU=0.000  BERT=0.892
           O: ROUGE-L=0.286  BLEU=0.000  BERT=0.905
           A: ROUGE-L=0.045  BLEU=0.000  BERT=0.899
           P: ROUGE-L=0.000  BLEU=0.000  BERT=0.000

  PIPELINE COMPLETE  200.6s  (5-shot, smallest)
  Entities: 9 | Rejected: 25
  SNOMED: 5 | ICD-10: 2 (2 crosswalk)
  CPT: 2
  Quality: Judge=4.0/5.0  KW=100%  Halluc=0

#################################################################
EXPORT COMPLETE
TXT : Clinical_Report_20260519_173350.txt
JSON: Clinical_Result_20260519_173350.json
ZIP : Clinical_Output_20260519_173350.zip
Total run time: 736.98s
########################################

/kaggle/working/Clinical_Report_20260519_173350.txt

/kaggle/working/Clinical_Result_20260519_173350.json

/kaggle/working/Clinical_Output_20260519_173350.zip

In [20]:
# Cell 18 - A/B comparison runner: benchmarks all configs and prints comparison table
def run_comparison(transcript_text, reference_soap=None):
    """
    Benchmark all configurations and print comparison table.
    Runs: 0-shot baseline, 3-shot smallest, 3-shot query-aware, 5-shot smallest.
    """
    configs = [
        {"name": "0-shot v1 baseline", "k_shot": 0, "strategy": "smallest"},
        {"name": "3-shot smallest",    "k_shot": 3, "strategy": "smallest"},
        {"name": "3-shot query-aware", "k_shot": 3, "strategy": "query_aware"},
        {"name": "5-shot smallest",    "k_shot": 5, "strategy": "smallest"},
    ]

    all_results = []
    for cfg in configs:
        print(f"\n{'#'*65}")
        print(f"# RUNNING: {cfg['name']}")
        print(f"{'#'*65}")
        result = run_full_pipeline(
            transcript_text,
            k_shot=cfg["k_shot"],
            strategy=cfg["strategy"],
            reference_soap=reference_soap,
        )
        all_results.append({"config": cfg["name"], "result": result})

    print(f"\n\n{'='*80}")
    print("COMPARISON RESULTS")
    print(f"{'='*80}")

    header = f"{'Configuration':<25} {'KW Cov':>7} {'Halluc':>7} {'Judge':>7}"
    if reference_soap:
        header += f" {'R1-F1':>7} {'BLEU':>7} {'BS-F1':>7} {'MIST':>7}"
    header += f" {'Time':>6}"
    print(header)
    print("-" * len(header))

    for entry in all_results:
        m = entry["result"]["eval_metrics"]
        r = entry["result"]
        line = (
            f"{entry['config']:<25} "
            f"{m['keyword_coverage']['coverage']:>7.1%} "
            f"{m['hallucination']['count']:>7} "
            f"{m['llm_judge']['composite']:>7.2f}"
        )
        if reference_soap:
            line += (
                f" {m['rouge1']['f1']:>7.3f}"
                f" {m.get('bleu',{}).get('bleu',0):>7.3f}"
                f" {m['bertscore_sent']['f1']:>7.3f}"
                f" {m['mist_composite']:>7.3f}"
            )
        line += f" {r['time_s']:>5.0f}s"
        print(line)

    print(f"{'='*80}")

    print(f"\nLLM Judge Breakdown (1-5 scale):")
    dims = ["completeness", "correctness", "coherence", "assessment_quality", "plan_safety"]
    header2 = f"{'Configuration':<25}"
    for d in dims:
        header2 += f" {d[:8]:>8}"
    print(header2)
    print("-" * len(header2))
    for entry in all_results:
        j = entry["result"]["eval_metrics"]["llm_judge"]
        line2 = f"{entry['config']:<25}"
        for d in dims:
            line2 += f" {j[d]:>8}"
        print(line2)
    print()

    return all_results

print("Cell E done  Comparison runner loaded")
print()
print("USAGE:")
print("  # Single run:")
print("  result = run_full_pipeline(transcript, k_shot=3)")
print()
print("  # Full comparison (0/3/5 shot):")
print("  results = run_comparison(transcript)")
print()
print("  # With gold reference:")
print('  results = run_comparison(transcript, reference_soap="S: ... O: ... A: ... P: ...")')


Cell E done  Comparison runner loaded

USAGE:
  # Single run:
  result = run_full_pipeline(transcript, k_shot=3)

  # Full comparison (0/3/5 shot):
  results = run_comparison(transcript)

  # With gold reference:
  results = run_comparison(transcript, reference_soap="S: ... O: ... A: ... P: ...")


In [21]:
# Cell 19a - Install Gradio
!pip install -q gradio
!pip install --upgrade typing_extensions


In [22]:
# Cell 19b - Gradio handler: runs 3-shot K-SOAP pipeline and returns SOAP + report
import gradio as gr

def generate_for_gradio(prompt, max_new_tokens, temperature, top_p):
    global transcript, result
    transcript = prompt
    result = run_full_pipeline(transcript, k_shot=3, strategy="query_aware")
    soap_raw = result.get("soap_raw", "")
    report = build_report(result)
    em = result.get("eval_metrics", {})
    j = em.get("llm_judge", {})
    kc = em.get("keyword_coverage", {})
    report += f"\n\nEVAL: Judge={j.get('composite','?')}/5.0  KW={kc.get('coverage',0):.0%}  Halluc={em.get('hallucination',{}).get('count',0)}"
    return (soap_raw, report)


In [23]:
# Cell 19c - Gradio UI: dropdown for transcript selection, SOAP + report output
with gr.Blocks() as demo:
    gr.Markdown("# Clinical NLP Pipeline  K-SOAP + NER + ICD + SNOMED + CPT + Metrics")
    with gr.Row():
        with gr.Column(scale=4):
            prompt = gr.Dropdown(label="Select transcript", choices=processed_asr_transcript, value="", interactive=True)
        with gr.Column(scale=1, min_width=50):
            btn = gr.Button("Submit")
    with gr.Accordion("Advanced options", open=False):
        with gr.Row():
            with gr.Column():
                max_new_tokens = gr.Slider(label="Max New Tokens", minimum=128, maximum=16000, step=512, value=4096)
                top_p = gr.Slider(label="top_p", minimum=0.7, maximum=0.95, step=0.05, value=0.7)
            with gr.Column():
                temperature = gr.Slider(label="temperature", minimum=0.4, maximum=0.9, step=0.05, value=0.9)
    with gr.Row():
        output = gr.Textbox(label="K-SOAP Note", lines=10)
        full_report = gr.Textbox(label="Full Report + Eval Metrics", lines=10)
    btn.click(fn=generate_for_gradio, inputs=[prompt, max_new_tokens, temperature, top_p], outputs=[output, full_report])


In [24]:
# Cell 20 - Launch Gradio server
import subprocess
gr.close_all()
demo.launch(server_name="0.0.0.0", server_port=6006, share=True)
subprocess.run(["lt", "--port", "6006"])


* Running on local URL:  http://0.0.0.0:6006
* Running on public URL: https://753c96f38798d98a22.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


FileNotFoundError: [Errno 2] No such file or directory: 'lt'